In [ ]:
import math

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Imports · lumapi setup · Logging · I/O paths                     ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import sys, os, platform, time, logging
from pathlib import Path
from datetime import datetime
from typing import Optional, Tuple, Dict, Any

import numpy as np
import h5py
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

LUMERICAL_VERSION = "v202"
if platform.system() == "Windows":
    LUMERICAL_ROOT = rf"C:\Program Files\Lumerical\{LUMERICAL_VERSION}"
    LUMERICAL_API  = rf"{LUMERICAL_ROOT}\api\python"
    LUMERICAL_BIN  = rf"{LUMERICAL_ROOT}\bin"
else:
    LUMERICAL_ROOT = f"/opt/lumerical/{LUMERICAL_VERSION}"
    LUMERICAL_API  = f"{LUMERICAL_ROOT}/api/python"
    LUMERICAL_BIN  = f"{LUMERICAL_ROOT}/bin"

if "lumapi" in sys.modules:
    del sys.modules["lumapi"]
if LUMERICAL_API not in sys.path:
    sys.path.insert(0, LUMERICAL_API)
if platform.system() == "Windows":
    if hasattr(os, "add_dll_directory"):
        os.add_dll_directory(str(LUMERICAL_BIN))
    else:
        os.environ["PATH"] = str(LUMERICAL_BIN) + ";" + os.environ.get("PATH", "")

assert Path(LUMERICAL_API).exists(), f"API path not found: {LUMERICAL_API}"
assert Path(LUMERICAL_BIN).exists(), f"bin path not found: {LUMERICAL_BIN}"

import lumapi

print(f"lumapi imported from:\n  {lumapi.__file__}")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s │ %(levelname)s │ %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("ICNT_Cascade")

# ── V3: OPMs replaced by ONA multiport — version name updated ─────────────────
VERSION_NAME = "ICNT_14Ring_Cascade_UniDir_neff_sweep_V3"
PROJECT_DIR  = Path.cwd()
DATA_DIR     = PROJECT_DIR / "data_ICNT_cascade_ring_sweep"
DATA_DIR.mkdir(parents=True, exist_ok=True)
HDF5_PATH    = DATA_DIR / f"{VERSION_NAME}.h5"
FIGURES_DIR  = DATA_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n  Data directory : {DATA_DIR}")
print(f"  HDF5 output    : {HDF5_PATH}")
print(f"  Figures dir    : {FIGURES_DIR}")

# ═════════════════════════════════════════════════════════════════════════════
#  END CELL 1
# ═════════════════════════════════════════════════════════════════════════════


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Simulation parameters                                            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

N_RINGS = 14

RING_RADIUS_M = np.array([
    19.0021e-6, 19.1818e-6, 19.1934e-6, 19.2051e-6,
    19.2168e-6, 19.2284e-6, 19.2401e-6, 19.2517e-6,
    19.4158e-6, 19.4275e-6, 19.4393e-6, 19.4511e-6,
    19.4628e-6, 19.4746e-6,
])
RING_LAMBDA_RES_M = np.array([
    1.5500000e-6, 1.5500000e-6, 1.5507692e-6, 1.5515385e-6,
    1.5523077e-6, 1.5530769e-6, 1.5538462e-6, 1.5546154e-6,
    1.5553846e-6, 1.5561538e-6, 1.5569231e-6, 1.5576923e-6,
    1.5584615e-6, 1.5582308e-6,
])
RING_NEFF_TE = np.array([
    1.609803, 1.633303, 1.633121, 1.632939,
    1.632758, 1.632576, 1.632394, 1.632213,
    1.631974, 1.631792, 1.631611, 1.631430,
    1.631248, 1.631067,
])
RING_NG_TE = np.array([
    2.020543, 1.991101, 1.990956, 1.990808,
    1.990659, 1.990509, 1.990356, 1.990203,
    1.990047, 1.989891, 1.989733, 1.989691,
    1.989528, 1.989364,
])
RING_D_TE_PS2_PER_KM = np.zeros(N_RINGS)
RING_D_TM_PS2_PER_KM = np.zeros(N_RINGS)
RING_NEFF_TM         = np.full(N_RINGS, 1.7000)
RING_NG_TM           = np.full(N_RINGS, 2.1000)
RING_KAPPA_INPUT_SQ  = np.array([
    0.145778, 0.145072, 0.145011, 0.144949,
    0.144888, 0.144827, 0.144765, 0.144704,
    0.145696, 0.145634, 0.145572, 0.145518,
    0.145456, 0.145394,
])
RING_KAPPA_DROP_SQ = np.array([
    0.143402, 0.142672, 0.142609, 0.142546,
    0.142484, 0.142420, 0.142357, 0.142294,
    0.143269, 0.143205, 0.143142, 0.143086,
    0.143022, 0.142958,
])
RING_LOSS_DB_PER_M = np.full(N_RINGS, 101.0)
RING_POLARIZATION  = ["TE"] * N_RINGS

ONA_LAMBDA_START_M = 1.549e-6
ONA_LAMBDA_STOP_M  = 1.559e-6
ONA_N_POINTS       = 1000
ONA_POWER_DBM      = 0.0

# ── V3 CHANGE: 15 ONA inputs ──────────────────────────────────────────────────
#   input  1 → RING_1  through   (sensor ring through port)
#   input  2 → RING_2  drop      (spectrometer ring 2 drop)
#   input  3 → RING_3  drop
#   ...
#   input 14 → RING_14 drop      (spectrometer ring 14 drop)
#   input 15 → RING_14 through   (final through port of cascade)
# ─────────────────────────────────────────────────────────────────────────────
ONA_N_INPUT_PORTS = 15   # was 2

# Number of spectrometer drops = rings 2..14
N_DROPS = 13   # replaces N_OPM; no physical OPM elements are used

SWEEP_N_POINTS = 200

SWEEP_NEFF = np.array([
    1.6095387512, 1.6095769971, 1.6096152579, 1.6096535338, 1.6096918246, 1.6097301304, 1.6097684512, 1.6098067871, 1.6098451380, 1.6098835039,
    1.6099218849, 1.6099602810, 1.6099986921, 1.6100371184, 1.6100755597, 1.6101140162, 1.6101524877, 1.6101909744, 1.6102294763, 1.6102679933,
    1.6103065255, 1.6103450728, 1.6103836354, 1.6104222131, 1.6104608060, 1.6104994142, 1.6105380376, 1.6105766762, 1.6106153301, 1.6106539993,
    1.6106926837, 1.6107313835, 1.6107700985, 1.6108088288, 1.6108475745, 1.6108863355, 1.6109251118, 1.6109639035, 1.6110027106, 1.6110415330,
    1.6110803708, 1.6111192240, 1.6111580927, 1.6111969767, 1.6112358762, 1.6112747912, 1.6113137215, 1.6113526674, 1.6113916287, 1.6114306056,
    1.6114695979, 1.6115086057, 1.6115476291, 1.6115866680, 1.6116257225, 1.6116647925, 1.6117038781, 1.6117429792, 1.6117820960, 1.6118212284,
    1.6118603763, 1.6118995400, 1.6119387192, 1.6119779141, 1.6120171247, 1.6120563509, 1.6120955929, 1.6121348505, 1.6121741238, 1.6122134129,
    1.6122527177, 1.6122920382, 1.6123313745, 1.6123707266, 1.6124100945, 1.6124494781, 1.6124888776, 1.6125282928, 1.6125677239, 1.6126071709,
    1.6126466336, 1.6126861123, 1.6127256068, 1.6127651172, 1.6128046436, 1.6128441858, 1.6128837439, 1.6129233180, 1.6129629081, 1.6130025141,
    1.6130421360, 1.6130817740, 1.6131214279, 1.6131610979, 1.6132007839, 1.6132404859, 1.6132802040, 1.6133199381, 1.6133596883, 1.6133994545,
    1.6134392369, 1.6134790354, 1.6135188500, 1.6135586807, 1.6135985276, 1.6136383906, 1.6136782698, 1.6137181652, 1.6137580767, 1.6137980045,
    1.6138379485, 1.6138779087, 1.6139178852, 1.6139578779, 1.6139978869, 1.6140379122, 1.6140779537, 1.6141180116, 1.6141580858, 1.6141981763,
    1.6142382832, 1.6142784064, 1.6143185460, 1.6143587019, 1.6143988743, 1.6144390631, 1.6144792683, 1.6145194899, 1.6145597279, 1.6145999825,
    1.6146402535, 1.6146805410, 1.6147208449, 1.6147611654, 1.6148015024, 1.6148418560, 1.6148822261, 1.6149226127, 1.6149630160, 1.6150034358,
    1.6150438722, 1.6150843252, 1.6151247949, 1.6151652812, 1.6152057842, 1.6152463038, 1.6152868401, 1.6153273931, 1.6153679628, 1.6154085492,
    1.6154491523, 1.6154897722, 1.6155304089, 1.6155710623, 1.6156117325, 1.6156524195, 1.6156931233, 1.6157338439, 1.6157745814, 1.6158153358,
    1.6158561070, 1.6158968950, 1.6159377000, 1.6159785219, 1.6160193607, 1.6160602164, 1.6161010891, 1.6161419787, 1.6161828853, 1.6162238089,
    1.6162647495, 1.6163057071, 1.6163466817, 1.6163876734, 1.6164286821, 1.6164697079, 1.6165107508, 1.6165518108, 1.6165928879, 1.6166339821,
    1.6166750935, 1.6167162220, 1.6167573676, 1.6167985305, 1.6168397106, 1.6168809078, 1.6169221223, 1.6169633540, 1.6170046030, 1.6170458692,
    1.6170871527, 1.6171284535, 1.6171697716, 1.6172111070, 1.6172524598, 1.6172938299, 1.6173352174, 1.6173766222, 1.6174180445, 1.6174594841,
])

SWEEP_NG = np.array([
    2.0218079294, 2.0217699008, 2.0217318512, 2.0216937805, 2.0216556887, 2.0216175759, 2.0215794420, 2.0215412869, 2.0215031108, 2.0214649135,
    2.0214266951, 2.0213884554, 2.0213501947, 2.0213119127, 2.0212736095, 2.0212352850, 2.0211969393, 2.0211585724, 2.0211201842, 2.0210817747,
    2.0210433438, 2.0210048917, 2.0209664182, 2.0209279233, 2.0208894071, 2.0208508695, 2.0208123105, 2.0207737300, 2.0207351281, 2.0206965047,
    2.0206578599, 2.0206191936, 2.0205805057, 2.0205417964, 2.0205030655, 2.0204643130, 2.0204255389, 2.0203867433, 2.0203479260, 2.0203090871,
    2.0202702265, 2.0202313443, 2.0201924404, 2.0201535147, 2.0201145674, 2.0200755983, 2.0200366074, 2.0199975948, 2.0199585603, 2.0199195041,
    2.0198804260, 2.0198413260, 2.0198022042, 2.0197630605, 2.0197238948, 2.0196847073, 2.0196454977, 2.0196062662, 2.0195670127, 2.0195277372,
    2.0194884397, 2.0194491201, 2.0194097784, 2.0193704147, 2.0193310288, 2.0192916208, 2.0192521906, 2.0192127383, 2.0191732638, 2.0191337670,
    2.0190942481, 2.0190547068, 2.0190151433, 2.0189755575, 2.0189359493, 2.0188963188, 2.0188566660, 2.0188169907, 2.0187772931, 2.0187375730,
    2.0186978305, 2.0186580655, 2.0186182780, 2.0185784679, 2.0185386354, 2.0184987803, 2.0184589025, 2.0184190022, 2.0183790793, 2.0183391337,
    2.0182991654, 2.0182591744, 2.0182191607, 2.0181791242, 2.0181390650, 2.0180989829, 2.0180588781, 2.0180187504, 2.0179785999, 2.0179384264,
    2.0178982301, 2.0178580108, 2.0178177685, 2.0177775033, 2.0177372150, 2.0176969037, 2.0176565693, 2.0176162119, 2.0175758313, 2.0175354277,
    2.0174950008, 2.0174545508, 2.0174140775, 2.0173735810, 2.0173330613, 2.0172925182, 2.0172519518, 2.0172113621, 2.0171707491, 2.0171301126,
    2.0170894527, 2.0170487694, 2.0170080626, 2.0169673322, 2.0169265784, 2.0168858010, 2.0168450000, 2.0168041754, 2.0167633272, 2.0167224553,
    2.0166815597, 2.0166406404, 2.0165996973, 2.0165587305, 2.0165177398, 2.0164767254, 2.0164356870, 2.0163946248, 2.0163535386, 2.0163124285,
    2.0162712945, 2.0162301364, 2.0161889542, 2.0161477481, 2.0161065178, 2.0160652634, 2.0160239848, 2.0159826820, 2.0159413551, 2.0159000039,
    2.0158586284, 2.0158172286, 2.0157758045, 2.0157343560, 2.0156928831, 2.0156513857, 2.0156098639, 2.0155683176, 2.0155267468, 2.0154851515,
    2.0154435315, 2.0154018869, 2.0153602177, 2.0153185238, 2.0152768051, 2.0152350618, 2.0151932936, 2.0151515006, 2.0151096828, 2.0150678401,
    2.0150259724, 2.0149840798, 2.0149421623, 2.0149002197, 2.0148582521, 2.0148162593, 2.0147742415, 2.0147321985, 2.0146901303, 2.0146480369,
    2.0146059182, 2.0145637743, 2.0145216050, 2.0144794103, 2.0144371902, 2.0143949447, 2.0143526738, 2.0143103773, 2.0142680552, 2.0142257076,
    2.0141833344, 2.0141409355, 2.0140985109, 2.0140560606, 2.0140135845, 2.0139710826, 2.0139285549, 2.0138860012, 2.0138434217, 2.0138008162,
])

# ── Validation ────────────────────────────────────────────────────────────────
for arr, name in [
    (RING_RADIUS_M,      "RING_RADIUS_M"),
    (RING_LAMBDA_RES_M,  "RING_LAMBDA_RES_M"),
    (RING_NEFF_TE,       "RING_NEFF_TE"),
    (RING_NG_TE,         "RING_NG_TE"),
    (RING_KAPPA_INPUT_SQ,"RING_KAPPA_INPUT_SQ"),
    (RING_KAPPA_DROP_SQ, "RING_KAPPA_DROP_SQ"),
    (RING_LOSS_DB_PER_M, "RING_LOSS_DB_PER_M"),
]:
    assert len(arr) == N_RINGS, f"{name} length mismatch"
assert len(SWEEP_NEFF) == SWEEP_N_POINTS
assert len(SWEEP_NG)   == SWEEP_N_POINTS

print("=" * 80)
print("  INTERCONNECT 14-Ring Cascade — Parameter Summary  [UNIDIRECTIONAL  V3]")
print("=" * 80)
print(f"  {'Ring':>5}  {'R [µm]':>9}  {'λ_res [nm]':>12}  "
      f"{'neff_TE':>9}  {'ng_TE':>8}  {'κ²_in':>8}  {'κ²_dr':>8}  {'Loss':>7}")
print("  " + "─" * 72)
for i in range(N_RINGS):
    tag = "  ← SENSOR (swept)" if i == 0 else ""
    print(f"  {i+1:>5}  {RING_RADIUS_M[i]*1e6:>9.4f}  "
          f"{RING_LAMBDA_RES_M[i]*1e9:>12.4f}  "
          f"{RING_NEFF_TE[i]:>9.6f}  {RING_NG_TE[i]:>8.6f}  "
          f"{RING_KAPPA_INPUT_SQ[i]:>8.6f}  {RING_KAPPA_DROP_SQ[i]:>8.6f}  "
          f"{RING_LOSS_DB_PER_M[i]:>7.1f}{tag}")
print()
print(f"  ONA   : λ {ONA_LAMBDA_START_M*1e9:.2f}–{ONA_LAMBDA_STOP_M*1e9:.2f} nm  "
      f"│  {ONA_N_POINTS} pts  │  {ONA_POWER_DBM} dBm  │  {ONA_N_INPUT_PORTS} input ports")
print(f"  Sweep : neff {SWEEP_NEFF[0]:.5f}→{SWEEP_NEFF[-1]:.5f}  "
      f"│  ng {SWEEP_NG[0]:.5f}→{SWEEP_NG[-1]:.5f}  │  {SWEEP_N_POINTS} pts")
print()
print("  CIRCUIT TOPOLOGY  [V3 — ONA multiport, no physical OPMs]")
print("  ─────────────────────────────────────────────────────────────────────")
print("  ONA  output    → RING_1  input")
print("  RING_1  out1   → ONA     input 1        (sensor through)")
print("  RING_1  out2   → RING_2  input           (sensor drop → cascade)")
print("  RING_n  out1   → RING_n+1 input  n=2..13 (through → next ring)")
print("  RING_n  out2   → ONA     input n  n=2..14 (drop → ONA input n)")
print("  RING_14 out1   → ONA     input 15        (final through)")
print("  ─────────────────────────────────────────────────────────────────────")
print("  ONA input mapping:")
print("    input  1 → RING_1  through  (sensor)")
for k in range(2, N_RINGS + 1):
    print(f"    input {k:>2} → RING_{k:<2} drop     (spectrometer {k-1:2d})")
print(f"    input 15 → RING_14 through  (cascade end)")
print("=" * 80)

# ═════════════════════════════════════════════════════════════════════════════
#  END CELL 2
# ═════════════════════════════════════════════════════════════════════════════


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Circuit builder + sweep engine + HDF5 storage  [V3]             ║
# ║                                                                            ║
# ║  KEY CHANGE vs V2                                                          ║
# ║  ─────────────────────────────────────────────────────────────────────     ║
# ║  The 13 physical OPM elements are REMOVED.  Instead, each spectrometer    ║
# ║  ring drop port is connected to a dedicated ONA input port (inputs 2-14). ║
# ║  The ONA then measures the full spectral transmission T(λ) at each drop.  ║
# ║                                                                            ║
# ║  The "detector power" that replicates laboratory conditions is computed    ║
# ║  in post-processing by integrating |T_drop(λ)|² over the ONA bandwidth:   ║
# ║                                                                            ║
# ║      P_det [W] = P_source [W] × ∫ |T_drop(λ)|² dλ / Δλ_ONA              ║
# ║                                                                            ║
# ║  This is physically identical to what a broadband photodetector does:      ║
# ║  it integrates all the spectral power that reaches it.  The result is a   ║
# ║  single power value per drop per sweep point → Power vs neff curve.       ║
# ║                                                                            ║
# ║  ONA PORT MAP (15 inputs)                                                  ║
# ║    input  1 → RING_1  output 1  (sensor through)                         ║
# ║    input  k → RING_k  output 2  k = 2..14  (spectrometer drops)          ║
# ║    input 15 → RING_14 output 1  (cascade end / final through)             ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

SPEED_OF_LIGHT = 299_792_458.0
ONA_NAME       = "ONA_1"

# ONA input port labels for easy reference
# drop_ona_input(k): ONA input number for RING_k drop  (k = 2..14)
def drop_ona_input(ring_k: int) -> int:
    """Return the ONA input port number (1-based) for RING_k drop.
    ring_k = 2..14  →  ONA input = ring_k  (i.e. 2..14)
    """
    assert 2 <= ring_k <= N_RINGS, f"ring_k must be 2..{N_RINGS}"
    return ring_k   # ONA input 2 = RING_2 drop, ..., input 14 = RING_14 drop

THROUGH_SENSOR_INPUT  = 1    # ONA input for RING_1 through
THROUGH_FINAL_INPUT   = 15   # ONA input for RING_14 through


def ring_name(ring_id: int) -> str:
    return f"RING_{ring_id}"


# ─────────────────────────────────────────────────────────────────────────────
# Scripting primitives (unchanged from V2)
# ─────────────────────────────────────────────────────────────────────────────
class ICScriptError(RuntimeError):
    pass


def _eval(ic, cmd: str) -> None:
    cmd = cmd.strip().rstrip(";") + ";"
    try:
        ic.eval(cmd)
    except Exception as exc:
        raise ICScriptError(
            f"\n  INTERCONNECT rejected:\n    {cmd}\n  Error: {exc}"
        ) from exc


def _try_eval(ic, cmd: str) -> bool:
    cmd = cmd.strip().rstrip(";") + ";"
    try:
        ic.eval(cmd)
        return True
    except Exception:
        return False


# ─────────────────────────────────────────────────────────────────────────────
# Ring parameter setter (unchanged from V2)
# ─────────────────────────────────────────────────────────────────────────────
def _apply_ring_params(ic, ring_idx: int,
                       neff_override: Optional[float] = None,
                       ng_override:   Optional[float] = None) -> None:
    name = ring_name(ring_idx + 1)
    neff = neff_override if neff_override is not None else float(RING_NEFF_TE[ring_idx])
    ng   = ng_override   if ng_override   is not None else float(RING_NG_TE[ring_idx])
    pol  = RING_POLARIZATION[ring_idx].upper()
    d_si = (float(RING_D_TE_PS2_PER_KM[ring_idx] if pol == "TE"
                  else RING_D_TM_PS2_PER_KM[ring_idx]) * 1e-15)
    res_hz        = SPEED_OF_LIGHT / float(RING_LAMBDA_RES_M[ring_idx])
    circumference = RING_RADIUS_M[ring_idx] * 2.0 * math.pi

    _eval(ic, f'setnamed("{name}", "length",                   {circumference:.12e})')
    _eval(ic, f'setnamed("{name}", "frequency",                {res_hz:.12e})')
    _eval(ic, f'setnamed("{name}", "effective index 1",        {neff:.12f})')
    _eval(ic, f'setnamed("{name}", "group index 1",            {ng:.12f})')
    _eval(ic, f'setnamed("{name}", "loss 1",                   {RING_LOSS_DB_PER_M[ring_idx]:.6f})')
    _eval(ic, f'setnamed("{name}", "dispersion 1",             {d_si:.12e})')
    _eval(ic, f'setnamed("{name}", "coupling coefficient 1 1", {RING_KAPPA_INPUT_SQ[ring_idx]:.12f})')
    _eval(ic, f'setnamed("{name}", "coupling coefficient 1 2", {RING_KAPPA_DROP_SQ[ring_idx]:.12f})')
    _eval(ic, f'setnamed("{name}", "configuration", "unidirectional")')


# ─────────────────────────────────────────────────────────────────────────────
# Sensor ring updater (unchanged from V2 — only RING_1 changes in sweep)
# ─────────────────────────────────────────────────────────────────────────────
def _update_ring1_neff_ng(ic, neff: float, ng: float) -> None:
    name = ring_name(1)
    _eval(ic, f'setnamed("{name}", "effective index 1", {neff:.12f})')
    _eval(ic, f'setnamed("{name}", "group index 1",     {ng:.12f})')


# ─────────────────────────────────────────────────────────────────────────────
# Circuit builder — V3 TOPOLOGY (ONA multiport, no OPMs)
# ─────────────────────────────────────────────────────────────────────────────
def _build_circuit(ic) -> None:
    """
    Build the 14-ring cascade with ONA multiport measurement.

    V3 TOPOLOGY
    ────────────────────────────────────────────────────────────────
    ONA  output    → RING_1  input

    RING_1  out1   → ONA  input 1          (sensor through)
    RING_1  out2   → RING_2 input           (sensor drop → cascade)

    For n = 2..13:
      RING_n  out1 → RING_{n+1} input       (through → next ring)
      RING_n  out2 → ONA  input n            (drop → ONA input n)

    RING_14 out1   → ONA  input 15          (final through)
    RING_14 out2   → ONA  input 14          (RING_14 drop → ONA input 14)

    ONA input mapping:
      input  1 = RING_1  through  (sensor)
      input  k = RING_k  drop     k = 2..14
      input 15 = RING_14 through  (cascade end)
    """
    _eval(ic, "switchtodesign")
    _try_eval(ic, "selectall")
    _try_eval(ic, "delete")

    pwr_W   = 10.0 ** (ONA_POWER_DBM / 10.0) * 1e-3
    f_start = SPEED_OF_LIGHT / ONA_LAMBDA_STOP_M
    f_stop  = SPEED_OF_LIGHT / ONA_LAMBDA_START_M

    # ── ONA — 15 input ports ──────────────────────────────────────────────────
    _eval(ic, 'addelement("Optical Network Analyzer")')
    _eval(ic, f'set("name", "{ONA_NAME}")')
    _try_eval(ic, 'set("x position", 0)')
    _try_eval(ic, 'set("y position", 0)')
    _eval(ic, f'setnamed("{ONA_NAME}", "input parameter",       "start and stop")')
    _eval(ic, f'setnamed("{ONA_NAME}", "number of input ports", {int(ONA_N_INPUT_PORTS)})')
    _eval(ic, f'setnamed("{ONA_NAME}", "start frequency",       {f_start:.12e})')
    _eval(ic, f'setnamed("{ONA_NAME}", "stop frequency",        {f_stop:.12e})')
    _eval(ic, f'setnamed("{ONA_NAME}", "number of points",      {int(ONA_N_POINTS)})')
    _eval(ic, f'setnamed("{ONA_NAME}", "power",                 {pwr_W:.12e})')
    log.info(f"  {ONA_NAME} added — {ONA_N_INPUT_PORTS} input ports configured.")

    # ── Rings ─────────────────────────────────────────────────────────────────
    for i in range(N_RINGS):
        rn = ring_name(i + 1)
        _eval(ic, 'addelement("Double Bus Ring Resonator")')
        _eval(ic, f'set("name", "{rn}")')
        _try_eval(ic, f'set("x position", {float((i + 1) * 220)})')
        _try_eval(ic, f'set("y position", 0)')
        _apply_ring_params(ic, ring_idx=i)
        log.info(
            f"  RING_{i+1:2d} added  "
            f"[unidir, neff={RING_NEFF_TE[i]:.6f}, "
            f"L={RING_RADIUS_M[i]*2*math.pi*1e6:.4f} µm]"
        )

    # ── NO OPM elements — drops go directly to ONA inputs ────────────────────

    # ── Connections ───────────────────────────────────────────────────────────
    def wire(elem_a: str, port_a: str, elem_b: str, port_b: str) -> None:
        _eval(ic, f'connect("{elem_a}", "{port_a}", "{elem_b}", "{port_b}")')

    # ONA output → RING_1 input
    wire(ONA_NAME, "output", ring_name(1), "input")

    # RING_1 through → ONA input 1  (sensor through)
    wire(ring_name(1), "output 1", ONA_NAME, f"input {THROUGH_SENSOR_INPUT}")

    # RING_1 drop → RING_2 input  (cascade entry)
    wire(ring_name(1), "output 2", ring_name(2), "input")

    # RING_2 .. RING_13: through → next ring,  drop → ONA input n
    for i in range(2, N_RINGS):      # i = 2, 3, ..., 13
        wire(ring_name(i), "output 1", ring_name(i + 1), "input")
        wire(ring_name(i), "output 2", ONA_NAME, f"input {drop_ona_input(i)}")
        log.info(f"  RING_{i} drop → ONA input {drop_ona_input(i)}")

    # RING_14 drop → ONA input 14
    wire(ring_name(N_RINGS), "output 2", ONA_NAME, f"input {drop_ona_input(N_RINGS)}")
    log.info(f"  RING_14 drop → ONA input {drop_ona_input(N_RINGS)}")

    # RING_14 through → ONA input 15  (final through)
    wire(ring_name(N_RINGS), "output 1", ONA_NAME, f"input {THROUGH_FINAL_INPUT}")
    log.info(f"  RING_14 through → ONA input {THROUGH_FINAL_INPUT}")

    log.info(
        f"V3 circuit built: {N_RINGS} rings, {N_DROPS} drop ports → ONA inputs 2-14, "
        f"no OPM elements."
    )


# ─────────────────────────────────────────────────────────────────────────────
# Result extraction — V3
# ─────────────────────────────────────────────────────────────────────────────
def _extract_results(ic) -> tuple:
    """
    Extract all ONA transmission spectra after run().

    ONA PORT MAP (V3)
    ─────────────────
    input  1 → RING_1  through   → T_sensor_through_dB   (n_wl,)
    input  k → RING_k  drop      → T_drop_dB[k-2, :]     k=2..14  (13, n_wl)
    input 15 → RING_14 through   → T_final_through_dB    (n_wl,)

    INTEGRATED DETECTOR POWER
    ─────────────────────────
    For each drop port k (ring 2..14):

        P_det[k] [W] = P_source [W] × mean(|T_drop_k(λ)|²)

    where mean is taken over the full ONA wavelength band.
    This replicates a broadband photodetector integrating all arriving power.
    The result is reported in dBm.

    Returns
    -------
    wl_m                  : (n_wl,)       wavelengths [m], ascending
    T_sensor_through_dB   : (n_wl,)       RING_1 through spectrum [dB]
    T_final_through_dB    : (n_wl,)       RING_14 through spectrum [dB]
    T_drop_dB             : (N_DROPS, n_wl)  drop spectra for rings 2..14 [dB]
    drop_power_dBm        : (N_DROPS,)    integrated detector power per drop [dBm]
    """
    # ── Frequency / wavelength axis from ONA input 1 ─────────────────────────
    raw_ref = ic.getresult(ONA_NAME, f"input {THROUGH_SENSOR_INPUT}/mode 1/transmission")
    f_arr   = np.asarray(raw_ref["frequency"]).flatten()
    sort_i  = np.argsort(f_arr)[::-1]          # descending f → ascending λ
    wl_m    = SPEED_OF_LIGHT / f_arr[sort_i]
    n_wl    = len(wl_m)

    # Source power in Watts (needed for absolute power calculation)
    p_source_W = 10.0 ** (ONA_POWER_DBM / 10.0) * 1e-3

    # ── Helper: read one ONA input, return T [linear] sorted ascending in λ ──
    def _read_T_linear(port_label: str) -> np.ndarray:
        raw = ic.getresult(ONA_NAME, f"{port_label}/mode 1/transmission")
        T   = np.asarray(raw["TE transmission"]).flatten()[sort_i]
        return np.abs(T)

    def _T_to_dB(T_lin: np.ndarray) -> np.ndarray:
        return 10.0 * np.log10(np.where(T_lin > 0, T_lin, 1e-30))

    # ── ONA input 1: sensor through ───────────────────────────────────────────
    T_sensor_lin        = _read_T_linear(f"input {THROUGH_SENSOR_INPUT}")
    T_sensor_through_dB = _T_to_dB(T_sensor_lin)

    # ── ONA input 15: cascade final through ───────────────────────────────────
    T_final_lin          = _read_T_linear(f"input {THROUGH_FINAL_INPUT}")
    T_final_through_dB   = _T_to_dB(T_final_lin)

    # ── ONA inputs 2..14: spectrometer drops ─────────────────────────────────
    T_drop_dB      = np.full((N_DROPS, n_wl), np.nan)
    drop_power_dBm = np.full(N_DROPS, np.nan)

    for k in range(2, N_RINGS + 1):          # k = 2..14
        drop_idx = k - 2                      # 0-based index into N_DROPS arrays
        ona_port = drop_ona_input(k)          # ONA input number (= k)
        try:
            T_lin = _read_T_linear(f"input {ona_port}")
            T_drop_dB[drop_idx, :] = _T_to_dB(T_lin)

            # ── Integrated detector power ─────────────────────────────────────
            # mean(|T|²) over the ONA band = fraction of source power reaching
            # the detector.  For a broadband (incoherent) flat-spectrum source
            # this is the physical power seen by a wideband photodetector.
            # |T|² is already the power transmittance (ONA returns amplitude
            # transmission, so |T_lin|² is the power ratio).
            mean_T_sq  = float(np.mean(T_lin ** 2))
            p_det_W    = p_source_W * mean_T_sq
            drop_power_dBm[drop_idx] = 10.0 * np.log10(max(p_det_W, 1e-40) * 1e3)

        except Exception as exc:
            log.warning(f"  Drop extraction failed RING_{k} (ONA input {ona_port}): {exc}")

    return wl_m, T_sensor_through_dB, T_final_through_dB, T_drop_dB, drop_power_dBm


# ─────────────────────────────────────────────────────────────────────────────
# HDF5 initialisation — V3 dataset names
# ─────────────────────────────────────────────────────────────────────────────
def _init_hdf5(wl_ref_m: np.ndarray) -> None:
    n_pts = SWEEP_N_POINTS
    n_wl  = len(wl_ref_m)
    with h5py.File(HDF5_PATH, "w") as f:
        md = f.create_group("metadata")
        md.create_dataset("neff_sweep",    data=SWEEP_NEFF)
        md.create_dataset("ng_sweep",      data=SWEEP_NG)
        md.create_dataset("wavelengths_m", data=wl_ref_m)
        md.attrs["version_name"]          = VERSION_NAME
        md.attrs["n_rings"]               = N_RINGS
        md.attrs["n_drops"]               = N_DROPS
        md.attrs["drop_layout"]           = "drop_k monitors RING_(k+2), k=0..12 (0-based)"
        md.attrs["sweep_n_points"]        = SWEEP_N_POINTS
        md.attrs["ring_model"]            = "Double Bus Ring Resonator"
        md.attrs["ring_configuration"]    = "unidirectional"
        md.attrs["topology"]              = (
            "V3: ONA multiport, no OPMs. "
            "ONA input 1=RING_1 through, input k=RING_k drop (k=2-14), "
            "input 15=RING_14 through"
        )
        md.attrs["ona_lambda_start_m"]    = ONA_LAMBDA_START_M
        md.attrs["ona_lambda_stop_m"]     = ONA_LAMBDA_STOP_M
        md.attrs["ona_n_points"]          = ONA_N_POINTS
        md.attrs["ona_power_dBm"]         = ONA_POWER_DBM
        md.attrs["ona_n_input_ports"]     = ONA_N_INPUT_PORTS
        md.attrs["power_calc_method"]     = (
            "P_det[W] = P_source[W] * mean(|T_drop(lambda)|^2) over ONA band"
        )
        md.attrs["timestamp_start"]       = datetime.now().isoformat()

        for i in range(N_RINGS):
            p = f"ring{i+1}_"
            md.attrs[p + "radius_m"]        = RING_RADIUS_M[i]
            md.attrs[p + "circumference_m"] = RING_RADIUS_M[i] * 2.0 * math.pi
            md.attrs[p + "lambda_res_m"]    = RING_LAMBDA_RES_M[i]
            md.attrs[p + "neff_TE"]         = RING_NEFF_TE[i]
            md.attrs[p + "ng_TE"]           = RING_NG_TE[i]
            md.attrs[p + "kappa_input_sq"]  = RING_KAPPA_INPUT_SQ[i]
            md.attrs[p + "kappa_drop_sq"]   = RING_KAPPA_DROP_SQ[i]
            md.attrs[p + "loss_dB_per_m"]   = RING_LOSS_DB_PER_M[i]

        for k in range(2, N_RINGS + 1):
            md.attrs[f"drop{k-1}_ring"]    = f"RING_{k} output 2 (drop) → ONA input {k}"

        rg = f.create_group("results")
        # Sensor through and final through spectra
        rg.create_dataset("T_sensor_through_dB",
                          data=np.full((n_pts, n_wl), np.nan), chunks=(1, n_wl))
        rg.create_dataset("T_final_through_dB",
                          data=np.full((n_pts, n_wl), np.nan), chunks=(1, n_wl))
        # Drop spectra: (sweep_pts, N_DROPS, n_wl)
        rg.create_dataset("T_drop_dB",
                          data=np.full((n_pts, N_DROPS, n_wl), np.nan),
                          chunks=(1, 1, n_wl))
        # Integrated detector power per drop: (sweep_pts, N_DROPS)
        rg.create_dataset("drop_power_dBm",
                          data=np.full((n_pts, N_DROPS), np.nan),
                          chunks=(1, N_DROPS))

        f.create_group("flags").create_dataset(
            "computed", data=np.zeros(n_pts, dtype=bool), chunks=(1,))

    log.info(f"HDF5 V3 initialised ({N_DROPS} drops, {n_wl} wavelengths) → {HDF5_PATH}")


# ─────────────────────────────────────────────────────────────────────────────
# Main sweep
# ─────────────────────────────────────────────────────────────────────────────
def run_interconnect_sweep(hide_gui: bool = False) -> Dict[str, Any]:
    n_pts = SWEEP_N_POINTS
    wavelengths_m         = None
    T_sensor_through_dB   = None
    T_final_through_dB    = None
    T_drop_dB             = None
    drop_power_dBm        = None
    computed   = np.zeros(n_pts, dtype=bool)
    hdf5_ready = False

    # ── Resume from cache ─────────────────────────────────────────────────────
    if HDF5_PATH.exists():
        log.info(f"Cache found → {HDF5_PATH}")
        try:
            with h5py.File(HDF5_PATH, "r") as f:
                wavelengths_m       = f["metadata/wavelengths_m"][:]
                T_sensor_through_dB = f["results/T_sensor_through_dB"][:]
                T_final_through_dB  = f["results/T_final_through_dB"][:]
                T_drop_dB           = f["results/T_drop_dB"][:]
                drop_power_dBm      = f["results/drop_power_dBm"][:]
                computed[:]         = f["flags/computed"][:]
            hdf5_ready = True
            n_cached   = int(computed.sum())
            log.info(f"Cached: {n_cached}/{n_pts}  |  Remaining: {n_pts - n_cached}")
            if n_pts - n_cached == 0:
                log.info("All sweep points cached — skipping INTERCONNECT launch.")
                return _pack_results(wavelengths_m, T_sensor_through_dB,
                                     T_final_through_dB, T_drop_dB,
                                     drop_power_dBm, computed)
        except Exception as exc:
            log.warning(f"Cache unreadable ({exc}). Starting fresh.")
            wavelengths_m = T_sensor_through_dB = T_final_through_dB = None
            T_drop_dB = drop_power_dBm = None
            computed[:] = False
            hdf5_ready  = False
    else:
        log.info("No cache — starting fresh sweep.")

    log.info("Launching INTERCONNECT …")
    ic         = lumapi.INTERCONNECT(hide=hide_gui)
    runs_done  = 0
    runs_total = int((~computed).sum())
    t_start    = time.time()

    try:
        _build_circuit(ic)
        log.info(f"Circuit ready — {runs_total} sweep points to compute …")

        for s_idx in range(n_pts):
            if computed[s_idx]:
                continue

            neff_val = float(SWEEP_NEFF[s_idx])
            ng_val   = float(SWEEP_NG[s_idx])

            _eval(ic, "switchtodesign")
            _update_ring1_neff_ng(ic, neff_val, ng_val)

            # ── Run ───────────────────────────────────────────────────────────
            try:
                _eval(ic, "run")
            except ICScriptError as exc:
                log.warning(
                    f"  RUN FAILED  pt={s_idx:3d}  "
                    f"neff={neff_val:.6f}  ng={ng_val:.6f}  →  {exc}"
                )
                computed[s_idx] = True
                if hdf5_ready:
                    with h5py.File(HDF5_PATH, "r+") as hf:
                        hf["flags/computed"][s_idx] = True
                        hf.flush()
                continue

            # ── Extract ───────────────────────────────────────────────────────
            try:
                wl_m, t_sen, t_fin, t_drop, p_drop = _extract_results(ic)
            except Exception as exc:
                log.warning(f"  EXTRACT FAILED  pt={s_idx:3d}: {exc}")
                computed[s_idx] = True
                continue

            # ── Initialise arrays on first valid point ────────────────────────
            if wavelengths_m is None:
                n_wl                = len(wl_m)
                wavelengths_m       = wl_m
                T_sensor_through_dB = np.full((n_pts, n_wl),          np.nan)
                T_final_through_dB  = np.full((n_pts, n_wl),          np.nan)
                T_drop_dB           = np.full((n_pts, N_DROPS, n_wl), np.nan)
                drop_power_dBm      = np.full((n_pts, N_DROPS),        np.nan)
                if not hdf5_ready:
                    _init_hdf5(wl_m)
                    hdf5_ready = True

            # ── Store in memory ───────────────────────────────────────────────
            T_sensor_through_dB[s_idx, :]    = t_sen
            T_final_through_dB [s_idx, :]    = t_fin
            T_drop_dB          [s_idx, :, :] = t_drop
            drop_power_dBm     [s_idx, :]    = p_drop
            computed           [s_idx]        = True

            # ── Flush to HDF5 ─────────────────────────────────────────────────
            with h5py.File(HDF5_PATH, "r+") as hf:
                hf["results/T_sensor_through_dB"][s_idx, :]    = t_sen
                hf["results/T_final_through_dB"] [s_idx, :]    = t_fin
                hf["results/T_drop_dB"]          [s_idx, :, :] = t_drop
                hf["results/drop_power_dBm"]     [s_idx, :]    = p_drop
                hf["flags/computed"]             [s_idx]        = True
                hf.flush()

            runs_done += 1
            if runs_done % 5 == 0 or runs_done == runs_total:
                elapsed = time.time() - t_start
                rate    = runs_done / elapsed if elapsed > 0 else 1e-9
                eta     = (runs_total - runs_done) / rate
                log.info(
                    f"  [{runs_done:3d}/{runs_total}]  "
                    f"neff={neff_val:.6f}  ng={ng_val:.6f}  │  "
                    f"{rate:.2f} sim/s  │  ETA {eta:5.0f} s"
                )

        if hdf5_ready:
            with h5py.File(HDF5_PATH, "r+") as hf:
                hf["metadata"].attrs["timestamp_end"]  = datetime.now().isoformat()
                hf["metadata"].attrs["runs_completed"] = int(computed.sum())

    finally:
        try:
            ic.close()
        except Exception:
            pass
        log.info("INTERCONNECT session closed.")

    elapsed = time.time() - t_start
    log.info(
        f"Sweep done │ {runs_done} new runs │ "
        f"total={elapsed:.1f} s │ avg={elapsed/max(runs_done, 1):.2f} s/sim"
    )
    return _pack_results(wavelengths_m, T_sensor_through_dB,
                         T_final_through_dB, T_drop_dB, drop_power_dBm, computed)


def _pack_results(wl, t_sen, t_fin, t_drop, p_drop, comp) -> Dict[str, Any]:
    return dict(
        neff_sweep            = SWEEP_NEFF,
        ng_sweep              = SWEEP_NG,
        wavelengths_m         = wl,
        T_sensor_through_dB   = t_sen,
        T_final_through_dB    = t_fin,
        T_drop_dB             = t_drop,
        drop_power_dBm        = p_drop,
        computed              = comp,
    )


# ── Execute sweep ──────────────────────────────────────────────────────────────
sweep_results = run_interconnect_sweep(hide_gui=False)

neff_sweep          = sweep_results["neff_sweep"]
ng_sweep            = sweep_results["ng_sweep"]
wavelengths_m       = sweep_results["wavelengths_m"]
T_sensor_through_dB = sweep_results["T_sensor_through_dB"]
T_final_through_dB  = sweep_results["T_final_through_dB"]
T_drop_dB           = sweep_results["T_drop_dB"]
drop_power_dBm      = sweep_results["drop_power_dBm"]
computed            = sweep_results["computed"]
wavelengths_nm      = wavelengths_m * 1e9 if wavelengths_m is not None else None

print(f"\n  Sweep complete — {computed.sum()} / {len(computed)} pts computed")
if wavelengths_m is not None:
    print(f"  T_sensor_through_dB  shape : {T_sensor_through_dB.shape}")
    print(f"  T_final_through_dB   shape : {T_final_through_dB.shape}")
    print(f"  T_drop_dB            shape : {T_drop_dB.shape}   (RING_2..14 drops)")
    print(f"  drop_power_dBm       shape : {drop_power_dBm.shape}")
print(f"  HDF5                        : {HDF5_PATH}")

# ═════════════════════════════════════════════════════════════════════════════
#  END CELL 3
# ═════════════════════════════════════════════════════════════════════════════


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Post-processing · Visualisation  [V3]                            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

plt.rcParams.update({
    "font.family"    : "serif",
    "font.serif"     : ["DejaVu Serif", "Georgia", "Times New Roman"],
    "font.size"      : 11, "axes.labelsize": 12, "axes.titlesize": 13,
    "legend.fontsize": 9,  "xtick.labelsize": 10, "ytick.labelsize": 10,
    "axes.linewidth" : 0.8, "axes.grid": True, "grid.alpha": 0.3,
    "grid.linewidth" : 0.5, "lines.linewidth": 1.4,
    "figure.dpi"     : 120, "savefig.dpi": 300, "savefig.bbox": "tight",
})

# ── Spectrometer ring labels (drop index 0-based → RING 2..14) ───────────────
DROP_LABELS = [f"RING_{k+2}" for k in range(N_DROPS)]   # RING_2 .. RING_14


# ─────────────────────────────────────────────────────────────────────────────
# Data loader helpers
# ─────────────────────────────────────────────────────────────────────────────
def load_results(path: Path = HDF5_PATH) -> Optional[Dict[str, Any]]:
    if not path.exists():
        return None
    try:
        with h5py.File(path, "r") as f:
            if "metadata/wavelengths_m" not in f:
                return None
            wl = f["metadata/wavelengths_m"][:]
            if wl is None or len(wl) == 0:
                return None
            return dict(
                neff_sweep            = f["metadata/neff_sweep"][:],
                ng_sweep              = f["metadata/ng_sweep"][:],
                wavelengths_m         = wl,
                T_sensor_through_dB   = f["results/T_sensor_through_dB"][:],
                T_final_through_dB    = f["results/T_final_through_dB"][:],
                T_drop_dB             = f["results/T_drop_dB"][:],
                drop_power_dBm        = f["results/drop_power_dBm"][:],
                computed              = f["flags/computed"][:],
            )
    except Exception as exc:
        log.warning(f"Could not read HDF5 ({exc}): {path}")
        return None


def get_results(path: Path = HDF5_PATH) -> Dict[str, Any]:
    mem_reason = ""
    try:
        r  = sweep_results
        wl = r.get("wavelengths_m")
        if wl is not None and len(wl) > 0:
            return r
        mem_reason = "sweep_results in memory but wavelengths_m is None"
    except NameError:
        mem_reason = "sweep_results not defined (Cell 3 not run)"
    r_hdf5 = load_results(path)
    if r_hdf5 is not None:
        log.info(f"Loaded from HDF5: {int(r_hdf5['computed'].sum())}/{SWEEP_N_POINTS} pts.")
        return r_hdf5
    hdf5_reason = (
        f"HDF5 exists but empty — delete and re-run Cell 3:\n    {path}"
        if path.exists() else f"HDF5 not found:\n    {path}"
    )
    raise RuntimeError(
        f"\n{'='*65}\n  No results available.\n"
        f"  Memory : {mem_reason}\n  HDF5   : {hdf5_reason}\n"
        f"  ► Run Cell 3.\n{'='*65}"
    )


def _valid_mask(r: Dict) -> np.ndarray:
    return r["computed"].astype(bool)


# ─────────────────────────────────────────────────────────────────────────────
# Plot 1 — ★ MAIN RESULT: Integrated detector power vs neff for all 13 drops ★
# ─────────────────────────────────────────────────────────────────────────────
def plot_drop_power_vs_neff(
    results=None, figsize=(11, 6), save: bool = True,
) -> plt.Figure:
    """
    Power vs neff — primary sensor readout plot.

    Each curve is the integrated detector power [dBm] at the drop port
    of one spectrometer ring as the sensor ring (RING_1) sweeps its neff.

    Physical meaning:
      As RING_1 neff increases, its resonance shifts to longer wavelengths.
      Each spectrometer ring has a fixed resonance at its design wavelength.
      When the sensor resonance approaches/crosses a spectrometer resonance,
      the drop power of that spectrometer ring changes — its unique response
      to the sensor shift.  Reading all 13 drops gives a power vector that
      encodes the sensor state with high redundancy.

    P_det [dBm] = 10 log10(P_source [mW] × mean_λ(|T_drop(λ)|²))
    """
    if results is None:
        results = get_results()
    neff_arr  = results["neff_sweep"]
    p_drop    = results["drop_power_dBm"]   # (n_pts, N_DROPS)
    mask      = _valid_mask(results)
    neff_v    = neff_arr[mask]
    p_v       = p_drop[mask, :]             # (n_valid, N_DROPS)

    cmap = plt.get_cmap("tab20")
    fig, ax = plt.subplots(figsize=figsize)

    for k in range(N_DROPS):
        ring_label = DROP_LABELS[k]
        ax.plot(
            neff_v, p_v[:, k],
            color=cmap(k / N_DROPS), lw=1.5,
            marker="o", ms=2.5, alpha=0.85,
            label=ring_label,
        )

    ax.set_xlabel("Ring 1 — $n_{eff}$ (TE)  [sensor]", fontsize=12)
    ax.set_ylabel("Detector power  (dBm)", fontsize=12)
    ax.set_title(
        "Integrated Drop Power vs Sensor $n_{eff}$  [V3 — ONA multiport]\n"
        r"$P_{det} = P_{src} \cdot \langle|T_{drop}(\lambda)|^2\rangle_\lambda$"
        "   —   13 spectrometer rings",
        fontsize=12,
    )
    ax.legend(ncol=3, framealpha=0.88, fontsize=8, loc="best")
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()

    if save:
        fig.savefig(FIGURES_DIR / "drop_power_vs_neff_all.png")
        fig.savefig(FIGURES_DIR / "drop_power_vs_neff_all.pdf")
        log.info("Saved → drop_power_vs_neff_all.png/pdf")
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Plot 2 — Sensor through-transmission spectra vs neff (coloured by neff)
# ─────────────────────────────────────────────────────────────────────────────
def plot_sensor_through_sweep(
    results=None, n_curves: int = 200,
    figsize=(10, 5), cmap_name: str = "plasma", save: bool = True,
) -> plt.Figure:
    """
    ONA input 1 (RING_1 through) transmission spectra.
    Colour encodes the sensor neff value for each sweep point.
    """
    if results is None:
        results = get_results()
    neff_arr = results["neff_sweep"]
    wl_nm    = results["wavelengths_m"] * 1e9
    T_data   = results["T_sensor_through_dB"]
    mask     = _valid_mask(results)
    valid_idx = np.where(mask)[0]
    n_sel    = min(n_curves, len(valid_idx))
    sel_idx  = valid_idx[
        np.round(np.linspace(0, len(valid_idx) - 1, n_sel)).astype(int)
    ]
    cmap = plt.get_cmap(cmap_name)
    norm = Normalize(vmin=neff_arr[sel_idx].min(), vmax=neff_arr[sel_idx].max())
    sm   = ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])

    fig, ax = plt.subplots(figsize=figsize)
    for idx in sel_idx:
        ax.plot(wl_nm, T_data[idx], color=cmap(norm(neff_arr[idx])),
                lw=0.8, alpha=0.70)
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label("Ring 1 — $n_{eff}$ (TE)", fontsize=10)
    ax.set_xlabel("Wavelength  (nm)")
    ax.set_ylabel("Transmission  (dB)")
    ax.set_title(
        f"Sensor Through Spectrum — ONA input 1  (RING_1 through)\n"
        f"({n_sel} curves,  colour = $n_{{eff,1}}$)"
    )
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()
    if save:
        stem = f"sensor_through_sweep_{n_sel}curves"
        fig.savefig(FIGURES_DIR / f"{stem}.png")
        fig.savefig(FIGURES_DIR / f"{stem}.pdf")
        log.info(f"Saved → {stem}.png/pdf")
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Plot 3 — Cascade final through (ONA input 15) spectra vs neff
# ─────────────────────────────────────────────────────────────────────────────
def plot_final_through_sweep(
    results=None, n_curves: int = 200,
    figsize=(10, 5), cmap_name: str = "viridis", save: bool = True,
) -> plt.Figure:
    """
    ONA input 15 (RING_14 through) transmission spectra — cascade output.
    """
    if results is None:
        results = get_results()
    neff_arr  = results["neff_sweep"]
    wl_nm     = results["wavelengths_m"] * 1e9
    T_data    = results["T_final_through_dB"]
    mask      = _valid_mask(results)
    valid_idx = np.where(mask)[0]
    n_sel     = min(n_curves, len(valid_idx))
    sel_idx   = valid_idx[
        np.round(np.linspace(0, len(valid_idx) - 1, n_sel)).astype(int)
    ]
    cmap = plt.get_cmap(cmap_name)
    norm = Normalize(vmin=neff_arr[sel_idx].min(), vmax=neff_arr[sel_idx].max())
    sm   = ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])

    fig, ax = plt.subplots(figsize=figsize)
    for idx in sel_idx:
        ax.plot(wl_nm, T_data[idx], color=cmap(norm(neff_arr[idx])),
                lw=0.8, alpha=0.70)
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label("Ring 1 — $n_{eff}$ (TE)", fontsize=10)
    ax.set_xlabel("Wavelength  (nm)")
    ax.set_ylabel("Transmission  (dB)")
    ax.set_title(
        f"Cascade Final Through — ONA input 15  (RING_14 through)\n"
        f"({n_sel} curves,  colour = $n_{{eff,1}}$)"
    )
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()
    if save:
        stem = f"final_through_sweep_{n_sel}curves"
        fig.savefig(FIGURES_DIR / f"{stem}.png")
        fig.savefig(FIGURES_DIR / f"{stem}.pdf")
        log.info(f"Saved → {stem}.png/pdf")
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Plot 4 — Drop transmission spectrum heatmap for a single spectrometer ring
# ─────────────────────────────────────────────────────────────────────────────
def plot_drop_spectrum_heatmap(
    results=None, drop_k: int = 1,
    figsize=(10, 3.5), cmap_name: str = "inferno",
    vmin_dB=None, vmax_dB=None, save: bool = True,
) -> plt.Figure:
    """
    Heatmap: neff (y) × wavelength (x) of drop transmission [dB].
    drop_k is 1-based (1..13):
      drop_k=1  → RING_2 drop  (ONA input 2)
      drop_k=13 → RING_14 drop (ONA input 14)
    """
    if results is None:
        results = get_results()
    assert 1 <= drop_k <= N_DROPS, f"drop_k must be 1..{N_DROPS}"
    ring_label = DROP_LABELS[drop_k - 1]

    neff_arr = results["neff_sweep"]
    wl_nm    = results["wavelengths_m"] * 1e9
    t_drop   = results["T_drop_dB"]            # (n_pts, N_DROPS, n_wl)
    mask     = _valid_mask(results)
    neff_v   = neff_arr[mask]
    spec_v   = t_drop[mask, drop_k - 1, :]     # (n_valid, n_wl)

    _vmin = vmin_dB if vmin_dB is not None else np.nanpercentile(spec_v, 2)
    _vmax = vmax_dB if vmax_dB is not None else np.nanpercentile(spec_v, 98)

    fig, ax = plt.subplots(figsize=figsize)
    img = ax.pcolormesh(wl_nm, neff_v, spec_v,
                        cmap=cmap_name, vmin=_vmin, vmax=_vmax, shading="auto")
    cbar = fig.colorbar(img, ax=ax, pad=0.01)
    cbar.set_label("Transmission  (dB)", fontsize=10)
    ax.set_xlabel("Wavelength  (nm)")
    ax.set_ylabel("Ring 1 — $n_{eff}$ (TE)")
    ax.set_title(
        f"Drop Spectrum Heatmap — {ring_label}  (ONA input {drop_k + 1})\n"
        f"neff sweep on sensor ring (RING_1)"
    )
    fig.tight_layout()
    if save:
        fname = f"drop{drop_k}_spectrum_heatmap"
        fig.savefig(FIGURES_DIR / f"{fname}.png")
        fig.savefig(FIGURES_DIR / f"{fname}.pdf")
        log.info(f"Saved → {fname}.png/pdf")
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Plot 5 — All 13 drop spectrum heatmaps in a grid
# ─────────────────────────────────────────────────────────────────────────────
def plot_all_drop_heatmaps(
    results=None, ncols: int = 4,
    cmap_name: str = "inferno", save: bool = True,
) -> plt.Figure:
    if results is None:
        results = get_results()
    neff_arr = results["neff_sweep"]
    wl_nm    = results["wavelengths_m"] * 1e9
    t_drop   = results["T_drop_dB"]
    mask     = _valid_mask(results)
    neff_v   = neff_arr[mask]

    nrows = math.ceil(N_DROPS / ncols)
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(ncols * 3.8, nrows * 2.8),
                             sharex=True, sharey=True)
    axes_flat = axes.flatten()

    all_vals = t_drop[mask].ravel()
    vmin = np.nanpercentile(all_vals, 2)
    vmax = np.nanpercentile(all_vals, 98)

    im = None
    for k in range(1, N_DROPS + 1):
        ax    = axes_flat[k - 1]
        spec  = t_drop[mask, k - 1, :]
        im    = ax.pcolormesh(wl_nm, neff_v, spec,
                              cmap=cmap_name, vmin=vmin, vmax=vmax, shading="auto")
        ax.set_title(f"{DROP_LABELS[k-1]} drop", fontsize=8)
        if k % ncols == 1:
            ax.set_ylabel("$n_{eff,1}$", fontsize=7)
        if k > (nrows - 1) * ncols:
            ax.set_xlabel("λ (nm)", fontsize=7)
        ax.tick_params(labelsize=6)

    for ax in axes_flat[N_DROPS:]:
        ax.set_visible(False)

    if im is not None:
        fig.colorbar(im, ax=axes_flat[:N_DROPS], shrink=0.6, pad=0.02,
                     label="Transmission (dB)", fraction=0.015)
    fig.suptitle(
        "All Spectrometer Drop Spectra  [V3 — ONA multiport]\n"
        "neff sweep on sensor ring (RING_1)",
        fontsize=11,
    )
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR / "all_drop_heatmaps_grid.png", dpi=200)
        fig.savefig(FIGURES_DIR / "all_drop_heatmaps_grid.pdf")
        log.info("Saved → all_drop_heatmaps_grid.png/pdf")
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Plot 6 — Power heatmap: drop index × neff (shows all drops simultaneously)
# ─────────────────────────────────────────────────────────────────────────────
def plot_power_heatmap_drops_vs_neff(
    results=None, figsize=(10, 5), cmap_name: str = "plasma", save: bool = True,
) -> plt.Figure:
    """
    2-D heatmap: X = neff sweep, Y = spectrometer ring index (1-13),
    colour = integrated detector power [dBm].

    Shows at a glance which ring is "lit up" at each sensor state.
    """
    if results is None:
        results = get_results()
    neff_arr = results["neff_sweep"]
    p_drop   = results["drop_power_dBm"]     # (n_pts, N_DROPS)
    mask     = _valid_mask(results)
    neff_v   = neff_arr[mask]
    p_v      = p_drop[mask, :].T             # (N_DROPS, n_valid)

    fig, ax = plt.subplots(figsize=figsize)
    img = ax.pcolormesh(
        neff_v,
        np.arange(1, N_DROPS + 1),
        p_v,
        cmap=cmap_name, shading="auto",
    )
    cbar = fig.colorbar(img, ax=ax, pad=0.02)
    cbar.set_label("Detector power  (dBm)", fontsize=10)
    ax.set_xlabel("Ring 1 — $n_{eff}$ (TE)  [sensor]", fontsize=11)
    ax.set_ylabel("Spectrometer ring index  (1 = RING_2 … 13 = RING_14)", fontsize=10)
    ax.set_yticks(np.arange(1, N_DROPS + 1))
    ax.set_yticklabels(DROP_LABELS, fontsize=7)
    ax.set_title(
        "Detector Power Heatmap — All Drops vs Sensor $n_{eff}$  [V3]\n"
        r"Colour: $P_{det}$ [dBm]  per spectrometer ring",
        fontsize=12,
    )
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR / "power_heatmap_drops_vs_neff.png", dpi=200)
        fig.savefig(FIGURES_DIR / "power_heatmap_drops_vs_neff.pdf")
        log.info("Saved → power_heatmap_drops_vs_neff.png/pdf")
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Plot 7 — Resonance tracking on sensor through (ONA input 1)
# ─────────────────────────────────────────────────────────────────────────────
def plot_resonance_tracking(
    results=None, figsize=(7, 4.5), save: bool = True,
) -> plt.Figure:
    if results is None:
        results = get_results()
    neff_arr = results["neff_sweep"]
    wl_nm    = results["wavelengths_m"] * 1e9
    T_data   = results["T_sensor_through_dB"]
    mask     = _valid_mask(results)
    neff_v   = neff_arr[mask]
    T_v      = T_data[mask, :]
    dip_idx  = np.argmin(T_v, axis=1)
    lam_dip  = wl_nm[dip_idx]
    coeffs   = np.polyfit(neff_v, lam_dip, 1)
    sens     = coeffs[0]

    fig, ax = plt.subplots(figsize=figsize)
    ax.scatter(neff_v, lam_dip, s=20, zorder=5, color="#2166ac", label="Resonance dip")
    ax.plot(neff_v, np.poly1d(coeffs)(neff_v), "r--", lw=1.5,
            label=f"Linear fit   ∂λ/∂n = {sens:.3f} nm / RIU")
    ax.set_xlabel("Ring 1 — $n_{eff}$ (TE)")
    ax.set_ylabel("Resonance wavelength  (nm)")
    ax.set_title(
        f"Resonance Tracking — Sensor Through (ONA input 1)\n"
        f"Sensitivity: {sens:.3f} nm / RIU"
    )
    ax.legend(framealpha=0.9)
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()
    if save:
        fname = "resonance_tracking_sensor"
        fig.savefig(FIGURES_DIR / f"{fname}.png")
        fig.savefig(FIGURES_DIR / f"{fname}.pdf")
        log.info(f"Saved → {fname}.png/pdf")
    log.info(f"Resonance sensitivity: {sens:.4f} nm/RIU")
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Run all plots
# ─────────────────────────────────────────────────────────────────────────────
try:
    _res = get_results()
except RuntimeError as exc:
    print(str(exc))
    raise
else:
    # ── Plot 1: PRIMARY — integrated power vs neff for all 13 drops ──────────
    fig1 = plot_drop_power_vs_neff(_res)

    # ── Plot 2: Sensor through spectra coloured by neff ───────────────────────
    fig2 = plot_sensor_through_sweep(_res, n_curves=200)

    # ── Plot 3: Cascade final through spectra ─────────────────────────────────
    fig3 = plot_final_through_sweep(_res, n_curves=200)

    # ── Plot 4: Drop heatmaps — first and last spectrometer rings ────────────
    fig4 = plot_drop_spectrum_heatmap(_res, drop_k=1)    # RING_2 drop
    fig5 = plot_drop_spectrum_heatmap(_res, drop_k=13)   # RING_14 drop

    # ── Plot 5: All 13 drop heatmaps in a grid ───────────────────────────────
    fig6 = plot_all_drop_heatmaps(_res)

    # ── Plot 6: 2-D power heatmap (rings × neff) ─────────────────────────────
    fig7 = plot_power_heatmap_drops_vs_neff(_res)

    # ── Plot 7: Resonance wavelength tracking ────────────────────────────────
    fig8 = plot_resonance_tracking(_res)

    plt.show()
    print(f"\n  Figures → {FIGURES_DIR}")
    print(f"  HDF5    → {HDF5_PATH}")

# ═════════════════════════════════════════════════════════════════════════════
#  END CELL 4
# ═════════════════════════════════════════════════════════════════════════════

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Cálculo teórico de resonancias y comparación con INTERCONNECT    ║
# ║                                                                             ║
# ║  OBJETIVO                                                                   ║
# ║  ─────────────────────────────────────────────────────────────────────────  ║
# ║  Contrastar las longitudes de onda de resonancia teóricas (calculadas      ║
# ║  desde neff, ng y la circunferencia del anillo) con las obtenidas en       ║
# ║  INTERCONNECT, para diagnosticar el shift observado.                        ║
# ║                                                                             ║
# ║  DEPENDENCIAS HEREDADAS DEL SCRIPT PRINCIPAL                               ║
# ║  ─────────────────────────────────────────────────────────────────────────  ║
# ║  Todas las variables de parámetros del código anterior se reutilizan       ║
# ║  directamente (no se re-definen aquí):                                     ║
# ║    RING_RADIUS_M, RING_LAMBDA_RES_M, RING_NEFF_TE, RING_NG_TE,            ║
# ║    N_RINGS, SPEED_OF_LIGHT, wavelengths_m, T_port1_dB, T_port2_dB,        ║
# ║    computed, SWEEP_NEFF, SWEEP_NG, FIGURES_DIR                             ║
# ║                                                                             ║
# ║  FÍSICA UTILIZADA                                                           ║
# ║  ─────────────────────────────────────────────────────────────────────────  ║
# ║  Condición de resonancia de un anillo:                                     ║
# ║      λ_res = (neff × L) / m     [m: orden de modo, entero]                ║
# ║                                                                             ║
# ║  Para encontrar m a partir de los datos de diseño (neff, L, λ_target):    ║
# ║      m_real   = (neff × L) / λ_target                                      ║
# ║      m_near   = round(m_real)           ← entero más cercano               ║
# ║      m_floor  = floor(m_real)           ← entero inmediatamente inferior   ║
# ║                                                                             ║
# ║  Longitud de resonancia teórica para cada m candidato:                    ║
# ║      λ_theo(m) = (neff × L) / m                                            ║
# ║                                                                             ║
# ║  FSR teórico:                                                               ║
# ║      FSR_theo = λ_res² / (ng × L)                                          ║
# ║                                                                             ║
# ║  NOTA SOBRE EL ANILLO SENSOR                                               ║
# ║  ─────────────────────────────────────────────────────────────────────────  ║
# ║  Para el barrido paramétrico, se usa SWEEP_NEFF[0] / SWEEP_NG[0] como     ║
# ║  estado de referencia inicial del sensor (primer punto del sweep).         ║
# ║  El neff/ng de diseño nominal (RING_NEFF_TE[0]/RING_NG_TE[0]) también     ║
# ║  se muestra para comparación.                                              ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ─────────────────────────────────────────────────────────────────────────────
# Parámetros de diseño de la tabla original (TABLE 1)
# Solo se definen aquí los valores que NO provienen del código anterior:
#   FSR de diseño [nm] por anillo (en el mismo orden que RING_RADIUS_M)
# ─────────────────────────────────────────────────────────────────────────────
# FSR_DESIGN_NM[0]  = Sensor  (Ring 1)
# FSR_DESIGN_NM[1]  = Spec_00 (Ring 2)  ...  FSR_DESIGN_NM[13] = Spec_12 (Ring 14)
FSR_DESIGN_NM = np.array([
     9.9590,   # Sensor
    10.0116,   # Spec_00
    10.0161,   # Spec_01
    10.0207,   # Spec_02
    10.0253,   # Spec_03
    10.0299,   # Spec_04
    10.0346,   # Spec_05
    10.0392,   # Spec_06
     9.9650,   # Spec_07
     9.9696,   # Spec_08
     9.9742,   # Spec_09
     9.9783,   # Spec_10  ← "9.9783" interpolado entre 9.9742 y 9.9829
     9.9829,   # Spec_11
     9.9875,   # Spec_12
])

# Etiquetas de los anillos
RING_LABELS = (
    ["Sensor"] +
    [f"Spec_{i:02d}" for i in range(N_RINGS - 1)]
)

# ─────────────────────────────────────────────────────────────────────────────
# Para el anillo sensor se usan los valores del PRIMER PUNTO DEL SWEEP
# (estado de referencia: cladding = estado inicial de la medición)
# ─────────────────────────────────────────────────────────────────────────────
neff_sensor_sweep0 = float(SWEEP_NEFF[0])
ng_sensor_sweep0   = float(SWEEP_NG[0])

# Arrays de neff / ng que se usarán para el cálculo teórico (por anillo)
# Ring 0 (sensor) → primer punto del sweep
# Rings 1-13      → valores fijos de diseño (RING_NEFF_TE / RING_NG_TE)
neff_calc = RING_NEFF_TE.copy().astype(float)
ng_calc   = RING_NG_TE.copy().astype(float)
neff_calc[0] = neff_sensor_sweep0
ng_calc[0]   = ng_sensor_sweep0

# ─────────────────────────────────────────────────────────────────────────────
# Circunferencias [m]  (ya definidas en el código anterior como RING_RADIUS_M)
# ─────────────────────────────────────────────────────────────────────────────
L_m = RING_RADIUS_M * 2.0 * math.pi   # circumference [m]  shape (14,)

# ─────────────────────────────────────────────────────────────────────────────
# Longitudes de onda de resonancia de diseño [m]
# (heredadas directamente: RING_LAMBDA_RES_M)
# ─────────────────────────────────────────────────────────────────────────────
lambda_design_m  = RING_LAMBDA_RES_M.copy()
lambda_design_nm = lambda_design_m * 1e9

# ─────────────────────────────────────────────────────────────────────────────
# Cálculo teórico
# ─────────────────────────────────────────────────────────────────────────────
# Número de onda real (continuo) correspondiente al diseño
m_real  = (neff_calc * L_m) / lambda_design_m          # flotante
m_near  = np.round(m_real).astype(int)                 # entero más cercano
m_floor = np.floor(m_real).astype(int)                 # entero inferior

# λ_theo para m_near y m_floor [nm]
lambda_theo_near_m  = (neff_calc * L_m) / m_near
lambda_theo_floor_m = (neff_calc * L_m) / m_floor
lambda_theo_near_nm  = lambda_theo_near_m  * 1e9
lambda_theo_floor_nm = lambda_theo_floor_m * 1e9

# FSR teórico [nm]  — calculado en el estado de diseño
fsr_theo_near_nm  = (lambda_theo_near_nm**2)  / (ng_calc * L_m * 1e9)
fsr_theo_floor_nm = (lambda_theo_floor_nm**2) / (ng_calc * L_m * 1e9)

# FSR de diseño
fsr_design_nm = FSR_DESIGN_NM.copy()

# ─────────────────────────────────────────────────────────────────────────────
# Longitud de onda de resonancia extraída de la simulación INTERCONNECT
# Se usa el mínimo de T_port1_dB en el PRIMER punto válido del sweep
# (índice s=0 si está computado, si no el primer punto válido)
# ─────────────────────────────────────────────────────────────────────────────
# ONA port1 monitorea RING_1 through — el dip principal corresponde a
# la resonancia de RING_1 (sensor) en su estado sweep[0].
# ONA port2 monitorea RING_14 through — refleja la resonancia acumulada
# de toda la cascada.

lambda_ic_port1_nm = None
lambda_ic_port2_nm = None

try:
    # Usamos el primer punto del sweep (índice 0) — mismo estado que neff_calc[0]
    if computed[0] and wavelengths_m is not None:
        wl_nm_ref = wavelengths_m * 1e9
        dip1_idx  = int(np.argmin(T_port1_dB[0, :]))
        dip2_idx  = int(np.argmin(T_port2_dB[0, :]))
        lambda_ic_port1_nm = float(wl_nm_ref[dip1_idx])
        lambda_ic_port2_nm = float(wl_nm_ref[dip2_idx])
        print(f"  INTERCONNECT (sweep pt 0) — ONA port1 dip : {lambda_ic_port1_nm:.4f} nm")
        print(f"  INTERCONNECT (sweep pt 0) — ONA port2 dip : {lambda_ic_port2_nm:.4f} nm")
    else:
        print("  [AVISO] computed[0] = False o wavelengths_m no disponible.")
        print("          La columna 'IC (ONA p1)' aparecerá como N/A.")
except Exception as exc:
    print(f"  [AVISO] No se pudo extraer dip de INTERCONNECT: {exc}")

# ─────────────────────────────────────────────────────────────────────────────
# Construcción del DataFrame de comparación
# ─────────────────────────────────────────────────────────────────────────────
rows = []
for i in range(N_RINGS):
    # Diferencias λ_theo − λ_design
    delta_near_nm  = lambda_theo_near_nm[i]  - lambda_design_nm[i]
    delta_floor_nm = lambda_theo_floor_nm[i] - lambda_design_nm[i]

    # Error fraccional del m_real respecto al m_near
    frac_err = m_real[i] - m_near[i]   # cuán "descentrado" está m_real

    row = {
        "Ring"              : RING_LABELS[i],
        "λ_design (nm)"     : round(lambda_design_nm[i], 4),
        "neff"              : round(neff_calc[i], 6),
        "ng"                : round(ng_calc[i], 6),
        "L (µm)"            : round(L_m[i] * 1e6, 4),
        "m_real"            : round(m_real[i], 6),
        "m_near"            : int(m_near[i]),
        "λ_theo_mNear (nm)" : round(lambda_theo_near_nm[i], 4),
        "Δλ_mNear (pm)"     : round(delta_near_nm * 1e3, 2),
        "m_floor"           : int(m_floor[i]),
        "λ_theo_mFloor (nm)": round(lambda_theo_floor_nm[i], 4),
        "Δλ_mFloor (pm)"    : round(delta_floor_nm * 1e3, 2),
        "FSR_design (nm)"   : round(fsr_design_nm[i], 4),
        "FSR_theo_mNear (nm)": round(fsr_theo_near_nm[i], 4),
        "FSR_delta (pm)"    : round((fsr_theo_near_nm[i] - fsr_design_nm[i]) * 1e3, 2),
        "m_frac_err"        : round(frac_err, 6),
    }
    rows.append(row)

df = pd.DataFrame(rows)
df.set_index("Ring", inplace=True)

# ─────────────────────────────────────────────────────────────────────────────
# Impresión en consola — tabla de diagnóstico completa
# ─────────────────────────────────────────────────────────────────────────────
SEP = "═" * 130
print()
print(SEP)
print("  TABLA DE COMPARACIÓN TEÓRICA vs DISEÑO  │  14-Ring Cascade  │  SiN 400 nm × 1000 nm")
print(f"  Anillo sensor (Ring 1): neff = {neff_sensor_sweep0:.6f}  ng = {ng_sensor_sweep0:.6f}  "
      f"(primer punto del sweep; diseño: neff={RING_NEFF_TE[0]:.6f}, ng={RING_NG_TE[0]:.6f})")
print(SEP)

# Cabecera
hdr = (f"  {'Ring':<10}  {'λ_design':>11}  {'neff':>10}  {'L (µm)':>9}  "
       f"{'m_real':>10}  {'m_near':>7}  {'λ_mNear':>10}  {'Δλ_mNear':>11}  "
       f"{'m_floor':>8}  {'λ_mFloor':>11}  {'Δλ_mFloor':>12}  "
       f"{'FSR_des':>9}  {'FSR_teo':>9}  {'ΔFSR':>8}")
print(hdr)
print("  " + "─" * 126)
units = (f"  {'':10}  {'(nm)':>11}  {'':>10}  {'':>9}  "
         f"{'':>10}  {'':>7}  {'(nm)':>10}  {'(pm)':>11}  "
         f"{'':>8}  {'(nm)':>11}  {'(pm)':>12}  "
         f"{'(nm)':>9}  {'(nm)':>9}  {'(pm)':>8}")
print(units)
print("  " + "─" * 126)

for i in range(N_RINGS):
    lbl  = RING_LABELS[i]
    star = " ◄ SENSOR" if i == 0 else ""
    flag_near  = "  ✓" if abs(df.loc[lbl, "Δλ_mNear (pm)"])  < 500 else "  !"
    flag_floor = "  ✓" if abs(df.loc[lbl, "Δλ_mFloor (pm)"]) < 500 else "  !"
    print(
        f"  {lbl:<10}  "
        f"{df.loc[lbl, 'λ_design (nm)']:>11.4f}  "
        f"{df.loc[lbl, 'neff']:>10.6f}  "
        f"{df.loc[lbl, 'L (µm)']:>9.4f}  "
        f"{df.loc[lbl, 'm_real']:>10.4f}  "
        f"{df.loc[lbl, 'm_near']:>7d}  "
        f"{df.loc[lbl, 'λ_theo_mNear (nm)']:>10.4f}{flag_near}  "
        f"{df.loc[lbl, 'Δλ_mNear (pm)']:>+9.1f} pm  "
        f"{df.loc[lbl, 'm_floor']:>8d}  "
        f"{df.loc[lbl, 'λ_theo_mFloor (nm)']:>11.4f}{flag_floor}  "
        f"{df.loc[lbl, 'Δλ_mFloor (pm)']:>+10.1f} pm  "
        f"{df.loc[lbl, 'FSR_design (nm)']:>9.4f}  "
        f"{df.loc[lbl, 'FSR_theo_mNear (nm)']:>9.4f}  "
        f"{df.loc[lbl, 'FSR_delta (pm)']:>+7.1f} pm"
        f"{star}"
    )

print()

# ─────────────────────────────────────────────────────────────────────────────
# Sección de diagnóstico INTERCONNECT vs teórico (anillo sensor)
# ─────────────────────────────────────────────────────────────────────────────
print(SEP)
print("  DIAGNÓSTICO — INTERCONNECT vs Teoría  (anillo sensor, sweep pt 0)")
print(SEP)
print(f"  λ_design   (diseño)          : {lambda_design_nm[0]:.4f} nm")
print(f"  λ_theo     (m_near={m_near[0]:d})      : {lambda_theo_near_nm[0]:.4f} nm   "
      f"Δ = {(lambda_theo_near_nm[0]-lambda_design_nm[0])*1e3:+.2f} pm")
print(f"  λ_theo     (m_floor={m_floor[0]:d})    : {lambda_theo_floor_nm[0]:.4f} nm   "
      f"Δ = {(lambda_theo_floor_nm[0]-lambda_design_nm[0])*1e3:+.2f} pm")
print(f"  m_real                       : {m_real[0]:.6f}   (fracción: {m_real[0]-m_near[0]:+.6f})")

if lambda_ic_port1_nm is not None:
    delta_ic_vs_design = lambda_ic_port1_nm - lambda_design_nm[0]
    delta_ic_vs_theo   = lambda_ic_port1_nm - lambda_theo_near_nm[0]
    print(f"  λ_IC       (ONA port1 dip)   : {lambda_ic_port1_nm:.4f} nm")
    print(f"    → Δ(IC − diseño)           : {delta_ic_vs_design*1e3:+.2f} pm")
    print(f"    → Δ(IC − theo m_near)      : {delta_ic_vs_theo*1e3:+.2f} pm")
    if abs(delta_ic_vs_design) > 0.05:
        print()
        print("  ⚠  SHIFT DETECTADO > 50 pm entre diseño e INTERCONNECT.")
        print("     Posibles causas:")
        print("     1. INTERCONNECT resuelve m = round(m_real) pero con neff re-evaluado")
        print("        en la frecuencia central (dispersión), no en λ_design.")
        print("     2. La propiedad 'frequency' se fija en c/λ_design, pero INTERCONNECT")
        print("        itera internamente hasta convergencia con neff(λ) → shift residual.")
        print("     3. Las pérdidas (101 dB/m) ensanchan la resonancia y desplazan el")
        print("        mínimo aparente de transmisión respecto al centro Lorentziano.")
        print("     4. El valor de m_real no es entero exacto → INTERCONNECT elige el")
        f"        m más cercano, produciendo Δλ = {(lambda_theo_near_nm[0]-lambda_design_nm[0])*1e3:+.2f} pm."
        print(f"        m más cercano: Δλ = {(lambda_theo_near_nm[0]-lambda_design_nm[0])*1e3:+.2f} pm.")
else:
    print("  λ_IC (ONA port1)             : N/A — sweep no ejecutado aún")

print(SEP)

# ─────────────────────────────────────────────────────────────────────────────
# Sección de diagnóstico para TODOS los anillos espectrómetros
# ─────────────────────────────────────────────────────────────────────────────
print()
print("  RESUMEN ESPECTRÓMETROS — Δλ (teoría m_near − diseño) [pm]")
print("  " + "─" * 65)
for i in range(N_RINGS):
    lbl   = RING_LABELS[i]
    delta = df.loc[lbl, "Δλ_mNear (pm)"]
    bar   = "█" * int(abs(delta) / 50)
    sign  = "+" if delta >= 0 else "-"
    print(f"  {lbl:<10}  {delta:>+8.1f} pm  {bar}")
print()

# ─────────────────────────────────────────────────────────────────────────────
# FIGURA 1 — λ_design vs λ_theo (m_near) vs λ_theo (m_floor) para todos los anillos
# ─────────────────────────────────────────────────────────────────────────────
fig_th, ax_th = plt.subplots(figsize=(12, 5))
x = np.arange(N_RINGS)

ax_th.scatter(x, lambda_design_nm,       marker="o", s=60,  zorder=6,
              color="#2166ac", label="λ diseño")
ax_th.scatter(x, lambda_theo_near_nm,    marker="^", s=60,  zorder=5,
              color="#d6604d", label="λ_theo  (m = round)")
ax_th.scatter(x, lambda_theo_floor_nm,   marker="s", s=40,  zorder=4,
              color="#4dac26", label="λ_theo  (m = floor)", alpha=0.75)

if lambda_ic_port1_nm is not None:
    ax_th.axhline(lambda_ic_port1_nm, color="#762a83", lw=1.5, ls="--",
                  label=f"IC ONA p1 dip  ({lambda_ic_port1_nm:.3f} nm)")

ax_th.set_xticks(x)
ax_th.set_xticklabels(RING_LABELS, rotation=35, ha="right", fontsize=8)
ax_th.set_ylabel("Longitud de onda (nm)")
ax_th.set_xlabel("Anillo")
ax_th.set_title(
    "Resonancias de diseño vs teóricas por anillo\n"
    "λ_theo = (neff × L) / m    [m = round(neff·L/λ_design) ó floor]"
)
ax_th.legend(framealpha=0.9, fontsize=9)
ax_th.yaxis.set_minor_locator(ticker.AutoMinorLocator())
fig_th.tight_layout()
fig_th.savefig(FIGURES_DIR / "theoretical_lambda_vs_design.png", dpi=200)
fig_th.savefig(FIGURES_DIR / "theoretical_lambda_vs_design.pdf")
print(f"  Guardada → theoretical_lambda_vs_design.png/pdf")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURA 2 — Δλ (pm) por anillo: cuantifica el shift teórico vs diseño
# ─────────────────────────────────────────────────────────────────────────────
fig_d, ax_d = plt.subplots(figsize=(12, 4))
bars = ax_d.bar(x, df["Δλ_mNear (pm)"].values,
                color=["#d6604d" if v >= 0 else "#4393c3"
                       for v in df["Δλ_mNear (pm)"].values],
                edgecolor="k", linewidth=0.5, zorder=4)
ax_d.axhline(0, color="k", lw=0.8, ls="-")
ax_d.set_xticks(x)
ax_d.set_xticklabels(RING_LABELS, rotation=35, ha="right", fontsize=8)
ax_d.set_ylabel("Δλ  (pm)  [λ_theo(m_near) − λ_diseño]")
ax_d.set_title(
    "Shift teórico respecto al diseño por anillo  [Δλ = λ_theo − λ_diseño]\n"
    "Causado por m_real no entero → INTERCONNECT elige m_near"
)
for bar, val in zip(bars, df["Δλ_mNear (pm)"].values):
    ax_d.text(bar.get_x() + bar.get_width() / 2, val + (5 if val >= 0 else -15),
              f"{val:+.0f}", ha="center", va="bottom", fontsize=7)
ax_d.yaxis.set_minor_locator(ticker.AutoMinorLocator())
fig_d.tight_layout()
fig_d.savefig(FIGURES_DIR / "delta_lambda_theoretical_vs_design.png", dpi=200)
fig_d.savefig(FIGURES_DIR / "delta_lambda_theoretical_vs_design.pdf")
print(f"  Guardada → delta_lambda_theoretical_vs_design.png/pdf")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURA 3 — FSR de diseño vs FSR teórico
# ─────────────────────────────────────────────────────────────────────────────
fig_fsr, ax_fsr = plt.subplots(figsize=(12, 4))
ax_fsr.scatter(x, fsr_design_nm,     marker="o", s=60, zorder=5,
               color="#2166ac",  label="FSR diseño (tabla)")
ax_fsr.scatter(x, fsr_theo_near_nm,  marker="^", s=60, zorder=5,
               color="#d6604d",  label="FSR teórico  (λ_theo/ng/L)")

ax_fsr.set_xticks(x)
ax_fsr.set_xticklabels(RING_LABELS, rotation=35, ha="right", fontsize=8)
ax_fsr.set_ylabel("FSR  (nm)")
ax_fsr.set_xlabel("Anillo")
ax_fsr.set_title("FSR de diseño vs FSR teórico calculado")
ax_fsr.legend(framealpha=0.9, fontsize=9)
ax_fsr.yaxis.set_minor_locator(ticker.AutoMinorLocator())
fig_fsr.tight_layout()
fig_fsr.savefig(FIGURES_DIR / "fsr_design_vs_theoretical.png", dpi=200)
fig_fsr.savefig(FIGURES_DIR / "fsr_design_vs_theoretical.pdf")
print(f"  Guardada → fsr_design_vs_theoretical.png/pdf")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURA 4 — Comparación espectral ONA port1 (sweep pt 0) vs λ_theo
#            (solo si INTERCONNECT ya corrió)
# ─────────────────────────────────────────────────────────────────────────────
if lambda_ic_port1_nm is not None and wavelengths_m is not None:
    fig_sp, ax_sp = plt.subplots(figsize=(10, 4))
    wl_nm_ref = wavelengths_m * 1e9
    ax_sp.plot(wl_nm_ref, T_port1_dB[0, :],
               color="#2166ac", lw=1.4, label="IC ONA port1 (sweep pt 0)")

    # Marcar resonancias teóricas de todos los anillos que caen en la ventana
    wl_min, wl_max = wl_nm_ref.min(), wl_nm_ref.max()
    for i in range(N_RINGS):
        lbl = RING_LABELS[i]
        lw  = lambda_theo_near_nm[i]
        if wl_min <= lw <= wl_max:
            color_i = "#d6604d" if i == 0 else "#4dac26"
            ax_sp.axvline(lw, color=color_i, lw=1.0, ls="--", alpha=0.7)
            ax_sp.text(lw, ax_sp.get_ylim()[0] + 0.5,
                       f"{lbl}\n{lw:.2f}", fontsize=5, ha="center",
                       color=color_i, rotation=90, va="bottom")
        ld = lambda_design_nm[i]
        if wl_min <= ld <= wl_max:
            ax_sp.axvline(ld, color="#762a83", lw=0.8, ls=":", alpha=0.5)

    from matplotlib.lines import Line2D
    legend_elems = [
        Line2D([0], [0], color="#2166ac", lw=1.4, label="IC ONA port1 (sweep 0)"),
        Line2D([0], [0], color="#d6604d", lw=1.0, ls="--", label="λ_theo sensor (m_near)"),
        Line2D([0], [0], color="#4dac26", lw=1.0, ls="--", label="λ_theo espectrómetros (m_near)"),
        Line2D([0], [0], color="#762a83", lw=0.8, ls=":",  label="λ diseño"),
    ]
    ax_sp.legend(handles=legend_elems, fontsize=8, framealpha=0.9, loc="lower right")
    ax_sp.set_xlabel("Longitud de onda (nm)")
    ax_sp.set_ylabel("Transmisión (dB)")
    ax_sp.set_title(
        "Espectro INTERCONNECT (sweep pt 0)  +  resonancias teóricas\n"
        "Líneas discontinuas: λ_theo(m_near)  │  Líneas punteadas: λ_diseño"
    )
    ax_sp.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig_sp.tight_layout()
    fig_sp.savefig(FIGURES_DIR / "spectrum_ic_vs_theoretical_resonances.png", dpi=200)
    fig_sp.savefig(FIGURES_DIR / "spectrum_ic_vs_theoretical_resonances.pdf")
    print(f"  Guardada → spectrum_ic_vs_theoretical_resonances.png/pdf")

# ─────────────────────────────────────────────────────────────────────────────
# Exportar tabla a CSV para referencia externa
# ─────────────────────────────────────────────────────────────────────────────
csv_path = DATA_DIR / f"{VERSION_NAME}_theoretical_comparison.csv"
df.to_csv(csv_path, float_format="%.6f")
print(f"\n  CSV exportado → {csv_path}")
print(f"  Figuras       → {FIGURES_DIR}")
print()

plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# Exponer variables para posible uso en celdas posteriores
# ─────────────────────────────────────────────────────────────────────────────
theo_results = dict(
    lambda_design_nm     = lambda_design_nm,
    lambda_theo_near_nm  = lambda_theo_near_nm,
    lambda_theo_floor_nm = lambda_theo_floor_nm,
    fsr_design_nm        = fsr_design_nm,
    fsr_theo_near_nm     = fsr_theo_near_nm,
    m_real               = m_real,
    m_near               = m_near,
    m_floor              = m_floor,
    neff_used            = neff_calc,
    ng_used              = ng_calc,
    L_m                  = L_m,
    df_table             = df,
)

print("  Variable 'theo_results' disponible para celdas posteriores.")

# ═════════════════════════════════════════════════════════════════════════════
#  END CELL 5
# ═════════════════════════════════════════════════════════════════════════════

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Parametrización: n_cladding vs n_eff del anillo sensor           ║
# ║                                                                             ║
# ║  DEPENDENCIAS                                                               ║
# ║    SWEEP_NEFF      : array (200,) — ya definido en Cell 2                  ║
# ║    SWEEP_N_POINTS  : 200          — ya definido en Cell 2                  ║
# ║    FIGURES_DIR     : Path         — ya definido en Cell 1                  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# Índice de refracción del cladding — 200 puntos de 1.33 a 1.37
# Cada punto i corresponde al par (n_cladding[i], SWEEP_NEFF[i])
N_CLADDING_START = 1.33
N_CLADDING_STOP  = 1.37
n_cladding = np.linspace(N_CLADDING_START, N_CLADDING_STOP, SWEEP_N_POINTS)

# ── Ajuste lineal para mostrar la sensibilidad dn_eff / dn_cladding ──────────
coeffs_nc   = np.polyfit(n_cladding, SWEEP_NEFF, 1)
slope_nc    = coeffs_nc[0]   # Δn_eff / Δn_cladding  [RIU/RIU]
fit_neff    = np.poly1d(coeffs_nc)(n_cladding)
r2          = 1.0 - np.sum((SWEEP_NEFF - fit_neff)**2) / \
                    np.sum((SWEEP_NEFF - SWEEP_NEFF.mean())**2)

print("=" * 62)
print("  n_cladding  ↔  n_eff (sensor ring)  —  Parametrización")
print("=" * 62)
print(f"  n_cladding  :  {N_CLADDING_START:.4f}  →  {N_CLADDING_STOP:.4f}   ({SWEEP_N_POINTS} pts)")
print(f"  n_eff       :  {SWEEP_NEFF[0]:.8f}  →  {SWEEP_NEFF[-1]:.8f}")
print(f"  Δn_eff      :  {SWEEP_NEFF[-1]-SWEEP_NEFF[0]:.6e}  RIU")
print(f"  Δn_cladding :  {N_CLADDING_STOP-N_CLADDING_START:.4f}  RIU")
print(f"  Sensibilidad:  dn_eff/dn_clad = {slope_nc:.6f}  RIU/RIU")
print(f"  R²          :  {r2:.8f}")
print("=" * 62)

# ── Figura ────────────────────────────────────────────────────────────────────
fig_nc, ax_nc = plt.subplots(figsize=(8, 5))

ax_nc.plot(
    SWEEP_NEFF, n_cladding,
    color="#2166ac", lw=2.0, marker="o", ms=3.5,
    markevery=10, alpha=0.85,
    label="Datos simulados (MODE/FDTD)",
    zorder=4,
)
ax_nc.plot(
    fit_neff, n_cladding,
    color="#d6604d", lw=1.5, ls="--",
    label=(f"Ajuste lineal\n"
           f"$\\partial n_{{eff}} / \\partial n_{{clad}}$ = {slope_nc:.5f}  RIU/RIU\n"
           f"$R^2$ = {r2:.6f}"),
    zorder=3,
)

ax_nc.set_xlabel("Índice efectivo del anillo sensor  $n_{eff}$ (TE)", fontsize=12)
ax_nc.set_ylabel("Índice de refracción del cladding  $n_{clad}$", fontsize=12)
ax_nc.set_title(
    "Parametrización: $n_{clad}$ vs $n_{eff}$ — Anillo sensor  [SiN 400 nm × 1000 nm]\n"
    f"$n_{{clad}} \\in [{N_CLADDING_START},\\,{N_CLADDING_STOP}]$  —  {SWEEP_N_POINTS} puntos",
    fontsize=12,
)
ax_nc.legend(framealpha=0.92, fontsize=9, loc="upper left")
ax_nc.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax_nc.yaxis.set_minor_locator(ticker.AutoMinorLocator())

# Anotaciones de inicio y fin
ax_nc.annotate(
    f"$n_{{clad}}$ = {N_CLADDING_START:.2f}\n$n_{{eff}}$ = {SWEEP_NEFF[0]:.6f}",
    xy=(SWEEP_NEFF[0], n_cladding[0]),
    xytext=(SWEEP_NEFF[0] + 0.00015, n_cladding[0] + 0.001),
    fontsize=8, color="#2166ac",
    arrowprops=dict(arrowstyle="->", color="#2166ac", lw=0.8),
)
ax_nc.annotate(
    f"$n_{{clad}}$ = {N_CLADDING_STOP:.2f}\n$n_{{eff}}$ = {SWEEP_NEFF[-1]:.6f}",
    xy=(SWEEP_NEFF[-1], n_cladding[-1]),
    xytext=(SWEEP_NEFF[-1] - 0.00055, n_cladding[-1] - 0.003),
    fontsize=8, color="#2166ac",
    arrowprops=dict(arrowstyle="->", color="#2166ac", lw=0.8),
)

fig_nc.tight_layout()
fig_nc.savefig(FIGURES_DIR / "ncladding_vs_neff_sensor.png", dpi=200)
fig_nc.savefig(FIGURES_DIR / "ncladding_vs_neff_sensor.pdf")
plt.show()
print(f"\n  Guardada → ncladding_vs_neff_sensor.png/pdf  ({FIGURES_DIR})")

# ── Exponer variable para celdas posteriores ──────────────────────────────────
# n_cladding[i]  ↔  SWEEP_NEFF[i]  ↔  SWEEP_NG[i]
# (trío correlacionado: cada índice de cladding tiene su neff y ng únicos)

# ═════════════════════════════════════════════════════════════════════════════
#  END CELL 6
# ═════════════════════════════════════════════════════════════════════════════

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Visualización completa con n_cladding en el eje X               ║
# ║                                                                            ║
# ║  DEPENDENCIAS                                                               ║
# ║    Todas las variables de Cell 4 (funciones de plot, DROP_LABELS, etc.)   ║
# ║    n_cladding   : (200,)  np.linspace(1.33, 1.37, 200) — definido Cell 6  ║
# ║    sweep_results / HDF5  — datos de simulación de Cell 3                  ║
# ║                                                                            ║
# ║  ESTRATEGIA                                                                 ║
# ║  Las funciones de Cell 4 reciben los mismos arrays de datos pero se        ║
# ║  sustituye el eje X de neff por n_cladding.  La correspondencia es         ║
# ║  biunívoca: n_cladding[i] ↔ SWEEP_NEFF[i] ↔ punto de sweep i.            ║
# ║  Se definen variantes "_nc" de cada función de plot que solo cambian       ║
# ║  la etiqueta y los valores del eje X; toda la lógica de datos es idéntica. ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

_res = get_results()
mask_nc   = _valid_mask(_res)
nclad_v   = n_cladding[mask_nc]          # n_cladding filtrado a puntos válidos
neff_v_nc = _res["neff_sweep"][mask_nc]  # neff correspondiente (para colorbars)

# ─────────────────────────────────────────────────────────────────────────────
# Plot NC-1 — ★ PRINCIPAL: Potencia integrada en drop vs n_cladding ★
# ─────────────────────────────────────────────────────────────────────────────
def plot_drop_power_vs_ncladding(results=None, figsize=(11, 6),
                                 save: bool = True) -> plt.Figure:
    """
    Versión n_cladding de plot_drop_power_vs_neff (Cell 4, Plot 1).
    Eje X: índice de refracción del cladding (1.33 → 1.37).
    Eje Y: potencia integrada en el detector [dBm].
    13 curvas, una por anillo espectrómetro (RING_2..14).
    """
    if results is None:
        results = get_results()
    mask   = _valid_mask(results)
    nc_v   = n_cladding[mask]
    p_v    = results["drop_power_dBm"][mask, :]   # (n_valid, N_DROPS)

    cmap = plt.get_cmap("tab20")
    fig, ax = plt.subplots(figsize=figsize)
    for k in range(N_DROPS):
        ax.plot(nc_v, p_v[:, k],
                color=cmap(k / N_DROPS), lw=1.5,
                marker="o", ms=2.5, alpha=0.85,
                label=DROP_LABELS[k])

    ax.set_xlabel("Índice de refracción del cladding  $n_{clad}$", fontsize=12)
    ax.set_ylabel("Potencia en el detector  (dBm)", fontsize=12)
    ax.set_title(
        "Potencia integrada en drop vs $n_{clad}$  [V3 — ONA multiport]\n"
        r"$P_{det} = P_{src} \cdot \langle|T_{drop}(\lambda)|^2\rangle_\lambda$"
        "   —   13 anillos espectrómetros",
        fontsize=12,
    )
    ax.legend(ncol=3, framealpha=0.88, fontsize=8, loc="best")
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR / "nc_drop_power_vs_ncladding_all.png")
        fig.savefig(FIGURES_DIR / "nc_drop_power_vs_ncladding_all.pdf")
        log.info("Saved → nc_drop_power_vs_ncladding_all.png/pdf")
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Plot NC-2 — Espectros sensor through coloreados por n_cladding
# ─────────────────────────────────────────────────────────────────────────────
def plot_sensor_through_sweep_nc(results=None, n_curves: int = 200,
                                  figsize=(10, 5), cmap_name: str = "plasma",
                                  save: bool = True) -> plt.Figure:
    """
    Versión n_cladding de plot_sensor_through_sweep (Cell 4, Plot 2).
    Espectros de transmisión ONA input 1 (RING_1 through).
    Colormap codifica n_cladding en lugar de neff.
    """
    if results is None:
        results = get_results()
    wl_nm    = results["wavelengths_m"] * 1e9
    T_data   = results["T_sensor_through_dB"]
    mask     = _valid_mask(results)
    valid_idx = np.where(mask)[0]
    n_sel    = min(n_curves, len(valid_idx))
    sel_idx  = valid_idx[
        np.round(np.linspace(0, len(valid_idx) - 1, n_sel)).astype(int)
    ]
    nc_sel   = n_cladding[sel_idx]

    cmap = plt.get_cmap(cmap_name)
    norm = Normalize(vmin=nc_sel.min(), vmax=nc_sel.max())
    sm   = ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])

    fig, ax = plt.subplots(figsize=figsize)
    for idx in sel_idx:
        ax.plot(wl_nm, T_data[idx],
                color=cmap(norm(n_cladding[idx])),
                lw=0.8, alpha=0.70)
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label("$n_{clad}$", fontsize=10)
    ax.set_xlabel("Longitud de onda  (nm)")
    ax.set_ylabel("Transmisión  (dB)")
    ax.set_title(
        f"Espectro sensor through — ONA input 1  (RING_1)\n"
        f"({n_sel} curvas,  color = $n_{{clad}}$)"
    )
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()
    if save:
        stem = f"nc_sensor_through_sweep_{n_sel}curves"
        fig.savefig(FIGURES_DIR / f"{stem}.png")
        fig.savefig(FIGURES_DIR / f"{stem}.pdf")
        log.info(f"Saved → {stem}.png/pdf")
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Plot NC-3 — Espectros through final (ONA input 15) coloreados por n_cladding
# ─────────────────────────────────────────────────────────────────────────────
def plot_final_through_sweep_nc(results=None, n_curves: int = 200,
                                 figsize=(10, 5), cmap_name: str = "viridis",
                                 save: bool = True) -> plt.Figure:
    """
    Versión n_cladding de plot_final_through_sweep (Cell 4, Plot 3).
    Espectros ONA input 15 (RING_14 through), colormap = n_cladding.
    """
    if results is None:
        results = get_results()
    wl_nm    = results["wavelengths_m"] * 1e9
    T_data   = results["T_final_through_dB"]
    mask     = _valid_mask(results)
    valid_idx = np.where(mask)[0]
    n_sel    = min(n_curves, len(valid_idx))
    sel_idx  = valid_idx[
        np.round(np.linspace(0, len(valid_idx) - 1, n_sel)).astype(int)
    ]
    nc_sel   = n_cladding[sel_idx]

    cmap = plt.get_cmap(cmap_name)
    norm = Normalize(vmin=nc_sel.min(), vmax=nc_sel.max())
    sm   = ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])

    fig, ax = plt.subplots(figsize=figsize)
    for idx in sel_idx:
        ax.plot(wl_nm, T_data[idx],
                color=cmap(norm(n_cladding[idx])),
                lw=0.8, alpha=0.70)
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label("$n_{clad}$", fontsize=10)
    ax.set_xlabel("Longitud de onda  (nm)")
    ax.set_ylabel("Transmisión  (dB)")
    ax.set_title(
        f"Through final de cascada — ONA input 15  (RING_14)\n"
        f"({n_sel} curvas,  color = $n_{{clad}}$)"
    )
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()
    if save:
        stem = f"nc_final_through_sweep_{n_sel}curves"
        fig.savefig(FIGURES_DIR / f"{stem}.png")
        fig.savefig(FIGURES_DIR / f"{stem}.pdf")
        log.info(f"Saved → {stem}.png/pdf")
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Plot NC-4 — Heatmap espectral de drop: n_cladding × longitud de onda
# ─────────────────────────────────────────────────────────────────────────────
def plot_drop_spectrum_heatmap_nc(results=None, drop_k: int = 1,
                                   figsize=(10, 3.5), cmap_name: str = "inferno",
                                   vmin_dB=None, vmax_dB=None,
                                   save: bool = True) -> plt.Figure:
    """
    Versión n_cladding de plot_drop_spectrum_heatmap (Cell 4, Plot 4).
    Heatmap: eje Y = n_cladding (en lugar de neff),  eje X = longitud de onda.
    drop_k 1-based (1=RING_2 drop … 13=RING_14 drop).
    """
    if results is None:
        results = get_results()
    assert 1 <= drop_k <= N_DROPS
    ring_label = DROP_LABELS[drop_k - 1]

    wl_nm  = results["wavelengths_m"] * 1e9
    t_drop = results["T_drop_dB"]
    mask   = _valid_mask(results)
    nc_v   = n_cladding[mask]
    spec_v = t_drop[mask, drop_k - 1, :]

    _vmin = vmin_dB if vmin_dB is not None else np.nanpercentile(spec_v, 2)
    _vmax = vmax_dB if vmax_dB is not None else np.nanpercentile(spec_v, 98)

    fig, ax = plt.subplots(figsize=figsize)
    img = ax.pcolormesh(wl_nm, nc_v, spec_v,
                        cmap=cmap_name, vmin=_vmin, vmax=_vmax, shading="auto")
    cbar = fig.colorbar(img, ax=ax, pad=0.01)
    cbar.set_label("Transmisión  (dB)", fontsize=10)
    ax.set_xlabel("Longitud de onda  (nm)")
    ax.set_ylabel("$n_{clad}$")
    ax.set_title(
        f"Heatmap espectral drop — {ring_label}  (ONA input {drop_k + 1})\n"
        f"Barrido de $n_{{clad}}$ sobre anillo sensor (RING_1)"
    )
    fig.tight_layout()
    if save:
        fname = f"nc_drop{drop_k}_spectrum_heatmap"
        fig.savefig(FIGURES_DIR / f"{fname}.png")
        fig.savefig(FIGURES_DIR / f"{fname}.pdf")
        log.info(f"Saved → {fname}.png/pdf")
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Plot NC-5 — Grid con los 13 heatmaps de drop (eje Y = n_cladding)
# ─────────────────────────────────────────────────────────────────────────────
def plot_all_drop_heatmaps_nc(results=None, ncols: int = 4,
                               cmap_name: str = "inferno",
                               save: bool = True) -> plt.Figure:
    """
    Versión n_cladding de plot_all_drop_heatmaps (Cell 4, Plot 5).
    Grid 4×4 con los 13 heatmaps; eje Y = n_cladding.
    """
    if results is None:
        results = get_results()
    wl_nm  = results["wavelengths_m"] * 1e9
    t_drop = results["T_drop_dB"]
    mask   = _valid_mask(results)
    nc_v   = n_cladding[mask]

    nrows = math.ceil(N_DROPS / ncols)
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(ncols * 3.8, nrows * 2.8),
                             sharex=True, sharey=True)
    axes_flat = axes.flatten()

    all_vals = t_drop[mask].ravel()
    vmin = np.nanpercentile(all_vals, 2)
    vmax = np.nanpercentile(all_vals, 98)

    im = None
    for k in range(1, N_DROPS + 1):
        ax   = axes_flat[k - 1]
        spec = t_drop[mask, k - 1, :]
        im   = ax.pcolormesh(wl_nm, nc_v, spec,
                             cmap=cmap_name, vmin=vmin, vmax=vmax, shading="auto")
        ax.set_title(f"{DROP_LABELS[k-1]} drop", fontsize=8)
        if k % ncols == 1:
            ax.set_ylabel("$n_{clad}$", fontsize=7)
        if k > (nrows - 1) * ncols:
            ax.set_xlabel("λ (nm)", fontsize=7)
        ax.tick_params(labelsize=6)

    for ax in axes_flat[N_DROPS:]:
        ax.set_visible(False)

    if im is not None:
        fig.colorbar(im, ax=axes_flat[:N_DROPS], shrink=0.6, pad=0.02,
                     label="Transmisión (dB)", fraction=0.015)
    fig.suptitle(
        "Espectros drop — 13 anillos espectrómetros  [V3 — ONA multiport]\n"
        "Barrido de $n_{clad}$ sobre anillo sensor (RING_1)",
        fontsize=11,
    )
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR / "nc_all_drop_heatmaps_grid.png", dpi=200)
        fig.savefig(FIGURES_DIR / "nc_all_drop_heatmaps_grid.pdf")
        log.info("Saved → nc_all_drop_heatmaps_grid.png/pdf")
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Plot NC-6 — Heatmap 2D de potencia: anillo × n_cladding
# ─────────────────────────────────────────────────────────────────────────────
def plot_power_heatmap_drops_vs_ncladding(results=None, figsize=(10, 5),
                                           cmap_name: str = "plasma",
                                           save: bool = True) -> plt.Figure:
    """
    Versión n_cladding de plot_power_heatmap_drops_vs_neff (Cell 4, Plot 6).
    Eje X = n_cladding,  eje Y = índice de anillo espectrómetro.
    Color = potencia integrada en el detector [dBm].
    """
    if results is None:
        results = get_results()
    mask  = _valid_mask(results)
    nc_v  = n_cladding[mask]
    p_v   = results["drop_power_dBm"][mask, :].T    # (N_DROPS, n_valid)

    fig, ax = plt.subplots(figsize=figsize)
    img = ax.pcolormesh(nc_v, np.arange(1, N_DROPS + 1), p_v,
                        cmap=cmap_name, shading="auto")
    cbar = fig.colorbar(img, ax=ax, pad=0.02)
    cbar.set_label("Potencia en detector  (dBm)", fontsize=10)
    ax.set_xlabel("Índice de refracción del cladding  $n_{clad}$", fontsize=11)
    ax.set_ylabel("Índice de anillo espectrómetro  (1=RING_2 … 13=RING_14)", fontsize=10)
    ax.set_yticks(np.arange(1, N_DROPS + 1))
    ax.set_yticklabels(DROP_LABELS, fontsize=7)
    ax.set_title(
        "Heatmap de potencia — Todos los drops vs $n_{clad}$  [V3]\n"
        r"Color: $P_{det}$ [dBm]  por anillo espectrómetro",
        fontsize=12,
    )
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR / "nc_power_heatmap_drops_vs_ncladding.png", dpi=200)
        fig.savefig(FIGURES_DIR / "nc_power_heatmap_drops_vs_ncladding.pdf")
        log.info("Saved → nc_power_heatmap_drops_vs_ncladding.png/pdf")
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Plot NC-7 — Tracking de resonancia del sensor vs n_cladding
# ─────────────────────────────────────────────────────────────────────────────
def plot_resonance_tracking_nc(results=None, figsize=(7, 4.5),
                                save: bool = True) -> plt.Figure:
    """
    Versión n_cladding de plot_resonance_tracking (Cell 4, Plot 7).
    Eje X = n_cladding.  Sensibilidad reportada en nm / RIU de cladding.
    """
    if results is None:
        results = get_results()
    wl_nm  = results["wavelengths_m"] * 1e9
    T_data = results["T_sensor_through_dB"]
    mask   = _valid_mask(results)
    nc_v   = n_cladding[mask]
    T_v    = T_data[mask, :]
    dip_idx = np.argmin(T_v, axis=1)
    lam_dip = wl_nm[dip_idx]

    coeffs = np.polyfit(nc_v, lam_dip, 1)
    sens   = coeffs[0]   # nm / RIU_cladding
    r2     = 1.0 - np.sum((lam_dip - np.poly1d(coeffs)(nc_v))**2) / \
                   np.sum((lam_dip - lam_dip.mean())**2)

    fig, ax = plt.subplots(figsize=figsize)
    ax.scatter(nc_v, lam_dip, s=20, zorder=5,
               color="#2166ac", label="Mínimo de transmisión (dip)")
    ax.plot(nc_v, np.poly1d(coeffs)(nc_v), "r--", lw=1.5,
            label=(f"Ajuste lineal\n"
                   f"$S = \\partial\\lambda / \\partial n_{{clad}}$ "
                   f"= {sens:.2f} nm/RIU\n"
                   f"$R^2$ = {r2:.6f}"))
    ax.set_xlabel("Índice de refracción del cladding  $n_{clad}$", fontsize=12)
    ax.set_ylabel(r"\lambda  (nm)", fontsize=12)
    ax.set_title(
        f"Tracking de resonancia — Sensor through (ONA input 1)\n"
        f"Sensibilidad: {sens:.2f} nm/RIU  ($n_{{clad}}$)",
        fontsize=12,
    )
    ax.legend(framealpha=0.9, fontsize=9)
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR / "nc_resonance_tracking_vs_ncladding.png")
        fig.savefig(FIGURES_DIR / "nc_resonance_tracking_vs_ncladding.pdf")
        log.info("Saved → nc_resonance_tracking_vs_ncladding.png/pdf")
    log.info(f"Sensibilidad (n_cladding): {sens:.4f} nm/RIU   R²={r2:.8f}")
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Ejecutar todas las gráficas NC
# ─────────────────────────────────────────────────────────────────────────────
fig_nc1 = plot_drop_power_vs_ncladding(_res)
fig_nc2 = plot_sensor_through_sweep_nc(_res, n_curves=200)
fig_nc3 = plot_final_through_sweep_nc(_res, n_curves=200)
fig_nc4 = plot_drop_spectrum_heatmap_nc(_res, drop_k=1)    # RING_2 drop
fig_nc5 = plot_drop_spectrum_heatmap_nc(_res, drop_k=13)   # RING_14 drop
fig_nc6 = plot_all_drop_heatmaps_nc(_res)
fig_nc7 = plot_power_heatmap_drops_vs_ncladding(_res)
fig_nc8 = plot_resonance_tracking_nc(_res)

plt.show()
print(f"\n  Figuras (n_cladding) → {FIGURES_DIR}")

# ═════════════════════════════════════════════════════════════════════════════
#  END CELL 7
# ═════════════════════════════════════════════════════════════════════════════

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Barrido de radio: 2 anillos (sensor fijo + espectrómetro swept) ║
# ║                                                                            ║
# ║  TOPOLOGÍA                                                                  ║
# ║  ─────────────────────────────────────────────────────────────────────     ║
# ║  ONA  output    → RING_S  input                                            ║
# ║  RING_S  out1   → ONA     input 1   (sensor through)                      ║
# ║  RING_S  out2   → RING_E  input     (sensor drop → espectrómetro)         ║
# ║  RING_E  out1   → ONA     input 3   (espectrómetro through)               ║
# ║  RING_E  out2   → ONA     input 2   (espectrómetro drop  ← MEDICIÓN)      ║
# ║                                                                            ║
# ║  SWEEP                                                                      ║
# ║  Variable: radio de RING_E  (circunferencia = 2π × R)                     ║
# ║  Inicio  : RING_RADIUS_M[1] = 19.1818 µm  (primer espectrómetro V3)       ║
# ║  Fin     : definido por RADIUS_SWEEP_STOP_M  (configurable)               ║
# ║  Fijo    : RING_S usa RING_RADIUS_M[0] = 19.0021 µm y todos sus           ║
# ║            parámetros ópticos de RING_NEFF_TE[0], RING_NG_TE[0], etc.    ║
# ║            RING_E usa todos los parámetros ópticos de RING_NEFF_TE[1],    ║
# ║            RING_NG_TE[1], etc. — solo su radio varía.                     ║
# ║                                                                            ║
# ║  DEPENDENCIAS (de celdas anteriores, no se redefinen)                     ║
# ║    RING_RADIUS_M, RING_LAMBDA_RES_M, RING_NEFF_TE, RING_NG_TE            ║
# ║    RING_KAPPA_INPUT_SQ, RING_KAPPA_DROP_SQ, RING_LOSS_DB_PER_M           ║
# ║    RING_D_TE_PS2_PER_KM, RING_POLARIZATION                               ║
# ║    SPEED_OF_LIGHT, ONA_NAME, ONA_LAMBDA_START_M, ONA_LAMBDA_STOP_M       ║
# ║    ONA_N_POINTS, ONA_POWER_DBM                                            ║
# ║    _eval, _try_eval, ICScriptError, ring_name                             ║
# ║    lumapi, log, DATA_DIR, FIGURES_DIR                                     ║
# ║    plt, np, h5py, math, ticker, Normalize, ScalarMappable, Path           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────────────
# Parámetros del sweep de radio
# ─────────────────────────────────────────────────────────────────────────────

# Anillo sensor (S): índice 0 en los arrays globales — FIJO durante el sweep
RS_IDX = 0   # posición en los arrays RING_* para el sensor

# Anillo espectrómetro (E): índice 1 en los arrays globales — radio VARIABLE
RE_IDX = 1   # posición en los arrays RING_* para el espectrómetro

# Rango del sweep de radio para RING_E
RADIUS_SWEEP_START_M = RING_RADIUS_M[RE_IDX]          # 19.1818 µm — valor de diseño
RADIUS_SWEEP_STOP_M  = RING_RADIUS_M[RE_IDX] * 1.005  # +0.5 % → ~19.278 µm
RADIUS_SWEEP_N       = 200

# Array de radios a barrer [m]
SWEEP_RADIUS_E_M = np.linspace(
    RADIUS_SWEEP_START_M,
    RADIUS_SWEEP_STOP_M,
    RADIUS_SWEEP_N,
)
# Circunferencias correspondientes [m] y [µm]
SWEEP_CIRCUM_E_M  = SWEEP_RADIUS_E_M * 2.0 * math.pi
SWEEP_CIRCUM_E_UM = SWEEP_CIRCUM_E_M * 1e6

# Nombres de elementos en INTERCONNECT para este sub-circuito
RS_NAME = "RS_SENSOR"      # anillo sensor
RE_NAME = "RE_SPEC"        # anillo espectrómetro (swept)
ONA2_NAME = "ONA_2ring"    # ONA dedicado para este sweep

# ONA: 3 inputs
#   input 1 → RING_S through (sensor)
#   input 2 → RING_E drop    (espectrómetro drop — MEDICIÓN PRINCIPAL)
#   input 3 → RING_E through (espectrómetro through)
ONA2_N_INPUTS         = 3
ONA2_THROUGH_SENSOR   = 1
ONA2_DROP_SPEC        = 2   # ← potencia de interés
ONA2_THROUGH_SPEC     = 3

# HDF5 dedicado para este sweep
HDF5_2RING_PATH = DATA_DIR / "ICNT_2Ring_RadiusSweep_V1.h5"

VERSION_2RING = "ICNT_2Ring_RadiusSweep_V1"

print("=" * 72)
print("  SWEEP DE RADIO — 2 anillos (sensor + espectrómetro)  [CELL 8]")
print("=" * 72)
print(f"  Sensor  (RS) : RING_RADIUS_M[{RS_IDX}] = {RING_RADIUS_M[RS_IDX]*1e6:.4f} µm  [FIJO]")
print(f"                 neff={RING_NEFF_TE[RS_IDX]:.6f}  ng={RING_NG_TE[RS_IDX]:.6f}")
print(f"                 λ_res={RING_LAMBDA_RES_M[RS_IDX]*1e9:.4f} nm")
print(f"  Espec.  (RE) : radio barrido de {RADIUS_SWEEP_START_M*1e6:.4f} → "
      f"{RADIUS_SWEEP_STOP_M*1e6:.4f} µm  ({RADIUS_SWEEP_N} pts)")
print(f"                 Δcircunf. = {(SWEEP_CIRCUM_E_M[-1]-SWEEP_CIRCUM_E_M[0])*1e9:.2f} nm")
print(f"                 neff={RING_NEFF_TE[RE_IDX]:.6f}  ng={RING_NG_TE[RE_IDX]:.6f}  [fijos]")
print(f"                 λ_res={RING_LAMBDA_RES_M[RE_IDX]*1e9:.4f} nm  [fijo]")
print(f"  ONA          : λ {ONA_LAMBDA_START_M*1e9:.2f}–{ONA_LAMBDA_STOP_M*1e9:.2f} nm  "
      f"│  {ONA_N_POINTS} pts  │  {ONA2_N_INPUTS} inputs")
print(f"  HDF5         : {HDF5_2RING_PATH}")
print("=" * 72)


# ─────────────────────────────────────────────────────────────────────────────
# Constructor del circuito de 2 anillos
# ─────────────────────────────────────────────────────────────────────────────
def _build_2ring_circuit(ic) -> None:
    """
    Construye el circuito de 2 anillos en INTERCONNECT.

    TOPOLOGÍA
    ──────────────────────────────────────────────────────
    ONA_2ring  output  → RS_SENSOR  input
    RS_SENSOR  out1    → ONA_2ring  input 1   (sensor through)
    RS_SENSOR  out2    → RE_SPEC    input     (sensor drop → espectrómetro)
    RE_SPEC    out2    → ONA_2ring  input 2   (espectrómetro DROP ← medición)
    RE_SPEC    out1    → ONA_2ring  input 3   (espectrómetro through)
    """
    _eval(ic, "switchtodesign")
    _try_eval(ic, "selectall")
    _try_eval(ic, "delete")

    pwr_W   = 10.0 ** (ONA_POWER_DBM / 10.0) * 1e-3
    f_start = SPEED_OF_LIGHT / ONA_LAMBDA_STOP_M
    f_stop  = SPEED_OF_LIGHT / ONA_LAMBDA_START_M

    # ── ONA ──────────────────────────────────────────────────────────────────
    _eval(ic, 'addelement("Optical Network Analyzer")')
    _eval(ic, f'set("name", "{ONA2_NAME}")')
    _try_eval(ic, 'set("x position", 0)')
    _try_eval(ic, 'set("y position", 0)')
    _eval(ic, f'setnamed("{ONA2_NAME}", "input parameter",       "start and stop")')
    _eval(ic, f'setnamed("{ONA2_NAME}", "number of input ports", {ONA2_N_INPUTS})')
    _eval(ic, f'setnamed("{ONA2_NAME}", "start frequency",       {f_start:.12e})')
    _eval(ic, f'setnamed("{ONA2_NAME}", "stop frequency",        {f_stop:.12e})')
    _eval(ic, f'setnamed("{ONA2_NAME}", "number of points",      {ONA_N_POINTS})')
    _eval(ic, f'setnamed("{ONA2_NAME}", "power",                 {pwr_W:.12e})')

    # ── Anillo sensor RS (parámetros fijos, índice RS_IDX=0) ─────────────────
    _eval(ic, 'addelement("Double Bus Ring Resonator")')
    _eval(ic, f'set("name", "{RS_NAME}")')
    _try_eval(ic, 'set("x position", 220)')
    _try_eval(ic, 'set("y position", 0)')
    # Aplica todos los parámetros del índice RS_IDX usando la función existente
    # pero apuntando al nombre RS_NAME (no ring_name(RS_IDX+1))
    _eval(ic, f'setnamed("{RS_NAME}", "length",                   '
              f'{RING_RADIUS_M[RS_IDX]*2.0*math.pi:.12e})')
    _eval(ic, f'setnamed("{RS_NAME}", "frequency",                '
              f'{SPEED_OF_LIGHT/RING_LAMBDA_RES_M[RS_IDX]:.12e})')
    _eval(ic, f'setnamed("{RS_NAME}", "effective index 1",        '
              f'{RING_NEFF_TE[RS_IDX]:.12f})')
    _eval(ic, f'setnamed("{RS_NAME}", "group index 1",            '
              f'{RING_NG_TE[RS_IDX]:.12f})')
    _eval(ic, f'setnamed("{RS_NAME}", "loss 1",                   '
              f'{RING_LOSS_DB_PER_M[RS_IDX]:.6f})')
    _eval(ic, f'setnamed("{RS_NAME}", "dispersion 1",             '
              f'{RING_D_TE_PS2_PER_KM[RS_IDX]*1e-15:.12e})')
    _eval(ic, f'setnamed("{RS_NAME}", "coupling coefficient 1 1", '
              f'{RING_KAPPA_INPUT_SQ[RS_IDX]:.12f})')
    _eval(ic, f'setnamed("{RS_NAME}", "coupling coefficient 1 2", '
              f'{RING_KAPPA_DROP_SQ[RS_IDX]:.12f})')
    _eval(ic, f'setnamed("{RS_NAME}", "configuration", "unidirectional")')
    log.info(f"  {RS_NAME} added  [sensor, R={RING_RADIUS_M[RS_IDX]*1e6:.4f} µm, FIJO]")

    # ── Anillo espectrómetro RE (radio inicial = RING_RADIUS_M[RE_IDX]) ──────
    _eval(ic, 'addelement("Double Bus Ring Resonator")')
    _eval(ic, f'set("name", "{RE_NAME}")')
    _try_eval(ic, 'set("x position", 440)')
    _try_eval(ic, 'set("y position", 0)')
    # Radio inicial: SWEEP_RADIUS_E_M[0] = RING_RADIUS_M[RE_IDX]
    _eval(ic, f'setnamed("{RE_NAME}", "length",                   '
              f'{SWEEP_CIRCUM_E_M[0]:.12e})')
    _eval(ic, f'setnamed("{RE_NAME}", "frequency",                '
              f'{SPEED_OF_LIGHT/RING_LAMBDA_RES_M[RE_IDX]:.12e})')
    _eval(ic, f'setnamed("{RE_NAME}", "effective index 1",        '
              f'{RING_NEFF_TE[RE_IDX]:.12f})')
    _eval(ic, f'setnamed("{RE_NAME}", "group index 1",            '
              f'{RING_NG_TE[RE_IDX]:.12f})')
    _eval(ic, f'setnamed("{RE_NAME}", "loss 1",                   '
              f'{RING_LOSS_DB_PER_M[RE_IDX]:.6f})')
    _eval(ic, f'setnamed("{RE_NAME}", "dispersion 1",             '
              f'{RING_D_TE_PS2_PER_KM[RE_IDX]*1e-15:.12e})')
    _eval(ic, f'setnamed("{RE_NAME}", "coupling coefficient 1 1", '
              f'{RING_KAPPA_INPUT_SQ[RE_IDX]:.12f})')
    _eval(ic, f'setnamed("{RE_NAME}", "coupling coefficient 1 2", '
              f'{RING_KAPPA_DROP_SQ[RE_IDX]:.12f})')
    _eval(ic, f'setnamed("{RE_NAME}", "configuration", "unidirectional")')
    log.info(f"  {RE_NAME} added  [espectrómetro, R_init={SWEEP_RADIUS_E_M[0]*1e6:.4f} µm]")

    # ── Conexiones ────────────────────────────────────────────────────────────
    def wire(a, pa, b, pb):
        _eval(ic, f'connect("{a}", "{pa}", "{b}", "{pb}")')

    wire(ONA2_NAME, "output",   RS_NAME,   "input")
    wire(RS_NAME,   "output 1", ONA2_NAME, f"input {ONA2_THROUGH_SENSOR}")
    wire(RS_NAME,   "output 2", RE_NAME,   "input")
    wire(RE_NAME,   "output 2", ONA2_NAME, f"input {ONA2_DROP_SPEC}")
    wire(RE_NAME,   "output 1", ONA2_NAME, f"input {ONA2_THROUGH_SPEC}")

    log.info(
        f"  2-ring circuit built: {RS_NAME} → {RE_NAME} → ONA_2ring  "
        f"(inputs: 1=sensor_thr, 2=spec_drop, 3=spec_thr)"
    )


# ─────────────────────────────────────────────────────────────────────────────
# Updater del radio de RING_E en el loop
# ─────────────────────────────────────────────────────────────────────────────
def _update_re_radius(ic, circum_m: float) -> None:
    """Actualiza solo la circunferencia (length) de RE_SPEC en cada punto."""
    _eval(ic, f'setnamed("{RE_NAME}", "length", {circum_m:.12e})')


# ─────────────────────────────────────────────────────────────────────────────
# Extracción de resultados del circuito de 2 anillos
# ─────────────────────────────────────────────────────────────────────────────
def _extract_2ring_results(ic) -> tuple:
    """
    Extrae espectros ONA_2ring después de run().

    Retorna
    -------
    wl_m              : (n_wl,)   longitudes de onda [m], ascendente
    T_sensor_thr_dB   : (n_wl,)   sensor through [dB]
    T_spec_drop_dB    : (n_wl,)   espectrómetro drop [dB]  ← MEDICIÓN
    T_spec_thr_dB     : (n_wl,)   espectrómetro through [dB]
    drop_power_dBm    : float      potencia integrada drop [dBm]
    """
    # Eje de frecuencias desde input 1
    raw_ref = ic.getresult(ONA2_NAME,
                           f"input {ONA2_THROUGH_SENSOR}/mode 1/transmission")
    f_arr  = np.asarray(raw_ref["frequency"]).flatten()
    sort_i = np.argsort(f_arr)[::-1]
    wl_m   = SPEED_OF_LIGHT / f_arr[sort_i]

    p_source_W = 10.0 ** (ONA_POWER_DBM / 10.0) * 1e-3

    def _T_lin(port_n: int) -> np.ndarray:
        raw = ic.getresult(ONA2_NAME,
                           f"input {port_n}/mode 1/transmission")
        return np.abs(np.asarray(raw["TE transmission"]).flatten()[sort_i])

    def _to_dB(T):
        return 10.0 * np.log10(np.where(T > 0, T, 1e-30))

    T_s_lin  = _T_lin(ONA2_THROUGH_SENSOR)
    T_ed_lin = _T_lin(ONA2_DROP_SPEC)
    T_et_lin = _T_lin(ONA2_THROUGH_SPEC)

    # Potencia integrada en el drop del espectrómetro
    mean_T2    = float(np.mean(T_ed_lin ** 2))
    p_drop_W   = p_source_W * mean_T2
    drop_p_dBm = 10.0 * np.log10(max(p_drop_W, 1e-40) * 1e3)

    return (wl_m,
            _to_dB(T_s_lin),
            _to_dB(T_ed_lin),
            _to_dB(T_et_lin),
            drop_p_dBm)


# ─────────────────────────────────────────────────────────────────────────────
# HDF5 para el sweep de radio
# ─────────────────────────────────────────────────────────────────────────────
def _init_hdf5_2ring(wl_ref_m: np.ndarray) -> None:
    n_pts = RADIUS_SWEEP_N
    n_wl  = len(wl_ref_m)
    with h5py.File(HDF5_2RING_PATH, "w") as f:
        md = f.create_group("metadata")
        md.create_dataset("radius_sweep_m",   data=SWEEP_RADIUS_E_M)
        md.create_dataset("circum_sweep_m",   data=SWEEP_CIRCUM_E_M)
        md.create_dataset("wavelengths_m",    data=wl_ref_m)
        md.attrs["version_name"]           = VERSION_2RING
        md.attrs["sweep_variable"]         = "RING_E radius (circumference)"
        md.attrs["sweep_n_points"]         = RADIUS_SWEEP_N
        md.attrs["radius_start_m"]         = RADIUS_SWEEP_START_M
        md.attrs["radius_stop_m"]          = RADIUS_SWEEP_STOP_M
        md.attrs["sensor_ring_radius_m"]   = RING_RADIUS_M[RS_IDX]
        md.attrs["sensor_ring_neff"]       = RING_NEFF_TE[RS_IDX]
        md.attrs["sensor_ring_ng"]         = RING_NG_TE[RS_IDX]
        md.attrs["sensor_ring_lambda_res"] = RING_LAMBDA_RES_M[RS_IDX]
        md.attrs["spec_ring_neff"]         = RING_NEFF_TE[RE_IDX]
        md.attrs["spec_ring_ng"]           = RING_NG_TE[RE_IDX]
        md.attrs["spec_ring_lambda_res"]   = RING_LAMBDA_RES_M[RE_IDX]
        md.attrs["ona_lambda_start_m"]     = ONA_LAMBDA_START_M
        md.attrs["ona_lambda_stop_m"]      = ONA_LAMBDA_STOP_M
        md.attrs["ona_n_points"]           = ONA_N_POINTS
        md.attrs["ona_power_dBm"]          = ONA_POWER_DBM
        md.attrs["power_calc_method"]      = (
            "P_det[W] = P_source[W] * mean(|T_drop(lambda)|^2) over ONA band"
        )
        md.attrs["timestamp_start"]        = datetime.now().isoformat()

        rg = f.create_group("results")
        rg.create_dataset("T_sensor_thr_dB",
                          data=np.full((n_pts, n_wl), np.nan), chunks=(1, n_wl))
        rg.create_dataset("T_spec_drop_dB",
                          data=np.full((n_pts, n_wl), np.nan), chunks=(1, n_wl))
        rg.create_dataset("T_spec_thr_dB",
                          data=np.full((n_pts, n_wl), np.nan), chunks=(1, n_wl))
        rg.create_dataset("drop_power_dBm",
                          data=np.full(n_pts, np.nan), chunks=(1,))

        f.create_group("flags").create_dataset(
            "computed", data=np.zeros(n_pts, dtype=bool), chunks=(1,))

    log.info(f"HDF5 2-ring initialised ({n_wl} wl pts) → {HDF5_2RING_PATH}")


# ─────────────────────────────────────────────────────────────────────────────
# Motor del sweep de radio
# ─────────────────────────────────────────────────────────────────────────────
def run_radius_sweep(hide_gui: bool = False) -> dict:
    n_pts = RADIUS_SWEEP_N

    wl_m = T_s_thr = T_e_drop = T_e_thr = p_drop = None
    computed   = np.zeros(n_pts, dtype=bool)
    hdf5_ready = False

    # ── Resume desde caché ────────────────────────────────────────────────────
    if HDF5_2RING_PATH.exists():
        log.info(f"Caché 2-ring encontrado → {HDF5_2RING_PATH}")
        try:
            with h5py.File(HDF5_2RING_PATH, "r") as f:
                wl_m    = f["metadata/wavelengths_m"][:]
                T_s_thr = f["results/T_sensor_thr_dB"][:]
                T_e_drop= f["results/T_spec_drop_dB"][:]
                T_e_thr = f["results/T_spec_thr_dB"][:]
                p_drop  = f["results/drop_power_dBm"][:]
                computed[:] = f["flags/computed"][:]
            hdf5_ready = True
            n_cached   = int(computed.sum())
            log.info(f"  Cached: {n_cached}/{n_pts}  |  Remaining: {n_pts-n_cached}")
            if n_pts - n_cached == 0:
                log.info("  Todos los puntos en caché — sin lanzar INTERCONNECT.")
                return _pack_2ring(wl_m, T_s_thr, T_e_drop, T_e_thr, p_drop, computed)
        except Exception as exc:
            log.warning(f"Caché ilegible ({exc}). Empezando desde cero.")
            wl_m = T_s_thr = T_e_drop = T_e_thr = p_drop = None
            computed[:] = False
            hdf5_ready  = False
    else:
        log.info("Sin caché — sweep desde cero.")

    log.info("Lanzando INTERCONNECT para sweep de radio …")
    ic         = lumapi.INTERCONNECT(hide=hide_gui)
    runs_done  = 0
    runs_total = int((~computed).sum())
    t_start    = time.time()

    try:
        _build_2ring_circuit(ic)
        log.info(f"Circuito 2-ring listo — {runs_total} puntos por computar …")

        for s_idx in range(n_pts):
            if computed[s_idx]:
                continue

            circum_val = float(SWEEP_CIRCUM_E_M[s_idx])
            radius_val = float(SWEEP_RADIUS_E_M[s_idx])

            _eval(ic, "switchtodesign")
            _update_re_radius(ic, circum_val)

            # ── Run ───────────────────────────────────────────────────────────
            try:
                _eval(ic, "run")
            except ICScriptError as exc:
                log.warning(f"  RUN FAILED  pt={s_idx:3d}  R={radius_val*1e6:.5f} µm  → {exc}")
                computed[s_idx] = True
                if hdf5_ready:
                    with h5py.File(HDF5_2RING_PATH, "r+") as hf:
                        hf["flags/computed"][s_idx] = True
                        hf.flush()
                continue

            # ── Extract ───────────────────────────────────────────────────────
            try:
                wl_i, ts_i, ted_i, tet_i, pd_i = _extract_2ring_results(ic)
            except Exception as exc:
                log.warning(f"  EXTRACT FAILED  pt={s_idx:3d}: {exc}")
                computed[s_idx] = True
                continue

            # ── Inicializar arrays en el primer punto válido ──────────────────
            if wl_m is None:
                n_wl    = len(wl_i)
                wl_m    = wl_i
                T_s_thr = np.full((n_pts, n_wl), np.nan)
                T_e_drop= np.full((n_pts, n_wl), np.nan)
                T_e_thr = np.full((n_pts, n_wl), np.nan)
                p_drop  = np.full(n_pts, np.nan)
                if not hdf5_ready:
                    _init_hdf5_2ring(wl_i)
                    hdf5_ready = True

            T_s_thr [s_idx, :] = ts_i
            T_e_drop[s_idx, :] = ted_i
            T_e_thr [s_idx, :] = tet_i
            p_drop  [s_idx]    = pd_i
            computed[s_idx]    = True

            # ── Flush HDF5 ────────────────────────────────────────────────────
            with h5py.File(HDF5_2RING_PATH, "r+") as hf:
                hf["results/T_sensor_thr_dB"][s_idx, :] = ts_i
                hf["results/T_spec_drop_dB"] [s_idx, :] = ted_i
                hf["results/T_spec_thr_dB"]  [s_idx, :] = tet_i
                hf["results/drop_power_dBm"] [s_idx]    = pd_i
                hf["flags/computed"]         [s_idx]    = True
                hf.flush()

            runs_done += 1
            if runs_done % 10 == 0 or runs_done == runs_total:
                elapsed = time.time() - t_start
                rate    = runs_done / elapsed if elapsed > 0 else 1e-9
                eta     = (runs_total - runs_done) / rate
                log.info(
                    f"  [{runs_done:3d}/{runs_total}]  "
                    f"R={radius_val*1e6:.5f} µm  │  "
                    f"P_drop={pd_i:.2f} dBm  │  "
                    f"{rate:.2f} sim/s  │  ETA {eta:.0f} s"
                )

        if hdf5_ready:
            with h5py.File(HDF5_2RING_PATH, "r+") as hf:
                hf["metadata"].attrs["timestamp_end"]  = datetime.now().isoformat()
                hf["metadata"].attrs["runs_completed"] = int(computed.sum())

    finally:
        try:
            ic.close()
        except Exception:
            pass
        log.info("INTERCONNECT 2-ring session closed.")

    elapsed = time.time() - t_start
    log.info(
        f"Sweep radio listo │ {runs_done} runs │ "
        f"total={elapsed:.1f} s │ avg={elapsed/max(runs_done,1):.2f} s/sim"
    )
    return _pack_2ring(wl_m, T_s_thr, T_e_drop, T_e_thr, p_drop, computed)


def _pack_2ring(wl, ts, ted, tet, pd, comp) -> dict:
    return dict(
        radius_sweep_m  = SWEEP_RADIUS_E_M,
        circum_sweep_m  = SWEEP_CIRCUM_E_M,
        wavelengths_m   = wl,
        T_sensor_thr_dB = ts,
        T_spec_drop_dB  = ted,
        T_spec_thr_dB   = tet,
        drop_power_dBm  = pd,
        computed        = comp,
    )


# ── Ejecutar sweep ─────────────────────────────────────────────────────────────
radius_sweep_results = run_radius_sweep(hide_gui=False)

rs_wl_m      = radius_sweep_results["wavelengths_m"]
rs_radius_m  = radius_sweep_results["radius_sweep_m"]
rs_circum_m  = radius_sweep_results["circum_sweep_m"]
rs_T_sen_thr = radius_sweep_results["T_sensor_thr_dB"]
rs_T_spc_drp = radius_sweep_results["T_spec_drop_dB"]
rs_T_spc_thr = radius_sweep_results["T_spec_thr_dB"]
rs_p_drop    = radius_sweep_results["drop_power_dBm"]
rs_computed  = radius_sweep_results["computed"]
rs_wl_nm     = rs_wl_m * 1e9 if rs_wl_m is not None else None
rs_circum_um = rs_circum_m * 1e6

print(f"\n  Sweep radio completo — {rs_computed.sum()} / {RADIUS_SWEEP_N} pts")
if rs_wl_m is not None:
    print(f"  T_sensor_thr_dB  shape : {rs_T_sen_thr.shape}")
    print(f"  T_spec_drop_dB   shape : {rs_T_spc_drp.shape}")
    print(f"  drop_power_dBm   shape : {rs_p_drop.shape}")
print(f"  HDF5                   : {HDF5_2RING_PATH}")


# ─────────────────────────────────────────────────────────────────────────────
# Graficación
# ─────────────────────────────────────────────────────────────────────────────
def _rs_mask() -> np.ndarray:
    return radius_sweep_results["computed"].astype(bool)


# ── RS Plot 1 — ★ PRINCIPAL: Potencia en drop del espectrómetro vs radio ★ ──
def plot_rs_drop_power_vs_radius(figsize=(10, 5), save: bool = True) -> plt.Figure:
    """
    Potencia integrada en el drop de RE_SPEC vs radio (y circunferencia).
    Eje X inferior: radio [µm].  Eje X superior: circunferencia [µm].
    """
    mask  = _rs_mask()
    R_v   = rs_radius_m[mask]  * 1e6    # µm
    C_v   = rs_circum_m[mask]  * 1e6    # µm
    P_v   = rs_p_drop[mask]             # dBm

    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(R_v, P_v, color="#2166ac", lw=1.8,
            marker="o", ms=3, alpha=0.9, label="$P_{drop}$ espectrómetro")

    # Marcar radio de diseño del espectrómetro
    R_design_um = RING_RADIUS_M[RE_IDX] * 1e6
    ax.axvline(R_design_um, color="#d6604d", lw=1.2, ls="--",
               label=f"Radio diseño = {R_design_um:.4f} µm")

    ax.set_xlabel("Radio del anillo espectrómetro  $R_E$  (µm)", fontsize=12)
    ax.set_ylabel("Potencia en detector  (dBm)", fontsize=12)
    ax.set_title(
        "Potencia drop del espectrómetro vs radio  [2-ring circuit]\n"
        r"$P_{det} = P_{src} \cdot \langle|T_{drop}(\lambda)|^2\rangle_\lambda$",
        fontsize=12,
    )

    # Eje X superior con circunferencia
    ax2 = ax.twiny()
    ax2.set_xlim(
        ax.get_xlim()[0] * 2.0 * math.pi,
        ax.get_xlim()[1] * 2.0 * math.pi,
    )
    ax2.set_xlabel("Circunferencia  $L_E = 2\\pi R_E$  (µm)", fontsize=10)
    ax2.xaxis.set_minor_locator(ticker.AutoMinorLocator())

    ax.legend(framealpha=0.9, fontsize=9)
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()

    if save:
        fig.savefig(FIGURES_DIR / "rs_drop_power_vs_radius.png")
        fig.savefig(FIGURES_DIR / "rs_drop_power_vs_radius.pdf")
        log.info("Saved → rs_drop_power_vs_radius.png/pdf")
    return fig


# ── RS Plot 2 — Espectros del drop del espectrómetro coloreados por radio ────
def plot_rs_drop_spectra_vs_radius(n_curves: int = 200,
                                    figsize=(10, 5), save: bool = True) -> plt.Figure:
    """
    Espectros de transmisión del drop de RE_SPEC para cada valor de radio.
    Colormap = radio del espectrómetro.
    """
    mask      = _rs_mask()
    valid_idx = np.where(mask)[0]
    n_sel     = min(n_curves, len(valid_idx))
    sel_idx   = valid_idx[
        np.round(np.linspace(0, len(valid_idx) - 1, n_sel)).astype(int)
    ]
    R_sel = rs_radius_m[sel_idx] * 1e6

    cmap = plt.get_cmap("plasma")
    norm = Normalize(vmin=R_sel.min(), vmax=R_sel.max())
    sm   = ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])

    fig, ax = plt.subplots(figsize=figsize)
    for idx in sel_idx:
        ax.plot(rs_wl_nm, rs_T_spc_drp[idx],
                color=cmap(norm(rs_radius_m[idx] * 1e6)),
                lw=0.8, alpha=0.70)
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label("$R_E$  (µm)", fontsize=10)
    ax.set_xlabel("Longitud de onda  (nm)")
    ax.set_ylabel("Transmisión drop  (dB)")
    ax.set_title(
        f"Espectros drop del espectrómetro vs radio  [{n_sel} curvas]\n"
        "Color = $R_E$ [µm]  —  2-ring circuit"
    )
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR / "rs_drop_spectra_vs_radius.png")
        fig.savefig(FIGURES_DIR / "rs_drop_spectra_vs_radius.pdf")
        log.info("Saved → rs_drop_spectra_vs_radius.png/pdf")
    return fig


# ── RS Plot 3 — Heatmap espectral: radio × longitud de onda (drop) ───────────
def plot_rs_drop_heatmap(figsize=(10, 4), cmap_name: str = "inferno",
                          save: bool = True) -> plt.Figure:
    """
    Heatmap:  eje X = longitud de onda [nm],  eje Y = radio [µm],
    color = transmisión drop [dB].
    """
    mask   = _rs_mask()
    R_v    = rs_radius_m[mask] * 1e6
    spec_v = rs_T_spc_drp[mask, :]

    vmin = np.nanpercentile(spec_v, 2)
    vmax = np.nanpercentile(spec_v, 98)

    fig, ax = plt.subplots(figsize=figsize)
    img = ax.pcolormesh(rs_wl_nm, R_v, spec_v,
                        cmap=cmap_name, vmin=vmin, vmax=vmax, shading="auto")
    cbar = fig.colorbar(img, ax=ax, pad=0.01)
    cbar.set_label("Transmisión drop  (dB)", fontsize=10)
    ax.set_xlabel("Longitud de onda  (nm)")
    ax.set_ylabel("Radio espectrómetro  $R_E$  (µm)")
    ax.set_title(
        "Heatmap espectral drop del espectrómetro  [2-ring circuit]\n"
        "Eje Y = $R_E$  —  barrido de radio"
    )
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR / "rs_drop_heatmap_radius_vs_wl.png", dpi=200)
        fig.savefig(FIGURES_DIR / "rs_drop_heatmap_radius_vs_wl.pdf")
        log.info("Saved → rs_drop_heatmap_radius_vs_wl.png/pdf")
    return fig


# ── RS Plot 4 — Sensor through coloreado por radio ────────────────────────────
def plot_rs_sensor_through(n_curves: int = 200,
                            figsize=(10, 5), save: bool = True) -> plt.Figure:
    """Espectros sensor through para cada radio del espectrómetro."""
    mask      = _rs_mask()
    valid_idx = np.where(mask)[0]
    n_sel     = min(n_curves, len(valid_idx))
    sel_idx   = valid_idx[
        np.round(np.linspace(0, len(valid_idx) - 1, n_sel)).astype(int)
    ]
    R_sel = rs_radius_m[sel_idx] * 1e6

    cmap = plt.get_cmap("viridis")
    norm = Normalize(vmin=R_sel.min(), vmax=R_sel.max())
    sm   = ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])

    fig, ax = plt.subplots(figsize=figsize)
    for idx in sel_idx:
        ax.plot(rs_wl_nm, rs_T_sen_thr[idx],
                color=cmap(norm(rs_radius_m[idx] * 1e6)),
                lw=0.8, alpha=0.70)
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label("$R_E$  (µm)", fontsize=10)
    ax.set_xlabel("Longitud de onda  (nm)")
    ax.set_ylabel("Transmisión sensor through  (dB)")
    ax.set_title(
        f"Sensor through vs radio del espectrómetro  [{n_sel} curvas]\n"
        "Color = $R_E$ [µm]  —  2-ring circuit"
    )
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR / "rs_sensor_through_vs_radius.png")
        fig.savefig(FIGURES_DIR / "rs_sensor_through_vs_radius.pdf")
        log.info("Saved → rs_sensor_through_vs_radius.png/pdf")
    return fig


# ── Ejecutar todas las gráficas ───────────────────────────────────────────────
if rs_wl_m is not None and rs_computed.sum() > 0:
    fig_rs1 = plot_rs_drop_power_vs_radius()
    fig_rs2 = plot_rs_drop_spectra_vs_radius(n_curves=200)
    fig_rs3 = plot_rs_drop_heatmap()
    fig_rs4 = plot_rs_sensor_through(n_curves=200)
    plt.show()
    print(f"\n  Figuras sweep radio → {FIGURES_DIR}")
    print(f"  HDF5              → {HDF5_2RING_PATH}")
else:
    print("  Sin datos disponibles para graficar. Ejecuta run_radius_sweep().")

# ═════════════════════════════════════════════════════════════════════════════
#  END CELL 8
# ═════════════════════════════════════════════════════════════════════════════

In [ ]:
RING_KAPPA_INPUT_SQ_03 = np.array([
    0.090264, 0.089812, 0.089773, 0.089734,
    0.089695, 0.089655, 0.089616, 0.089577,
    0.090212, 0.090172, 0.090132, 0.090098,
    0.090058, 0.090018,
])


RING_KAPPA_DROP_SQ_03 = np.array([
    0.087734, 0.087257, 0.087217, 0.087176,
    0.087135, 0.087094, 0.087053, 0.087012,
    0.087627, 0.087586, 0.087544, 0.087508,
    0.087466, 0.087425,
])

In [ ]:
RING_KAPPA_INPUT_SQ_01 = np.array([
    0.031052, 0.030891, 0.030877, 0.030863,
    0.030849, 0.030835, 0.030821, 0.030807,
    0.031033, 0.031019, 0.031005, 0.030993,
    0.030979, 0.030964,
])

RING_KAPPA_DROP_SQ_01 = np.array([
    0.028358, 0.028177, 0.028155, 0.028140,
    0.028124, 0.028109, 0.028093, 0.028077,
    0.028280, 0.028264, 0.028249, 0.028235,
    0.028219, 0.028203,
])

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL A — Sweep V3 con coeficientes de acoplamiento _03                    ║
# ║                                                                            ║
# ║  FWHM objetivo: par kappa_03  (acoplamiento crítico diseño _03)            ║
# ║                                                                            ║
# ║  DEPENDENCIAS HEREDADAS (no se redefinen):                                 ║
# ║    Todas las variables, funciones y constantes de Cells 1–7                ║
# ║    RING_KAPPA_INPUT_SQ_03, RING_KAPPA_DROP_SQ_03  (celda previa)          ║
# ║    n_cladding  (Cell 6)                                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ── Rutas de salida propias para este par de kappas ───────────────────────────
VERSION_NAME_03  = "ICNT_14Ring_Cascade_UniDir_neff_sweep_V3_kappa03"
HDF5_PATH_03     = DATA_DIR / f"{VERSION_NAME_03}.h5"
FIGURES_DIR_03   = DATA_DIR / "figures_kappa03"
FIGURES_DIR_03.mkdir(parents=True, exist_ok=True)

print(f"  Versión kappa_03   : {VERSION_NAME_03}")
print(f"  HDF5               : {HDF5_PATH_03}")
print(f"  Figuras            : {FIGURES_DIR_03}")


# ─────────────────────────────────────────────────────────────────────────────
# Setter de parámetros de anillo para kappa_03
# Idéntico a _apply_ring_params pero usa los nuevos arrays de kappa
# ─────────────────────────────────────────────────────────────────────────────
def _apply_ring_params_03(ic, ring_idx: int,
                           neff_override=None, ng_override=None) -> None:
    name = ring_name(ring_idx + 1)
    neff = neff_override if neff_override is not None else float(RING_NEFF_TE[ring_idx])
    ng   = ng_override   if ng_override   is not None else float(RING_NG_TE[ring_idx])
    pol  = RING_POLARIZATION[ring_idx].upper()
    d_si = (float(RING_D_TE_PS2_PER_KM[ring_idx] if pol == "TE"
                  else RING_D_TM_PS2_PER_KM[ring_idx]) * 1e-15)
    res_hz        = SPEED_OF_LIGHT / float(RING_LAMBDA_RES_M[ring_idx])
    circumference = RING_RADIUS_M[ring_idx] * 2.0 * math.pi

    _eval(ic, f'setnamed("{name}", "length",                   {circumference:.12e})')
    _eval(ic, f'setnamed("{name}", "frequency",                {res_hz:.12e})')
    _eval(ic, f'setnamed("{name}", "effective index 1",        {neff:.12f})')
    _eval(ic, f'setnamed("{name}", "group index 1",            {ng:.12f})')
    _eval(ic, f'setnamed("{name}", "loss 1",                   {RING_LOSS_DB_PER_M[ring_idx]:.6f})')
    _eval(ic, f'setnamed("{name}", "dispersion 1",             {d_si:.12e})')
    _eval(ic, f'setnamed("{name}", "coupling coefficient 1 1", {RING_KAPPA_INPUT_SQ_03[ring_idx]:.12f})')
    _eval(ic, f'setnamed("{name}", "coupling coefficient 1 2", {RING_KAPPA_DROP_SQ_03[ring_idx]:.12f})')
    _eval(ic, f'setnamed("{name}", "configuration", "unidirectional")')


# ─────────────────────────────────────────────────────────────────────────────
# Constructor de circuito para kappa_03
# Topología V3 idéntica; único cambio: _apply_ring_params_03
# ─────────────────────────────────────────────────────────────────────────────
def _build_circuit_03(ic) -> None:
    _eval(ic, "switchtodesign")
    _try_eval(ic, "selectall")
    _try_eval(ic, "delete")

    pwr_W   = 10.0 ** (ONA_POWER_DBM / 10.0) * 1e-3
    f_start = SPEED_OF_LIGHT / ONA_LAMBDA_STOP_M
    f_stop  = SPEED_OF_LIGHT / ONA_LAMBDA_START_M

    # ONA — 15 puertos de entrada
    _eval(ic, 'addelement("Optical Network Analyzer")')
    _eval(ic, f'set("name", "{ONA_NAME}")')
    _try_eval(ic, 'set("x position", 0)')
    _try_eval(ic, 'set("y position", 0)')
    _eval(ic, f'setnamed("{ONA_NAME}", "input parameter",       "start and stop")')
    _eval(ic, f'setnamed("{ONA_NAME}", "number of input ports", {int(ONA_N_INPUT_PORTS)})')
    _eval(ic, f'setnamed("{ONA_NAME}", "start frequency",       {f_start:.12e})')
    _eval(ic, f'setnamed("{ONA_NAME}", "stop frequency",        {f_stop:.12e})')
    _eval(ic, f'setnamed("{ONA_NAME}", "number of points",      {int(ONA_N_POINTS)})')
    _eval(ic, f'setnamed("{ONA_NAME}", "power",                 {pwr_W:.12e})')

    # Anillos con kappa_03
    for i in range(N_RINGS):
        rn = ring_name(i + 1)
        _eval(ic, 'addelement("Double Bus Ring Resonator")')
        _eval(ic, f'set("name", "{rn}")')
        _try_eval(ic, f'set("x position", {float((i + 1) * 220)})')
        _try_eval(ic, f'set("y position", 0)')
        _apply_ring_params_03(ic, ring_idx=i)
        log.info(
            f"  [03] RING_{i+1:2d} — "
            f"κ²_in={RING_KAPPA_INPUT_SQ_03[i]:.6f}  "
            f"κ²_dr={RING_KAPPA_DROP_SQ_03[i]:.6f}"
        )

    # Conexiones — topología V3 idéntica a _build_circuit
    def wire(elem_a, port_a, elem_b, port_b):
        _eval(ic, f'connect("{elem_a}", "{port_a}", "{elem_b}", "{port_b}")')

    wire(ONA_NAME,        "output",   ring_name(1), "input")
    wire(ring_name(1),    "output 1", ONA_NAME,     f"input {THROUGH_SENSOR_INPUT}")
    wire(ring_name(1),    "output 2", ring_name(2), "input")
    for i in range(2, N_RINGS):
        wire(ring_name(i), "output 1", ring_name(i + 1), "input")
        wire(ring_name(i), "output 2", ONA_NAME, f"input {drop_ona_input(i)}")
    wire(ring_name(N_RINGS), "output 2", ONA_NAME, f"input {drop_ona_input(N_RINGS)}")
    wire(ring_name(N_RINGS), "output 1", ONA_NAME, f"input {THROUGH_FINAL_INPUT}")

    log.info("V3-kappa03 circuit built.")


# ─────────────────────────────────────────────────────────────────────────────
# HDF5 para kappa_03
# ─────────────────────────────────────────────────────────────────────────────
def _init_hdf5_03(wl_ref_m: np.ndarray) -> None:
    n_pts = SWEEP_N_POINTS
    n_wl  = len(wl_ref_m)
    with h5py.File(HDF5_PATH_03, "w") as f:
        md = f.create_group("metadata")
        md.create_dataset("neff_sweep",    data=SWEEP_NEFF)
        md.create_dataset("ng_sweep",      data=SWEEP_NG)
        md.create_dataset("wavelengths_m", data=wl_ref_m)
        md.attrs["version_name"]        = VERSION_NAME_03
        md.attrs["kappa_variant"]       = "kappa_03"
        md.attrs["n_rings"]             = N_RINGS
        md.attrs["n_drops"]             = N_DROPS
        md.attrs["sweep_n_points"]      = SWEEP_N_POINTS
        md.attrs["ring_model"]          = "Double Bus Ring Resonator"
        md.attrs["ring_configuration"]  = "unidirectional"
        md.attrs["topology"]            = (
            "V3-kappa03: ONA multiport, no OPMs. "
            "ONA input 1=RING_1 through, input k=RING_k drop (k=2-14), "
            "input 15=RING_14 through"
        )
        md.attrs["ona_lambda_start_m"]  = ONA_LAMBDA_START_M
        md.attrs["ona_lambda_stop_m"]   = ONA_LAMBDA_STOP_M
        md.attrs["ona_n_points"]        = ONA_N_POINTS
        md.attrs["ona_power_dBm"]       = ONA_POWER_DBM
        md.attrs["ona_n_input_ports"]   = ONA_N_INPUT_PORTS
        md.attrs["power_calc_method"]   = (
            "P_det[W] = P_source[W] * mean(|T_drop(lambda)|^2) over ONA band"
        )
        md.attrs["timestamp_start"]     = datetime.now().isoformat()
        for i in range(N_RINGS):
            p = f"ring{i+1}_"
            md.attrs[p + "kappa_input_sq"] = RING_KAPPA_INPUT_SQ_03[i]
            md.attrs[p + "kappa_drop_sq"]  = RING_KAPPA_DROP_SQ_03[i]
            md.attrs[p + "radius_m"]       = RING_RADIUS_M[i]
            md.attrs[p + "lambda_res_m"]   = RING_LAMBDA_RES_M[i]
            md.attrs[p + "neff_TE"]        = RING_NEFF_TE[i]
            md.attrs[p + "ng_TE"]          = RING_NG_TE[i]
            md.attrs[p + "loss_dB_per_m"]  = RING_LOSS_DB_PER_M[i]

        rg = f.create_group("results")
        rg.create_dataset("T_sensor_through_dB",
                          data=np.full((n_pts, n_wl), np.nan), chunks=(1, n_wl))
        rg.create_dataset("T_final_through_dB",
                          data=np.full((n_pts, n_wl), np.nan), chunks=(1, n_wl))
        rg.create_dataset("T_drop_dB",
                          data=np.full((n_pts, N_DROPS, n_wl), np.nan),
                          chunks=(1, 1, n_wl))
        rg.create_dataset("drop_power_dBm",
                          data=np.full((n_pts, N_DROPS), np.nan),
                          chunks=(1, N_DROPS))
        f.create_group("flags").create_dataset(
            "computed", data=np.zeros(n_pts, dtype=bool), chunks=(1,))

    log.info(f"HDF5 kappa03 inicializado → {HDF5_PATH_03}")


# ─────────────────────────────────────────────────────────────────────────────
# Motor del sweep para kappa_03
# ─────────────────────────────────────────────────────────────────────────────
def run_interconnect_sweep_03(hide_gui: bool = False):
    n_pts = SWEEP_N_POINTS
    wavelengths_m = T_sensor_thr = T_final_thr = T_drop = p_drop = None
    computed_03   = np.zeros(n_pts, dtype=bool)
    hdf5_ready    = False

    if HDF5_PATH_03.exists():
        log.info(f"[03] Caché encontrado → {HDF5_PATH_03}")
        try:
            with h5py.File(HDF5_PATH_03, "r") as f:
                wavelengths_m = f["metadata/wavelengths_m"][:]
                T_sensor_thr  = f["results/T_sensor_through_dB"][:]
                T_final_thr   = f["results/T_final_through_dB"][:]
                T_drop        = f["results/T_drop_dB"][:]
                p_drop        = f["results/drop_power_dBm"][:]
                computed_03[:]= f["flags/computed"][:]
            hdf5_ready = True
            n_cached   = int(computed_03.sum())
            log.info(f"[03] Cached: {n_cached}/{n_pts}  |  Pending: {n_pts - n_cached}")
            if n_pts - n_cached == 0:
                log.info("[03] Todos los puntos en caché — sin lanzar INTERCONNECT.")
                return _pack_results_03(
                    wavelengths_m, T_sensor_thr, T_final_thr, T_drop, p_drop, computed_03)
        except Exception as exc:
            log.warning(f"[03] Caché ilegible ({exc}). Empezando desde cero.")
            wavelengths_m = T_sensor_thr = T_final_thr = T_drop = p_drop = None
            computed_03[:] = False
            hdf5_ready = False
    else:
        log.info("[03] Sin caché — sweep desde cero.")

    log.info("[03] Lanzando INTERCONNECT …")
    ic         = lumapi.INTERCONNECT(hide=hide_gui)
    runs_done  = 0
    runs_total = int((~computed_03).sum())
    t_start    = time.time()

    try:
        _build_circuit_03(ic)
        log.info(f"[03] Circuito listo — {runs_total} puntos por computar …")

        for s_idx in range(n_pts):
            if computed_03[s_idx]:
                continue

            neff_val = float(SWEEP_NEFF[s_idx])
            ng_val   = float(SWEEP_NG[s_idx])

            _eval(ic, "switchtodesign")
            _update_ring1_neff_ng(ic, neff_val, ng_val)   # reutiliza la función de Cell 3

            try:
                _eval(ic, "run")
            except ICScriptError as exc:
                log.warning(f"  [03] RUN FAILED pt={s_idx:3d} neff={neff_val:.6f} → {exc}")
                computed_03[s_idx] = True
                if hdf5_ready:
                    with h5py.File(HDF5_PATH_03, "r+") as hf:
                        hf["flags/computed"][s_idx] = True; hf.flush()
                continue

            try:
                wl_m, t_sen, t_fin, t_drp, p_drp = _extract_results(ic)  # reutiliza Cell 3
            except Exception as exc:
                log.warning(f"  [03] EXTRACT FAILED pt={s_idx:3d}: {exc}")
                computed_03[s_idx] = True
                continue

            if wavelengths_m is None:
                n_wl          = len(wl_m)
                wavelengths_m = wl_m
                T_sensor_thr  = np.full((n_pts, n_wl),          np.nan)
                T_final_thr   = np.full((n_pts, n_wl),          np.nan)
                T_drop        = np.full((n_pts, N_DROPS, n_wl), np.nan)
                p_drop        = np.full((n_pts, N_DROPS),        np.nan)
                if not hdf5_ready:
                    _init_hdf5_03(wl_m)
                    hdf5_ready = True

            T_sensor_thr[s_idx, :]    = t_sen
            T_final_thr [s_idx, :]    = t_fin
            T_drop      [s_idx, :, :] = t_drp
            p_drop      [s_idx, :]    = p_drp
            computed_03 [s_idx]        = True

            with h5py.File(HDF5_PATH_03, "r+") as hf:
                hf["results/T_sensor_through_dB"][s_idx, :]    = t_sen
                hf["results/T_final_through_dB"] [s_idx, :]    = t_fin
                hf["results/T_drop_dB"]          [s_idx, :, :] = t_drp
                hf["results/drop_power_dBm"]     [s_idx, :]    = p_drp
                hf["flags/computed"]             [s_idx]        = True
                hf.flush()

            runs_done += 1
            if runs_done % 5 == 0 or runs_done == runs_total:
                elapsed = time.time() - t_start
                rate    = runs_done / elapsed if elapsed > 0 else 1e-9
                eta     = (runs_total - runs_done) / rate
                log.info(
                    f"  [03][{runs_done:3d}/{runs_total}]  "
                    f"neff={neff_val:.6f}  ng={ng_val:.6f}  │  "
                    f"{rate:.2f} sim/s  │  ETA {eta:5.0f} s"
                )

        if hdf5_ready:
            with h5py.File(HDF5_PATH_03, "r+") as hf:
                hf["metadata"].attrs["timestamp_end"]  = datetime.now().isoformat()
                hf["metadata"].attrs["runs_completed"] = int(computed_03.sum())

    finally:
        try: ic.close()
        except Exception: pass
        log.info("[03] INTERCONNECT session closed.")

    elapsed = time.time() - t_start
    log.info(
        f"[03] Sweep kappa03 listo │ {runs_done} runs │ "
        f"total={elapsed:.1f} s │ avg={elapsed/max(runs_done,1):.2f} s/sim"
    )
    return _pack_results_03(
        wavelengths_m, T_sensor_thr, T_final_thr, T_drop, p_drop, computed_03)


def _pack_results_03(wl, t_sen, t_fin, t_drop, p_drop, comp):
    return dict(
        neff_sweep            = SWEEP_NEFF,
        ng_sweep              = SWEEP_NG,
        wavelengths_m         = wl,
        T_sensor_through_dB   = t_sen,
        T_final_through_dB    = t_fin,
        T_drop_dB             = t_drop,
        drop_power_dBm        = p_drop,
        computed              = comp,
    )


# ── Ejecutar sweep kappa_03 ────────────────────────────────────────────────────
sweep_results_03 = run_interconnect_sweep_03(hide_gui=False)

wl_03          = sweep_results_03["wavelengths_m"]
wl_nm_03       = wl_03 * 1e9 if wl_03 is not None else None
computed_03    = sweep_results_03["computed"]

print(f"\n  [kappa_03] Sweep completo — {computed_03.sum()}/{SWEEP_N_POINTS} pts")
if wl_03 is not None:
    print(f"  T_sensor_through_dB shape : {sweep_results_03['T_sensor_through_dB'].shape}")
    print(f"  T_drop_dB           shape : {sweep_results_03['T_drop_dB'].shape}")
    print(f"  drop_power_dBm      shape : {sweep_results_03['drop_power_dBm'].shape}")
print(f"  HDF5                      : {HDF5_PATH_03}")


# ─────────────────────────────────────────────────────────────────────────────
# Protocolo de gráficas completo — kappa_03
# Réplica exacta de Cell 4 + Cell 7, con ejes neff y n_cladding
# ─────────────────────────────────────────────────────────────────────────────
def _valid_mask_03():
    return sweep_results_03["computed"].astype(bool)


# ── Plot 03-1 — Potencia integrada drop vs neff ────────────────────────────────
def plot_03_drop_power_vs_neff(figsize=(11, 6), save=True):
    mask   = _valid_mask_03()
    neff_v = sweep_results_03["neff_sweep"][mask]
    p_v    = sweep_results_03["drop_power_dBm"][mask, :]
    cmap   = plt.get_cmap("tab20")
    fig, ax = plt.subplots(figsize=figsize)
    for k in range(N_DROPS):
        ax.plot(neff_v, p_v[:, k], color=cmap(k / N_DROPS),
                lw=1.5, marker="o", ms=2.5, alpha=0.85, label=DROP_LABELS[k])
    ax.set_xlabel("Ring 1 — $n_{eff}$ (TE)  [sensor]", fontsize=12)
    ax.set_ylabel("Detector power  (dBm)", fontsize=12)
    ax.set_title(
        "Integrated Drop Power vs Sensor $n_{eff}$  [V3 — kappa_03]\n"
        r"$P_{det} = P_{src} \cdot \langle|T_{drop}(\lambda)|^2\rangle_\lambda$"
        "   —   13 spectrometer rings", fontsize=12)
    ax.legend(ncol=3, framealpha=0.88, fontsize=8, loc="best")
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR_03 / "drop_power_vs_neff_all.png")
        fig.savefig(FIGURES_DIR_03 / "drop_power_vs_neff_all.pdf")
    return fig


# ── Plot 03-2 — Espectros sensor through coloreados por neff ──────────────────
def plot_03_sensor_through_sweep(n_curves=200, figsize=(10, 5),
                                  cmap_name="plasma", save=True):
    neff_arr  = sweep_results_03["neff_sweep"]
    wl_nm     = sweep_results_03["wavelengths_m"] * 1e9
    T_data    = sweep_results_03["T_sensor_through_dB"]
    mask      = _valid_mask_03()
    valid_idx = np.where(mask)[0]
    n_sel     = min(n_curves, len(valid_idx))
    sel_idx   = valid_idx[np.round(np.linspace(0, len(valid_idx)-1, n_sel)).astype(int)]
    cmap = plt.get_cmap(cmap_name)
    norm = Normalize(vmin=neff_arr[sel_idx].min(), vmax=neff_arr[sel_idx].max())
    sm   = ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    fig, ax = plt.subplots(figsize=figsize)
    for idx in sel_idx:
        ax.plot(wl_nm, T_data[idx], color=cmap(norm(neff_arr[idx])), lw=0.8, alpha=0.70)
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label("Ring 1 — $n_{eff}$ (TE)", fontsize=10)
    ax.set_xlabel("Wavelength  (nm)"); ax.set_ylabel("Transmission  (dB)")
    ax.set_title(f"Sensor Through Spectrum — kappa_03\n({n_sel} curves, colour = $n_{{eff,1}}$)")
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR_03 / f"sensor_through_sweep_{n_sel}curves.png")
        fig.savefig(FIGURES_DIR_03 / f"sensor_through_sweep_{n_sel}curves.pdf")
    return fig


# ── Plot 03-3 — Espectros final through coloreados por neff ───────────────────
def plot_03_final_through_sweep(n_curves=200, figsize=(10, 5),
                                 cmap_name="viridis", save=True):
    neff_arr  = sweep_results_03["neff_sweep"]
    wl_nm     = sweep_results_03["wavelengths_m"] * 1e9
    T_data    = sweep_results_03["T_final_through_dB"]
    mask      = _valid_mask_03()
    valid_idx = np.where(mask)[0]
    n_sel     = min(n_curves, len(valid_idx))
    sel_idx   = valid_idx[np.round(np.linspace(0, len(valid_idx)-1, n_sel)).astype(int)]
    cmap = plt.get_cmap(cmap_name)
    norm = Normalize(vmin=neff_arr[sel_idx].min(), vmax=neff_arr[sel_idx].max())
    sm   = ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    fig, ax = plt.subplots(figsize=figsize)
    for idx in sel_idx:
        ax.plot(wl_nm, T_data[idx], color=cmap(norm(neff_arr[idx])), lw=0.8, alpha=0.70)
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label("Ring 1 — $n_{eff}$ (TE)", fontsize=10)
    ax.set_xlabel("Wavelength  (nm)"); ax.set_ylabel("Transmission  (dB)")
    ax.set_title(f"Cascade Final Through — kappa_03\n({n_sel} curves, colour = $n_{{eff,1}}$)")
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR_03 / f"final_through_sweep_{n_sel}curves.png")
        fig.savefig(FIGURES_DIR_03 / f"final_through_sweep_{n_sel}curves.pdf")
    return fig


# ── Plot 03-4 — Heatmap espectral de drop individual ─────────────────────────
def plot_03_drop_spectrum_heatmap(drop_k=1, figsize=(10, 3.5),
                                   cmap_name="inferno",
                                   vmin_dB=None, vmax_dB=None, save=True):
    assert 1 <= drop_k <= N_DROPS
    ring_label = DROP_LABELS[drop_k - 1]
    neff_arr   = sweep_results_03["neff_sweep"]
    wl_nm      = sweep_results_03["wavelengths_m"] * 1e9
    t_drop     = sweep_results_03["T_drop_dB"]
    mask       = _valid_mask_03()
    neff_v     = neff_arr[mask]
    spec_v     = t_drop[mask, drop_k - 1, :]
    _vmin = vmin_dB if vmin_dB is not None else np.nanpercentile(spec_v, 2)
    _vmax = vmax_dB if vmax_dB is not None else np.nanpercentile(spec_v, 98)
    fig, ax = plt.subplots(figsize=figsize)
    img = ax.pcolormesh(wl_nm, neff_v, spec_v,
                        cmap=cmap_name, vmin=_vmin, vmax=_vmax, shading="auto")
    cbar = fig.colorbar(img, ax=ax, pad=0.01)
    cbar.set_label("Transmission  (dB)", fontsize=10)
    ax.set_xlabel("Wavelength  (nm)"); ax.set_ylabel("Ring 1 — $n_{eff}$ (TE)")
    ax.set_title(
        f"Drop Spectrum Heatmap — {ring_label}  [kappa_03]\n"
        "neff sweep on sensor ring (RING_1)")
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR_03 / f"drop{drop_k}_spectrum_heatmap.png")
        fig.savefig(FIGURES_DIR_03 / f"drop{drop_k}_spectrum_heatmap.pdf")
    return fig


# ── Plot 03-5 — Grid con los 13 heatmaps de drop ─────────────────────────────
def plot_03_all_drop_heatmaps(ncols=4, cmap_name="inferno", save=True):
    neff_arr = sweep_results_03["neff_sweep"]
    wl_nm    = sweep_results_03["wavelengths_m"] * 1e9
    t_drop   = sweep_results_03["T_drop_dB"]
    mask     = _valid_mask_03()
    neff_v   = neff_arr[mask]
    nrows    = math.ceil(N_DROPS / ncols)
    fig, axes = plt.subplots(nrows, ncols,
                              figsize=(ncols * 3.8, nrows * 2.8),
                              sharex=True, sharey=True)
    axes_flat = axes.flatten()
    all_vals  = t_drop[mask].ravel()
    vmin = np.nanpercentile(all_vals, 2); vmax = np.nanpercentile(all_vals, 98)
    im = None
    for k in range(1, N_DROPS + 1):
        ax   = axes_flat[k - 1]
        spec = t_drop[mask, k - 1, :]
        im   = ax.pcolormesh(wl_nm, neff_v, spec,
                             cmap=cmap_name, vmin=vmin, vmax=vmax, shading="auto")
        ax.set_title(f"{DROP_LABELS[k-1]} drop", fontsize=8)
        if k % ncols == 1: ax.set_ylabel("$n_{eff,1}$", fontsize=7)
        if k > (nrows - 1) * ncols: ax.set_xlabel("λ (nm)", fontsize=7)
        ax.tick_params(labelsize=6)
    for ax in axes_flat[N_DROPS:]: ax.set_visible(False)
    if im is not None:
        fig.colorbar(im, ax=axes_flat[:N_DROPS], shrink=0.6, pad=0.02,
                     label="Transmission (dB)", fraction=0.015)
    fig.suptitle(
        "All Drop Spectra  [V3 — kappa_03]\nneff sweep on sensor ring (RING_1)",
        fontsize=11)
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR_03 / "all_drop_heatmaps_grid.png", dpi=200)
        fig.savefig(FIGURES_DIR_03 / "all_drop_heatmaps_grid.pdf")
    return fig


# ── Plot 03-6 — Heatmap 2D potencia: drops × neff ────────────────────────────
def plot_03_power_heatmap_drops_vs_neff(figsize=(10, 5),
                                         cmap_name="plasma", save=True):
    neff_arr = sweep_results_03["neff_sweep"]
    p_drop   = sweep_results_03["drop_power_dBm"]
    mask     = _valid_mask_03()
    neff_v   = neff_arr[mask]
    p_v      = p_drop[mask, :].T
    fig, ax  = plt.subplots(figsize=figsize)
    img = ax.pcolormesh(neff_v, np.arange(1, N_DROPS + 1), p_v,
                        cmap=cmap_name, shading="auto")
    cbar = fig.colorbar(img, ax=ax, pad=0.02)
    cbar.set_label("Detector power  (dBm)", fontsize=10)
    ax.set_xlabel("Ring 1 — $n_{eff}$ (TE)  [sensor]", fontsize=11)
    ax.set_ylabel("Spectrometer ring index  (1=RING_2 … 13=RING_14)", fontsize=10)
    ax.set_yticks(np.arange(1, N_DROPS + 1))
    ax.set_yticklabels(DROP_LABELS, fontsize=7)
    ax.set_title(
        "Detector Power Heatmap — All Drops vs $n_{eff}$  [kappa_03]\n"
        r"Colour: $P_{det}$ [dBm]  per spectrometer ring", fontsize=12)
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR_03 / "power_heatmap_drops_vs_neff.png", dpi=200)
        fig.savefig(FIGURES_DIR_03 / "power_heatmap_drops_vs_neff.pdf")
    return fig


# ── Plot 03-7 — Tracking de resonancia del sensor vs neff ─────────────────────
def plot_03_resonance_tracking(figsize=(7, 4.5), save=True):
    neff_arr = sweep_results_03["neff_sweep"]
    wl_nm    = sweep_results_03["wavelengths_m"] * 1e9
    T_data   = sweep_results_03["T_sensor_through_dB"]
    mask     = _valid_mask_03()
    neff_v   = neff_arr[mask]
    T_v      = T_data[mask, :]
    dip_idx  = np.argmin(T_v, axis=1)
    lam_dip  = wl_nm[dip_idx]
    coeffs   = np.polyfit(neff_v, lam_dip, 1)
    sens     = coeffs[0]
    fig, ax  = plt.subplots(figsize=figsize)
    ax.scatter(neff_v, lam_dip, s=20, zorder=5, color="#2166ac", label="Resonance dip")
    ax.plot(neff_v, np.poly1d(coeffs)(neff_v), "r--", lw=1.5,
            label=f"Linear fit   ∂λ/∂n = {sens:.3f} nm / RIU")
    ax.set_xlabel("Ring 1 — $n_{eff}$ (TE)")
    ax.set_ylabel("Resonance wavelength  (nm)")
    ax.set_title(
        f"Resonance Tracking — kappa_03\nSensitivity: {sens:.3f} nm / RIU")
    ax.legend(framealpha=0.9)
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR_03 / "resonance_tracking_sensor.png")
        fig.savefig(FIGURES_DIR_03 / "resonance_tracking_sensor.pdf")
    log.info(f"[03] Resonance sensitivity: {sens:.4f} nm/RIU")
    return fig


# ── Plots NC (n_cladding en eje X) — kappa_03 ─────────────────────────────────

def plot_03_drop_power_vs_ncladding(figsize=(11, 6), save=True):
    mask = _valid_mask_03()
    nc_v = n_cladding[mask]
    p_v  = sweep_results_03["drop_power_dBm"][mask, :]
    cmap = plt.get_cmap("tab20")
    fig, ax = plt.subplots(figsize=figsize)
    for k in range(N_DROPS):
        ax.plot(nc_v, p_v[:, k], color=cmap(k / N_DROPS),
                lw=1.5, marker="o", ms=2.5, alpha=0.85, label=DROP_LABELS[k])
    ax.set_xlabel("Índice de refracción del cladding  $n_{clad}$", fontsize=12)
    ax.set_ylabel("Potencia en el detector  (dBm)", fontsize=12)
    ax.set_title(
        "Potencia integrada en drop vs $n_{clad}$  [kappa_03]\n"
        r"$P_{det} = P_{src} \cdot \langle|T_{drop}(\lambda)|^2\rangle_\lambda$"
        "   —   13 anillos espectrómetros", fontsize=12)
    ax.legend(ncol=3, framealpha=0.88, fontsize=8, loc="best")
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR_03 / "nc_drop_power_vs_ncladding_all.png")
        fig.savefig(FIGURES_DIR_03 / "nc_drop_power_vs_ncladding_all.pdf")
    return fig


def plot_03_sensor_through_sweep_nc(n_curves=200, figsize=(10, 5),
                                     cmap_name="plasma", save=True):
    wl_nm     = sweep_results_03["wavelengths_m"] * 1e9
    T_data    = sweep_results_03["T_sensor_through_dB"]
    mask      = _valid_mask_03()
    valid_idx = np.where(mask)[0]
    n_sel     = min(n_curves, len(valid_idx))
    sel_idx   = valid_idx[np.round(np.linspace(0, len(valid_idx)-1, n_sel)).astype(int)]
    nc_sel    = n_cladding[sel_idx]
    cmap = plt.get_cmap(cmap_name)
    norm = Normalize(vmin=nc_sel.min(), vmax=nc_sel.max())
    sm   = ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    fig, ax = plt.subplots(figsize=figsize)
    for idx in sel_idx:
        ax.plot(wl_nm, T_data[idx], color=cmap(norm(n_cladding[idx])), lw=0.8, alpha=0.70)
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label("$n_{clad}$", fontsize=10)
    ax.set_xlabel("Longitud de onda  (nm)"); ax.set_ylabel("Transmisión  (dB)")
    ax.set_title(f"Espectro sensor through — kappa_03\n({n_sel} curvas, color = $n_{{clad}}$)")
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR_03 / f"nc_sensor_through_sweep_{n_sel}curves.png")
        fig.savefig(FIGURES_DIR_03 / f"nc_sensor_through_sweep_{n_sel}curves.pdf")
    return fig


def plot_03_final_through_sweep_nc(n_curves=200, figsize=(10, 5),
                                    cmap_name="viridis", save=True):
    wl_nm     = sweep_results_03["wavelengths_m"] * 1e9
    T_data    = sweep_results_03["T_final_through_dB"]
    mask      = _valid_mask_03()
    valid_idx = np.where(mask)[0]
    n_sel     = min(n_curves, len(valid_idx))
    sel_idx   = valid_idx[np.round(np.linspace(0, len(valid_idx)-1, n_sel)).astype(int)]
    nc_sel    = n_cladding[sel_idx]
    cmap = plt.get_cmap(cmap_name)
    norm = Normalize(vmin=nc_sel.min(), vmax=nc_sel.max())
    sm   = ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    fig, ax = plt.subplots(figsize=figsize)
    for idx in sel_idx:
        ax.plot(wl_nm, T_data[idx], color=cmap(norm(n_cladding[idx])), lw=0.8, alpha=0.70)
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label("$n_{clad}$", fontsize=10)
    ax.set_xlabel("Longitud de onda  (nm)"); ax.set_ylabel("Transmisión  (dB)")
    ax.set_title(f"Through final de cascada — kappa_03\n({n_sel} curvas, color = $n_{{clad}}$)")
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR_03 / f"nc_final_through_sweep_{n_sel}curves.png")
        fig.savefig(FIGURES_DIR_03 / f"nc_final_through_sweep_{n_sel}curves.pdf")
    return fig


def plot_03_drop_spectrum_heatmap_nc(drop_k=1, figsize=(10, 3.5),
                                      cmap_name="inferno",
                                      vmin_dB=None, vmax_dB=None, save=True):
    assert 1 <= drop_k <= N_DROPS
    ring_label = DROP_LABELS[drop_k - 1]
    wl_nm  = sweep_results_03["wavelengths_m"] * 1e9
    t_drop = sweep_results_03["T_drop_dB"]
    mask   = _valid_mask_03()
    nc_v   = n_cladding[mask]
    spec_v = t_drop[mask, drop_k - 1, :]
    _vmin  = vmin_dB if vmin_dB is not None else np.nanpercentile(spec_v, 2)
    _vmax  = vmax_dB if vmax_dB is not None else np.nanpercentile(spec_v, 98)
    fig, ax = plt.subplots(figsize=figsize)
    img = ax.pcolormesh(wl_nm, nc_v, spec_v,
                        cmap=cmap_name, vmin=_vmin, vmax=_vmax, shading="auto")
    cbar = fig.colorbar(img, ax=ax, pad=0.01)
    cbar.set_label("Transmisión  (dB)", fontsize=10)
    ax.set_xlabel("Longitud de onda  (nm)"); ax.set_ylabel("$n_{clad}$")
    ax.set_title(
        f"Heatmap espectral drop — {ring_label}  [kappa_03]\n"
        f"Barrido de $n_{{clad}}$ sobre RING_1")
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR_03 / f"nc_drop{drop_k}_spectrum_heatmap.png")
        fig.savefig(FIGURES_DIR_03 / f"nc_drop{drop_k}_spectrum_heatmap.pdf")
    return fig


def plot_03_all_drop_heatmaps_nc(ncols=4, cmap_name="inferno", save=True):
    wl_nm  = sweep_results_03["wavelengths_m"] * 1e9
    t_drop = sweep_results_03["T_drop_dB"]
    mask   = _valid_mask_03()
    nc_v   = n_cladding[mask]
    nrows  = math.ceil(N_DROPS / ncols)
    fig, axes = plt.subplots(nrows, ncols,
                              figsize=(ncols * 3.8, nrows * 2.8),
                              sharex=True, sharey=True)
    axes_flat = axes.flatten()
    all_vals  = t_drop[mask].ravel()
    vmin = np.nanpercentile(all_vals, 2); vmax = np.nanpercentile(all_vals, 98)
    im = None
    for k in range(1, N_DROPS + 1):
        ax   = axes_flat[k - 1]
        spec = t_drop[mask, k - 1, :]
        im   = ax.pcolormesh(wl_nm, nc_v, spec,
                             cmap=cmap_name, vmin=vmin, vmax=vmax, shading="auto")
        ax.set_title(f"{DROP_LABELS[k-1]} drop", fontsize=8)
        if k % ncols == 1: ax.set_ylabel("$n_{clad}$", fontsize=7)
        if k > (nrows - 1) * ncols: ax.set_xlabel("λ (nm)", fontsize=7)
        ax.tick_params(labelsize=6)
    for ax in axes_flat[N_DROPS:]: ax.set_visible(False)
    if im is not None:
        fig.colorbar(im, ax=axes_flat[:N_DROPS], shrink=0.6, pad=0.02,
                     label="Transmisión (dB)", fraction=0.015)
    fig.suptitle(
        "Espectros drop — 13 anillos  [kappa_03]\nBarrido de $n_{clad}$ sobre RING_1",
        fontsize=11)
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR_03 / "nc_all_drop_heatmaps_grid.png", dpi=200)
        fig.savefig(FIGURES_DIR_03 / "nc_all_drop_heatmaps_grid.pdf")
    return fig


def plot_03_power_heatmap_drops_vs_ncladding(figsize=(10, 5),
                                              cmap_name="plasma", save=True):
    mask = _valid_mask_03()
    nc_v = n_cladding[mask]
    p_v  = sweep_results_03["drop_power_dBm"][mask, :].T
    fig, ax = plt.subplots(figsize=figsize)
    img = ax.pcolormesh(nc_v, np.arange(1, N_DROPS + 1), p_v,
                        cmap=cmap_name, shading="auto")
    cbar = fig.colorbar(img, ax=ax, pad=0.02)
    cbar.set_label("Potencia en detector  (dBm)", fontsize=10)
    ax.set_xlabel("Índice de refracción del cladding  $n_{clad}$", fontsize=11)
    ax.set_ylabel("Índice de anillo espectrómetro  (1=RING_2 … 13=RING_14)", fontsize=10)
    ax.set_yticks(np.arange(1, N_DROPS + 1)); ax.set_yticklabels(DROP_LABELS, fontsize=7)
    ax.set_title(
        "Heatmap de potencia — Todos los drops vs $n_{clad}$  [kappa_03]\n"
        r"Color: $P_{det}$ [dBm]  por anillo espectrómetro", fontsize=12)
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR_03 / "nc_power_heatmap_drops_vs_ncladding.png", dpi=200)
        fig.savefig(FIGURES_DIR_03 / "nc_power_heatmap_drops_vs_ncladding.pdf")
    return fig


def plot_03_resonance_tracking_nc(figsize=(7, 4.5), save=True):
    wl_nm  = sweep_results_03["wavelengths_m"] * 1e9
    T_data = sweep_results_03["T_sensor_through_dB"]
    mask   = _valid_mask_03()
    nc_v   = n_cladding[mask]
    T_v    = T_data[mask, :]
    dip_idx = np.argmin(T_v, axis=1)
    lam_dip = wl_nm[dip_idx]
    coeffs  = np.polyfit(nc_v, lam_dip, 1)
    sens    = coeffs[0]
    r2      = 1.0 - np.sum((lam_dip - np.poly1d(coeffs)(nc_v))**2) / \
                    np.sum((lam_dip - lam_dip.mean())**2)
    fig, ax = plt.subplots(figsize=figsize)
    ax.scatter(nc_v, lam_dip, s=20, zorder=5,
               color="#2166ac", label="Mínimo de transmisión (dip)")
    ax.plot(nc_v, np.poly1d(coeffs)(nc_v), "r--", lw=1.5,
            label=(f"Ajuste lineal\n"
                   f"$S = \\partial\\lambda / \\partial n_{{clad}}$ "
                   f"= {sens:.2f} nm/RIU\n"
                   f"$R^2$ = {r2:.6f}"))
    ax.set_xlabel("Índice de refracción del cladding  $n_{clad}$", fontsize=12)
    ax.set_ylabel("Longitud de onda de resonancia  (nm)", fontsize=12)
    ax.set_title(
        f"Tracking de resonancia — kappa_03\nSensibilidad: {sens:.2f} nm/RIU  ($n_{{clad}}$)",
        fontsize=12)
    ax.legend(framealpha=0.9, fontsize=9)
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR_03 / "nc_resonance_tracking_vs_ncladding.png")
        fig.savefig(FIGURES_DIR_03 / "nc_resonance_tracking_vs_ncladding.pdf")
    log.info(f"[03] Sensibilidad (n_cladding): {sens:.4f} nm/RIU   R²={r2:.8f}")
    return fig


# ── Ejecutar todas las gráficas kappa_03 ──────────────────────────────────────
if wl_03 is not None and computed_03.sum() > 0:
    # Eje neff
    f03_1  = plot_03_drop_power_vs_neff()
    f03_2  = plot_03_sensor_through_sweep(n_curves=200)
    f03_3  = plot_03_final_through_sweep(n_curves=200)
    f03_4  = plot_03_drop_spectrum_heatmap(drop_k=1)
    f03_5  = plot_03_drop_spectrum_heatmap(drop_k=13)
    f03_6  = plot_03_all_drop_heatmaps()
    f03_7  = plot_03_power_heatmap_drops_vs_neff()
    f03_8  = plot_03_resonance_tracking()
    # Eje n_cladding
    f03_9  = plot_03_drop_power_vs_ncladding()
    f03_10 = plot_03_sensor_through_sweep_nc(n_curves=200)
    f03_11 = plot_03_final_through_sweep_nc(n_curves=200)
    f03_12 = plot_03_drop_spectrum_heatmap_nc(drop_k=1)
    f03_13 = plot_03_drop_spectrum_heatmap_nc(drop_k=13)
    f03_14 = plot_03_all_drop_heatmaps_nc()
    f03_15 = plot_03_power_heatmap_drops_vs_ncladding()
    f03_16 = plot_03_resonance_tracking_nc()
    plt.show()
    print(f"\n  [kappa_03] Figuras → {FIGURES_DIR_03}")
    print(f"  [kappa_03] HDF5   → {HDF5_PATH_03}")
else:
    print("  [kappa_03] Sin datos disponibles — ejecuta run_interconnect_sweep_03().")

# ═════════════════════════════════════════════════════════════════════════════
#  END CELL A  (kappa_03)
# ═════════════════════════════════════════════════════════════════════════════

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL B — Sweep V3 con coeficientes de acoplamiento _02                    ║
# ║                                                                            ║
# ║  FWHM objetivo: par kappa_02  (acoplamiento crítico diseño _02)            ║
# ║                                                                            ║
# ║  DEPENDENCIAS HEREDADAS (no se redefinen):                                 ║
# ║    Todas las variables, funciones y constantes de Cells 1–7 y Cell A       ║
# ║    RING_KAPPA_INPUT_SQ_01, RING_KAPPA_DROP_SQ_01  (celda previa)          ║
# ║    n_cladding  (Cell 6)                                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ── Rutas de salida propias para este par de kappas ───────────────────────────
VERSION_NAME_02  = "ICNT_14Ring_Cascade_UniDir_neff_sweep_V3_kappa02"
HDF5_PATH_02     = DATA_DIR / f"{VERSION_NAME_02}.h5"
FIGURES_DIR_02   = DATA_DIR / "figures_kappa02"
FIGURES_DIR_02.mkdir(parents=True, exist_ok=True)

print(f"  Versión kappa_02   : {VERSION_NAME_02}")
print(f"  HDF5               : {HDF5_PATH_02}")
print(f"  Figuras            : {FIGURES_DIR_02}")


# ─────────────────────────────────────────────────────────────────────────────
# Setter de parámetros de anillo para kappa_02
# ─────────────────────────────────────────────────────────────────────────────
def _apply_ring_params_02(ic, ring_idx: int,
                           neff_override=None, ng_override=None) -> None:
    name = ring_name(ring_idx + 1)
    neff = neff_override if neff_override is not None else float(RING_NEFF_TE[ring_idx])
    ng   = ng_override   if ng_override   is not None else float(RING_NG_TE[ring_idx])
    pol  = RING_POLARIZATION[ring_idx].upper()
    d_si = (float(RING_D_TE_PS2_PER_KM[ring_idx] if pol == "TE"
                  else RING_D_TM_PS2_PER_KM[ring_idx]) * 1e-15)
    res_hz        = SPEED_OF_LIGHT / float(RING_LAMBDA_RES_M[ring_idx])
    circumference = RING_RADIUS_M[ring_idx] * 2.0 * math.pi

    _eval(ic, f'setnamed("{name}", "length",                   {circumference:.12e})')
    _eval(ic, f'setnamed("{name}", "frequency",                {res_hz:.12e})')
    _eval(ic, f'setnamed("{name}", "effective index 1",        {neff:.12f})')
    _eval(ic, f'setnamed("{name}", "group index 1",            {ng:.12f})')
    _eval(ic, f'setnamed("{name}", "loss 1",                   {RING_LOSS_DB_PER_M[ring_idx]:.6f})')
    _eval(ic, f'setnamed("{name}", "dispersion 1",             {d_si:.12e})')
    _eval(ic, f'setnamed("{name}", "coupling coefficient 1 1", {RING_KAPPA_INPUT_SQ_01[ring_idx]:.12f})')
    _eval(ic, f'setnamed("{name}", "coupling coefficient 1 2", {RING_KAPPA_DROP_SQ_01[ring_idx]:.12f})')
    _eval(ic, f'setnamed("{name}", "configuration", "unidirectional")')


# ─────────────────────────────────────────────────────────────────────────────
# Constructor de circuito para kappa_02
# ─────────────────────────────────────────────────────────────────────────────
def _build_circuit_02(ic) -> None:
    _eval(ic, "switchtodesign")
    _try_eval(ic, "selectall")
    _try_eval(ic, "delete")

    pwr_W   = 10.0 ** (ONA_POWER_DBM / 10.0) * 1e-3
    f_start = SPEED_OF_LIGHT / ONA_LAMBDA_STOP_M
    f_stop  = SPEED_OF_LIGHT / ONA_LAMBDA_START_M

    _eval(ic, 'addelement("Optical Network Analyzer")')
    _eval(ic, f'set("name", "{ONA_NAME}")')
    _try_eval(ic, 'set("x position", 0)')
    _try_eval(ic, 'set("y position", 0)')
    _eval(ic, f'setnamed("{ONA_NAME}", "input parameter",       "start and stop")')
    _eval(ic, f'setnamed("{ONA_NAME}", "number of input ports", {int(ONA_N_INPUT_PORTS)})')
    _eval(ic, f'setnamed("{ONA_NAME}", "start frequency",       {f_start:.12e})')
    _eval(ic, f'setnamed("{ONA_NAME}", "stop frequency",        {f_stop:.12e})')
    _eval(ic, f'setnamed("{ONA_NAME}", "number of points",      {int(ONA_N_POINTS)})')
    _eval(ic, f'setnamed("{ONA_NAME}", "power",                 {pwr_W:.12e})')

    for i in range(N_RINGS):
        rn = ring_name(i + 1)
        _eval(ic, 'addelement("Double Bus Ring Resonator")')
        _eval(ic, f'set("name", "{rn}")')
        _try_eval(ic, f'set("x position", {float((i + 1) * 220)})')
        _try_eval(ic, f'set("y position", 0)')
        _apply_ring_params_02(ic, ring_idx=i)
        log.info(
            f"  [02] RING_{i+1:2d} — "
            f"κ²_in={RING_KAPPA_INPUT_SQ_01[i]:.6f}  "
            f"κ²_dr={RING_KAPPA_DROP_SQ_01[i]:.6f}"
        )

    def wire(elem_a, port_a, elem_b, port_b):
        _eval(ic, f'connect("{elem_a}", "{port_a}", "{elem_b}", "{port_b}")')

    wire(ONA_NAME,        "output",   ring_name(1), "input")
    wire(ring_name(1),    "output 1", ONA_NAME,     f"input {THROUGH_SENSOR_INPUT}")
    wire(ring_name(1),    "output 2", ring_name(2), "input")
    for i in range(2, N_RINGS):
        wire(ring_name(i), "output 1", ring_name(i + 1), "input")
        wire(ring_name(i), "output 2", ONA_NAME, f"input {drop_ona_input(i)}")
    wire(ring_name(N_RINGS), "output 2", ONA_NAME, f"input {drop_ona_input(N_RINGS)}")
    wire(ring_name(N_RINGS), "output 1", ONA_NAME, f"input {THROUGH_FINAL_INPUT}")

    log.info("V3-kappa02 circuit built.")


# ─────────────────────────────────────────────────────────────────────────────
# HDF5 para kappa_02
# ─────────────────────────────────────────────────────────────────────────────
def _init_hdf5_02(wl_ref_m: np.ndarray) -> None:
    n_pts = SWEEP_N_POINTS
    n_wl  = len(wl_ref_m)
    with h5py.File(HDF5_PATH_02, "w") as f:
        md = f.create_group("metadata")
        md.create_dataset("neff_sweep",    data=SWEEP_NEFF)
        md.create_dataset("ng_sweep",      data=SWEEP_NG)
        md.create_dataset("wavelengths_m", data=wl_ref_m)
        md.attrs["version_name"]        = VERSION_NAME_02
        md.attrs["kappa_variant"]       = "kappa_02"
        md.attrs["n_rings"]             = N_RINGS
        md.attrs["n_drops"]             = N_DROPS
        md.attrs["sweep_n_points"]      = SWEEP_N_POINTS
        md.attrs["ring_model"]          = "Double Bus Ring Resonator"
        md.attrs["ring_configuration"]  = "unidirectional"
        md.attrs["topology"]            = (
            "V3-kappa02: ONA multiport, no OPMs. "
            "ONA input 1=RING_1 through, input k=RING_k drop (k=2-14), "
            "input 15=RING_14 through"
        )
        md.attrs["ona_lambda_start_m"]  = ONA_LAMBDA_START_M
        md.attrs["ona_lambda_stop_m"]   = ONA_LAMBDA_STOP_M
        md.attrs["ona_n_points"]        = ONA_N_POINTS
        md.attrs["ona_power_dBm"]       = ONA_POWER_DBM
        md.attrs["ona_n_input_ports"]   = ONA_N_INPUT_PORTS
        md.attrs["power_calc_method"]   = (
            "P_det[W] = P_source[W] * mean(|T_drop(lambda)|^2) over ONA band"
        )
        md.attrs["timestamp_start"]     = datetime.now().isoformat()
        for i in range(N_RINGS):
            p = f"ring{i+1}_"
            md.attrs[p + "kappa_input_sq"] = RING_KAPPA_INPUT_SQ_01[i]
            md.attrs[p + "kappa_drop_sq"]  = RING_KAPPA_DROP_SQ_01[i]
            md.attrs[p + "radius_m"]       = RING_RADIUS_M[i]
            md.attrs[p + "lambda_res_m"]   = RING_LAMBDA_RES_M[i]
            md.attrs[p + "neff_TE"]        = RING_NEFF_TE[i]
            md.attrs[p + "ng_TE"]          = RING_NG_TE[i]
            md.attrs[p + "loss_dB_per_m"]  = RING_LOSS_DB_PER_M[i]

        rg = f.create_group("results")
        rg.create_dataset("T_sensor_through_dB",
                          data=np.full((n_pts, n_wl), np.nan), chunks=(1, n_wl))
        rg.create_dataset("T_final_through_dB",
                          data=np.full((n_pts, n_wl), np.nan), chunks=(1, n_wl))
        rg.create_dataset("T_drop_dB",
                          data=np.full((n_pts, N_DROPS, n_wl), np.nan),
                          chunks=(1, 1, n_wl))
        rg.create_dataset("drop_power_dBm",
                          data=np.full((n_pts, N_DROPS), np.nan),
                          chunks=(1, N_DROPS))
        f.create_group("flags").create_dataset(
            "computed", data=np.zeros(n_pts, dtype=bool), chunks=(1,))

    log.info(f"HDF5 kappa02 inicializado → {HDF5_PATH_02}")


# ─────────────────────────────────────────────────────────────────────────────
# Motor del sweep para kappa_02
# ─────────────────────────────────────────────────────────────────────────────
def run_interconnect_sweep_02(hide_gui: bool = False):
    n_pts = SWEEP_N_POINTS
    wavelengths_m = T_sensor_thr = T_final_thr = T_drop = p_drop = None
    computed_02   = np.zeros(n_pts, dtype=bool)
    hdf5_ready    = False

    if HDF5_PATH_02.exists():
        log.info(f"[02] Caché encontrado → {HDF5_PATH_02}")
        try:
            with h5py.File(HDF5_PATH_02, "r") as f:
                wavelengths_m = f["metadata/wavelengths_m"][:]
                T_sensor_thr  = f["results/T_sensor_through_dB"][:]
                T_final_thr   = f["results/T_final_through_dB"][:]
                T_drop        = f["results/T_drop_dB"][:]
                p_drop        = f["results/drop_power_dBm"][:]
                computed_02[:]= f["flags/computed"][:]
            hdf5_ready = True
            n_cached   = int(computed_02.sum())
            log.info(f"[02] Cached: {n_cached}/{n_pts}  |  Pending: {n_pts - n_cached}")
            if n_pts - n_cached == 0:
                log.info("[02] Todos los puntos en caché — sin lanzar INTERCONNECT.")
                return _pack_results_02(
                    wavelengths_m, T_sensor_thr, T_final_thr, T_drop, p_drop, computed_02)
        except Exception as exc:
            log.warning(f"[02] Caché ilegible ({exc}). Empezando desde cero.")
            wavelengths_m = T_sensor_thr = T_final_thr = T_drop = p_drop = None
            computed_02[:] = False
            hdf5_ready = False
    else:
        log.info("[02] Sin caché — sweep desde cero.")

    log.info("[02] Lanzando INTERCONNECT …")
    ic         = lumapi.INTERCONNECT(hide=hide_gui)
    runs_done  = 0
    runs_total = int((~computed_02).sum())
    t_start    = time.time()

    try:
        _build_circuit_02(ic)
        log.info(f"[02] Circuito listo — {runs_total} puntos por computar …")

        for s_idx in range(n_pts):
            if computed_02[s_idx]:
                continue

            neff_val = float(SWEEP_NEFF[s_idx])
            ng_val   = float(SWEEP_NG[s_idx])

            _eval(ic, "switchtodesign")
            _update_ring1_neff_ng(ic, neff_val, ng_val)

            try:
                _eval(ic, "run")
            except ICScriptError as exc:
                log.warning(f"  [02] RUN FAILED pt={s_idx:3d} neff={neff_val:.6f} → {exc}")
                computed_02[s_idx] = True
                if hdf5_ready:
                    with h5py.File(HDF5_PATH_02, "r+") as hf:
                        hf["flags/computed"][s_idx] = True; hf.flush()
                continue

            try:
                wl_m, t_sen, t_fin, t_drp, p_drp = _extract_results(ic)
            except Exception as exc:
                log.warning(f"  [02] EXTRACT FAILED pt={s_idx:3d}: {exc}")
                computed_02[s_idx] = True
                continue

            if wavelengths_m is None:
                n_wl          = len(wl_m)
                wavelengths_m = wl_m
                T_sensor_thr  = np.full((n_pts, n_wl),          np.nan)
                T_final_thr   = np.full((n_pts, n_wl),          np.nan)
                T_drop        = np.full((n_pts, N_DROPS, n_wl), np.nan)
                p_drop        = np.full((n_pts, N_DROPS),        np.nan)
                if not hdf5_ready:
                    _init_hdf5_02(wl_m)
                    hdf5_ready = True

            T_sensor_thr[s_idx, :]    = t_sen
            T_final_thr [s_idx, :]    = t_fin
            T_drop      [s_idx, :, :] = t_drp
            p_drop      [s_idx, :]    = p_drp
            computed_02 [s_idx]        = True

            with h5py.File(HDF5_PATH_02, "r+") as hf:
                hf["results/T_sensor_through_dB"][s_idx, :]    = t_sen
                hf["results/T_final_through_dB"] [s_idx, :]    = t_fin
                hf["results/T_drop_dB"]          [s_idx, :, :] = t_drp
                hf["results/drop_power_dBm"]     [s_idx, :]    = p_drp
                hf["flags/computed"]             [s_idx]        = True
                hf.flush()

            runs_done += 1
            if runs_done % 5 == 0 or runs_done == runs_total:
                elapsed = time.time() - t_start
                rate    = runs_done / elapsed if elapsed > 0 else 1e-9
                eta     = (runs_total - runs_done) / rate
                log.info(
                    f"  [02][{runs_done:3d}/{runs_total}]  "
                    f"neff={neff_val:.6f}  ng={ng_val:.6f}  │  "
                    f"{rate:.2f} sim/s  │  ETA {eta:5.0f} s"
                )

        if hdf5_ready:
            with h5py.File(HDF5_PATH_02, "r+") as hf:
                hf["metadata"].attrs["timestamp_end"]  = datetime.now().isoformat()
                hf["metadata"].attrs["runs_completed"] = int(computed_02.sum())

    finally:
        try: ic.close()
        except Exception: pass
        log.info("[02] INTERCONNECT session closed.")

    elapsed = time.time() - t_start
    log.info(
        f"[02] Sweep kappa02 listo │ {runs_done} runs │ "
        f"total={elapsed:.1f} s │ avg={elapsed/max(runs_done,1):.2f} s/sim"
    )
    return _pack_results_02(
        wavelengths_m, T_sensor_thr, T_final_thr, T_drop, p_drop, computed_02)


def _pack_results_02(wl, t_sen, t_fin, t_drop, p_drop, comp):
    return dict(
        neff_sweep            = SWEEP_NEFF,
        ng_sweep              = SWEEP_NG,
        wavelengths_m         = wl,
        T_sensor_through_dB   = t_sen,
        T_final_through_dB    = t_fin,
        T_drop_dB             = t_drop,
        drop_power_dBm        = p_drop,
        computed              = comp,
    )


# ── Ejecutar sweep kappa_02 ────────────────────────────────────────────────────
sweep_results_02 = run_interconnect_sweep_02(hide_gui=False)

wl_02          = sweep_results_02["wavelengths_m"]
wl_nm_02       = wl_02 * 1e9 if wl_02 is not None else None
computed_02    = sweep_results_02["computed"]

print(f"\n  [kappa_02] Sweep completo — {computed_02.sum()}/{SWEEP_N_POINTS} pts")
if wl_02 is not None:
    print(f"  T_sensor_through_dB shape : {sweep_results_02['T_sensor_through_dB'].shape}")
    print(f"  T_drop_dB           shape : {sweep_results_02['T_drop_dB'].shape}")
    print(f"  drop_power_dBm      shape : {sweep_results_02['drop_power_dBm'].shape}")
print(f"  HDF5                      : {HDF5_PATH_02}")


# ─────────────────────────────────────────────────────────────────────────────
# Protocolo de gráficas completo — kappa_02
# ─────────────────────────────────────────────────────────────────────────────
def _valid_mask_02():
    return sweep_results_02["computed"].astype(bool)


def plot_02_drop_power_vs_neff(figsize=(11, 6), save=True):
    mask   = _valid_mask_02()
    neff_v = sweep_results_02["neff_sweep"][mask]
    p_v    = sweep_results_02["drop_power_dBm"][mask, :]
    cmap   = plt.get_cmap("tab20")
    fig, ax = plt.subplots(figsize=figsize)
    for k in range(N_DROPS):
        ax.plot(neff_v, p_v[:, k], color=cmap(k / N_DROPS),
                lw=1.5, marker="o", ms=2.5, alpha=0.85, label=DROP_LABELS[k])
    ax.set_xlabel("Ring 1 — $n_{eff}$ (TE)  [sensor]", fontsize=12)
    ax.set_ylabel("Detector power  (dBm)", fontsize=12)
    ax.set_title(
        "Integrated Drop Power vs Sensor $n_{eff}$  [V3 — kappa_02]\n"
        r"$P_{det} = P_{src} \cdot \langle|T_{drop}(\lambda)|^2\rangle_\lambda$"
        "   —   13 spectrometer rings", fontsize=12)
    ax.legend(ncol=3, framealpha=0.88, fontsize=8, loc="best")
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR_02 / "drop_power_vs_neff_all.png")
        fig.savefig(FIGURES_DIR_02 / "drop_power_vs_neff_all.pdf")
    return fig


def plot_02_sensor_through_sweep(n_curves=200, figsize=(10, 5),
                                  cmap_name="plasma", save=True):
    neff_arr  = sweep_results_02["neff_sweep"]
    wl_nm     = sweep_results_02["wavelengths_m"] * 1e9
    T_data    = sweep_results_02["T_sensor_through_dB"]
    mask      = _valid_mask_02()
    valid_idx = np.where(mask)[0]
    n_sel     = min(n_curves, len(valid_idx))
    sel_idx   = valid_idx[np.round(np.linspace(0, len(valid_idx)-1, n_sel)).astype(int)]
    cmap = plt.get_cmap(cmap_name)
    norm = Normalize(vmin=neff_arr[sel_idx].min(), vmax=neff_arr[sel_idx].max())
    sm   = ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    fig, ax = plt.subplots(figsize=figsize)
    for idx in sel_idx:
        ax.plot(wl_nm, T_data[idx], color=cmap(norm(neff_arr[idx])), lw=0.8, alpha=0.70)
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label("Ring 1 — $n_{eff}$ (TE)", fontsize=10)
    ax.set_xlabel("Wavelength  (nm)"); ax.set_ylabel("Transmission  (dB)")
    ax.set_title(f"Sensor Through Spectrum — kappa_02\n({n_sel} curves, colour = $n_{{eff,1}}$)")
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR_02 / f"sensor_through_sweep_{n_sel}curves.png")
        fig.savefig(FIGURES_DIR_02 / f"sensor_through_sweep_{n_sel}curves.pdf")
    return fig


def plot_02_final_through_sweep(n_curves=200, figsize=(10, 5),
                                 cmap_name="viridis", save=True):
    neff_arr  = sweep_results_02["neff_sweep"]
    wl_nm     = sweep_results_02["wavelengths_m"] * 1e9
    T_data    = sweep_results_02["T_final_through_dB"]
    mask      = _valid_mask_02()
    valid_idx = np.where(mask)[0]
    n_sel     = min(n_curves, len(valid_idx))
    sel_idx   = valid_idx[np.round(np.linspace(0, len(valid_idx)-1, n_sel)).astype(int)]
    cmap = plt.get_cmap(cmap_name)
    norm = Normalize(vmin=neff_arr[sel_idx].min(), vmax=neff_arr[sel_idx].max())
    sm   = ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    fig, ax = plt.subplots(figsize=figsize)
    for idx in sel_idx:
        ax.plot(wl_nm, T_data[idx], color=cmap(norm(neff_arr[idx])), lw=0.8, alpha=0.70)
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label("Ring 1 — $n_{eff}$ (TE)", fontsize=10)
    ax.set_xlabel("Wavelength  (nm)"); ax.set_ylabel("Transmission  (dB)")
    ax.set_title(f"Cascade Final Through — kappa_02\n({n_sel} curves, colour = $n_{{eff,1}}$)")
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR_02 / f"final_through_sweep_{n_sel}curves.png")
        fig.savefig(FIGURES_DIR_02 / f"final_through_sweep_{n_sel}curves.pdf")
    return fig


def plot_02_drop_spectrum_heatmap(drop_k=1, figsize=(10, 3.5),
                                   cmap_name="inferno",
                                   vmin_dB=None, vmax_dB=None, save=True):
    assert 1 <= drop_k <= N_DROPS
    ring_label = DROP_LABELS[drop_k - 1]
    neff_arr   = sweep_results_02["neff_sweep"]
    wl_nm      = sweep_results_02["wavelengths_m"] * 1e9
    t_drop     = sweep_results_02["T_drop_dB"]
    mask       = _valid_mask_02()
    neff_v     = neff_arr[mask]
    spec_v     = t_drop[mask, drop_k - 1, :]
    _vmin = vmin_dB if vmin_dB is not None else np.nanpercentile(spec_v, 2)
    _vmax = vmax_dB if vmax_dB is not None else np.nanpercentile(spec_v, 98)
    fig, ax = plt.subplots(figsize=figsize)
    img = ax.pcolormesh(wl_nm, neff_v, spec_v,
                        cmap=cmap_name, vmin=_vmin, vmax=_vmax, shading="auto")
    cbar = fig.colorbar(img, ax=ax, pad=0.01)
    cbar.set_label("Transmission  (dB)", fontsize=10)
    ax.set_xlabel("Wavelength  (nm)"); ax.set_ylabel("Ring 1 — $n_{eff}$ (TE)")
    ax.set_title(
        f"Drop Spectrum Heatmap — {ring_label}  [kappa_02]\n"
        "neff sweep on sensor ring (RING_1)")
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR_02 / f"drop{drop_k}_spectrum_heatmap.png")
        fig.savefig(FIGURES_DIR_02 / f"drop{drop_k}_spectrum_heatmap.pdf")
    return fig


def plot_02_all_drop_heatmaps(ncols=4, cmap_name="inferno", save=True):
    neff_arr = sweep_results_02["neff_sweep"]
    wl_nm    = sweep_results_02["wavelengths_m"] * 1e9
    t_drop   = sweep_results_02["T_drop_dB"]
    mask     = _valid_mask_02()
    neff_v   = neff_arr[mask]
    nrows    = math.ceil(N_DROPS / ncols)
    fig, axes = plt.subplots(nrows, ncols,
                              figsize=(ncols * 3.8, nrows * 2.8),
                              sharex=True, sharey=True)
    axes_flat = axes.flatten()
    all_vals  = t_drop[mask].ravel()
    vmin = np.nanpercentile(all_vals, 2); vmax = np.nanpercentile(all_vals, 98)
    im = None
    for k in range(1, N_DROPS + 1):
        ax   = axes_flat[k - 1]
        spec = t_drop[mask, k - 1, :]
        im   = ax.pcolormesh(wl_nm, neff_v, spec,
                             cmap=cmap_name, vmin=vmin, vmax=vmax, shading="auto")
        ax.set_title(f"{DROP_LABELS[k-1]} drop", fontsize=8)
        if k % ncols == 1: ax.set_ylabel("$n_{eff,1}$", fontsize=7)
        if k > (nrows - 1) * ncols: ax.set_xlabel("λ (nm)", fontsize=7)
        ax.tick_params(labelsize=6)
    for ax in axes_flat[N_DROPS:]: ax.set_visible(False)
    if im is not None:
        fig.colorbar(im, ax=axes_flat[:N_DROPS], shrink=0.6, pad=0.02,
                     label="Transmission (dB)", fraction=0.015)
    fig.suptitle(
        "All Drop Spectra  [V3 — kappa_02]\nneff sweep on sensor ring (RING_1)",
        fontsize=11)
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR_02 / "all_drop_heatmaps_grid.png", dpi=200)
        fig.savefig(FIGURES_DIR_02 / "all_drop_heatmaps_grid.pdf")
    return fig


def plot_02_power_heatmap_drops_vs_neff(figsize=(10, 5),
                                         cmap_name="plasma", save=True):
    neff_arr = sweep_results_02["neff_sweep"]
    p_drop   = sweep_results_02["drop_power_dBm"]
    mask     = _valid_mask_02()
    neff_v   = neff_arr[mask]
    p_v      = p_drop[mask, :].T
    fig, ax  = plt.subplots(figsize=figsize)
    img = ax.pcolormesh(neff_v, np.arange(1, N_DROPS + 1), p_v,
                        cmap=cmap_name, shading="auto")
    cbar = fig.colorbar(img, ax=ax, pad=0.02)
    cbar.set_label("Detector power  (dBm)", fontsize=10)
    ax.set_xlabel("Ring 1 — $n_{eff}$ (TE)  [sensor]", fontsize=11)
    ax.set_ylabel("Spectrometer ring index  (1=RING_2 … 13=RING_14)", fontsize=10)
    ax.set_yticks(np.arange(1, N_DROPS + 1))
    ax.set_yticklabels(DROP_LABELS, fontsize=7)
    ax.set_title(
        "Detector Power Heatmap — All Drops vs $n_{eff}$  [kappa_02]\n"
        r"Colour: $P_{det}$ [dBm]  per spectrometer ring", fontsize=12)
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR_02 / "power_heatmap_drops_vs_neff.png", dpi=200)
        fig.savefig(FIGURES_DIR_02 / "power_heatmap_drops_vs_neff.pdf")
    return fig


def plot_02_resonance_tracking(figsize=(7, 4.5), save=True):
    neff_arr = sweep_results_02["neff_sweep"]
    wl_nm    = sweep_results_02["wavelengths_m"] * 1e9
    T_data   = sweep_results_02["T_sensor_through_dB"]
    mask     = _valid_mask_02()
    neff_v   = neff_arr[mask]
    T_v      = T_data[mask, :]
    dip_idx  = np.argmin(T_v, axis=1)
    lam_dip  = wl_nm[dip_idx]
    coeffs   = np.polyfit(neff_v, lam_dip, 1)
    sens     = coeffs[0]
    fig, ax  = plt.subplots(figsize=figsize)
    ax.scatter(neff_v, lam_dip, s=20, zorder=5, color="#2166ac", label="Resonance dip")
    ax.plot(neff_v, np.poly1d(coeffs)(neff_v), "r--", lw=1.5,
            label=f"Linear fit   ∂λ/∂n = {sens:.3f} nm / RIU")
    ax.set_xlabel("Ring 1 — $n_{eff}$ (TE)")
    ax.set_ylabel("Resonance wavelength  (nm)")
    ax.set_title(
        f"Resonance Tracking — kappa_02\nSensitivity: {sens:.3f} nm / RIU")
    ax.legend(framealpha=0.9)
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR_02 / "resonance_tracking_sensor.png")
        fig.savefig(FIGURES_DIR_02 / "resonance_tracking_sensor.pdf")
    log.info(f"[02] Resonance sensitivity: {sens:.4f} nm/RIU")
    return fig


# ── Plots NC (n_cladding en eje X) — kappa_02 ─────────────────────────────────

def plot_02_drop_power_vs_ncladding(figsize=(11, 6), save=True):
    mask = _valid_mask_02()
    nc_v = n_cladding[mask]
    p_v  = sweep_results_02["drop_power_dBm"][mask, :]
    cmap = plt.get_cmap("tab20")
    fig, ax = plt.subplots(figsize=figsize)
    for k in range(N_DROPS):
        ax.plot(nc_v, p_v[:, k], color=cmap(k / N_DROPS),
                lw=1.5, marker="o", ms=2.5, alpha=0.85, label=DROP_LABELS[k])
    ax.set_xlabel("Índice de refracción del cladding  $n_{clad}$", fontsize=12)
    ax.set_ylabel("Potencia en el detector  (dBm)", fontsize=12)
    ax.set_title(
        "Potencia integrada en drop vs $n_{clad}$  [kappa_02]\n"
        r"$P_{det} = P_{src} \cdot \langle|T_{drop}(\lambda)|^2\rangle_\lambda$"
        "   —   13 anillos espectrómetros", fontsize=12)
    ax.legend(ncol=3, framealpha=0.88, fontsize=8, loc="best")
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR_02 / "nc_drop_power_vs_ncladding_all.png")
        fig.savefig(FIGURES_DIR_02 / "nc_drop_power_vs_ncladding_all.pdf")
    return fig


def plot_02_sensor_through_sweep_nc(n_curves=200, figsize=(10, 5),
                                     cmap_name="plasma", save=True):
    wl_nm     = sweep_results_02["wavelengths_m"] * 1e9
    T_data    = sweep_results_02["T_sensor_through_dB"]
    mask      = _valid_mask_02()
    valid_idx = np.where(mask)[0]
    n_sel     = min(n_curves, len(valid_idx))
    sel_idx   = valid_idx[np.round(np.linspace(0, len(valid_idx)-1, n_sel)).astype(int)]
    nc_sel    = n_cladding[sel_idx]
    cmap = plt.get_cmap(cmap_name)
    norm = Normalize(vmin=nc_sel.min(), vmax=nc_sel.max())
    sm   = ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    fig, ax = plt.subplots(figsize=figsize)
    for idx in sel_idx:
        ax.plot(wl_nm, T_data[idx], color=cmap(norm(n_cladding[idx])), lw=0.8, alpha=0.70)
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label("$n_{clad}$", fontsize=10)
    ax.set_xlabel("Longitud de onda  (nm)"); ax.set_ylabel("Transmisión  (dB)")
    ax.set_title(f"Espectro sensor through — kappa_02\n({n_sel} curvas, color = $n_{{clad}}$)")
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR_02 / f"nc_sensor_through_sweep_{n_sel}curves.png")
        fig.savefig(FIGURES_DIR_02 / f"nc_sensor_through_sweep_{n_sel}curves.pdf")
    return fig


def plot_02_final_through_sweep_nc(n_curves=200, figsize=(10, 5),
                                    cmap_name="viridis", save=True):
    wl_nm     = sweep_results_02["wavelengths_m"] * 1e9
    T_data    = sweep_results_02["T_final_through_dB"]
    mask      = _valid_mask_02()
    valid_idx = np.where(mask)[0]
    n_sel     = min(n_curves, len(valid_idx))
    sel_idx   = valid_idx[np.round(np.linspace(0, len(valid_idx)-1, n_sel)).astype(int)]
    nc_sel    = n_cladding[sel_idx]
    cmap = plt.get_cmap(cmap_name)
    norm = Normalize(vmin=nc_sel.min(), vmax=nc_sel.max())
    sm   = ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    fig, ax = plt.subplots(figsize=figsize)
    for idx in sel_idx:
        ax.plot(wl_nm, T_data[idx], color=cmap(norm(n_cladding[idx])), lw=0.8, alpha=0.70)
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label("$n_{clad}$", fontsize=10)
    ax.set_xlabel("Longitud de onda  (nm)"); ax.set_ylabel("Transmisión  (dB)")
    ax.set_title(f"Through final de cascada — kappa_02\n({n_sel} curvas, color = $n_{{clad}}$)")
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR_02 / f"nc_final_through_sweep_{n_sel}curves.png")
        fig.savefig(FIGURES_DIR_02 / f"nc_final_through_sweep_{n_sel}curves.pdf")
    return fig


def plot_02_drop_spectrum_heatmap_nc(drop_k=1, figsize=(10, 3.5),
                                      cmap_name="inferno",
                                      vmin_dB=None, vmax_dB=None, save=True):
    assert 1 <= drop_k <= N_DROPS
    ring_label = DROP_LABELS[drop_k - 1]
    wl_nm  = sweep_results_02["wavelengths_m"] * 1e9
    t_drop = sweep_results_02["T_drop_dB"]
    mask   = _valid_mask_02()
    nc_v   = n_cladding[mask]
    spec_v = t_drop[mask, drop_k - 1, :]
    _vmin  = vmin_dB if vmin_dB is not None else np.nanpercentile(spec_v, 2)
    _vmax  = vmax_dB if vmax_dB is not None else np.nanpercentile(spec_v, 98)
    fig, ax = plt.subplots(figsize=figsize)
    img = ax.pcolormesh(wl_nm, nc_v, spec_v,
                        cmap=cmap_name, vmin=_vmin, vmax=_vmax, shading="auto")
    cbar = fig.colorbar(img, ax=ax, pad=0.01)
    cbar.set_label("Transmisión  (dB)", fontsize=10)
    ax.set_xlabel("Longitud de onda  (nm)"); ax.set_ylabel("$n_{clad}$")
    ax.set_title(
        f"Heatmap espectral drop — {ring_label}  [kappa_02]\n"
        f"Barrido de $n_{{clad}}$ sobre RING_1")
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR_02 / f"nc_drop{drop_k}_spectrum_heatmap.png")
        fig.savefig(FIGURES_DIR_02 / f"nc_drop{drop_k}_spectrum_heatmap.pdf")
    return fig


def plot_02_all_drop_heatmaps_nc(ncols=4, cmap_name="inferno", save=True):
    wl_nm  = sweep_results_02["wavelengths_m"] * 1e9
    t_drop = sweep_results_02["T_drop_dB"]
    mask   = _valid_mask_02()
    nc_v   = n_cladding[mask]
    nrows  = math.ceil(N_DROPS / ncols)
    fig, axes = plt.subplots(nrows, ncols,
                              figsize=(ncols * 3.8, nrows * 2.8),
                              sharex=True, sharey=True)
    axes_flat = axes.flatten()
    all_vals  = t_drop[mask].ravel()
    vmin = np.nanpercentile(all_vals, 2); vmax = np.nanpercentile(all_vals, 98)
    im = None
    for k in range(1, N_DROPS + 1):
        ax   = axes_flat[k - 1]
        spec = t_drop[mask, k - 1, :]
        im   = ax.pcolormesh(wl_nm, nc_v, spec,
                             cmap=cmap_name, vmin=vmin, vmax=vmax, shading="auto")
        ax.set_title(f"{DROP_LABELS[k-1]} drop", fontsize=8)
        if k % ncols == 1: ax.set_ylabel("$n_{clad}$", fontsize=7)
        if k > (nrows - 1) * ncols: ax.set_xlabel("λ (nm)", fontsize=7)
        ax.tick_params(labelsize=6)
    for ax in axes_flat[N_DROPS:]: ax.set_visible(False)
    if im is not None:
        fig.colorbar(im, ax=axes_flat[:N_DROPS], shrink=0.6, pad=0.02,
                     label="Transmisión (dB)", fraction=0.015)
    fig.suptitle(
        "Espectros drop — 13 anillos  [kappa_02]\nBarrido de $n_{clad}$ sobre RING_1",
        fontsize=11)
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR_02 / "nc_all_drop_heatmaps_grid.png", dpi=200)
        fig.savefig(FIGURES_DIR_02 / "nc_all_drop_heatmaps_grid.pdf")
    return fig


def plot_02_power_heatmap_drops_vs_ncladding(figsize=(10, 5),
                                              cmap_name="plasma", save=True):
    mask = _valid_mask_02()
    nc_v = n_cladding[mask]
    p_v  = sweep_results_02["drop_power_dBm"][mask, :].T
    fig, ax = plt.subplots(figsize=figsize)
    img = ax.pcolormesh(nc_v, np.arange(1, N_DROPS + 1), p_v,
                        cmap=cmap_name, shading="auto")
    cbar = fig.colorbar(img, ax=ax, pad=0.02)
    cbar.set_label("Potencia en detector  (dBm)", fontsize=10)
    ax.set_xlabel("Índice de refracción del cladding  $n_{clad}$", fontsize=11)
    ax.set_ylabel("Índice de anillo espectrómetro  (1=RING_2 … 13=RING_14)", fontsize=10)
    ax.set_yticks(np.arange(1, N_DROPS + 1)); ax.set_yticklabels(DROP_LABELS, fontsize=7)
    ax.set_title(
        "Heatmap de potencia — Todos los drops vs $n_{clad}$  [kappa_02]\n"
        r"Color: $P_{det}$ [dBm]  por anillo espectrómetro", fontsize=12)
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR_02 / "nc_power_heatmap_drops_vs_ncladding.png", dpi=200)
        fig.savefig(FIGURES_DIR_02 / "nc_power_heatmap_drops_vs_ncladding.pdf")
    return fig


def plot_02_resonance_tracking_nc(figsize=(7, 4.5), save=True):
    wl_nm  = sweep_results_02["wavelengths_m"] * 1e9
    T_data = sweep_results_02["T_sensor_through_dB"]
    mask   = _valid_mask_02()
    nc_v   = n_cladding[mask]
    T_v    = T_data[mask, :]
    dip_idx = np.argmin(T_v, axis=1)
    lam_dip = wl_nm[dip_idx]
    coeffs  = np.polyfit(nc_v, lam_dip, 1)
    sens    = coeffs[0]
    r2      = 1.0 - np.sum((lam_dip - np.poly1d(coeffs)(nc_v))**2) / \
                    np.sum((lam_dip - lam_dip.mean())**2)
    fig, ax = plt.subplots(figsize=figsize)
    ax.scatter(nc_v, lam_dip, s=20, zorder=5,
               color="#2166ac", label="Mínimo de transmisión (dip)")
    ax.plot(nc_v, np.poly1d(coeffs)(nc_v), "r--", lw=1.5,
            label=(f"Ajuste lineal\n"
                   f"$S = \\partial\\lambda / \\partial n_{{clad}}$ "
                   f"= {sens:.2f} nm/RIU\n"
                   f"$R^2$ = {r2:.6f}"))
    ax.set_xlabel("Índice de refracción del cladding  $n_{clad}$", fontsize=12)
    ax.set_ylabel("Longitud de onda de resonancia  (nm)", fontsize=12)
    ax.set_title(
        f"Tracking de resonancia — kappa_02\nSensibilidad: {sens:.2f} nm/RIU  ($n_{{clad}}$)",
        fontsize=12)
    ax.legend(framealpha=0.9, fontsize=9)
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR_02 / "nc_resonance_tracking_vs_ncladding.png")
        fig.savefig(FIGURES_DIR_02 / "nc_resonance_tracking_vs_ncladding.pdf")
    log.info(f"[02] Sensibilidad (n_cladding): {sens:.4f} nm/RIU   R²={r2:.8f}")
    return fig


# ── Ejecutar todas las gráficas kappa_02 ──────────────────────────────────────
if wl_02 is not None and computed_02.sum() > 0:
    # Eje neff
    f02_1  = plot_02_drop_power_vs_neff()
    f02_2  = plot_02_sensor_through_sweep(n_curves=200)
    f02_3  = plot_02_final_through_sweep(n_curves=200)
    f02_4  = plot_02_drop_spectrum_heatmap(drop_k=1)
    f02_5  = plot_02_drop_spectrum_heatmap(drop_k=13)
    f02_6  = plot_02_all_drop_heatmaps()
    f02_7  = plot_02_power_heatmap_drops_vs_neff()
    f02_8  = plot_02_resonance_tracking()
    # Eje n_cladding
    f02_9  = plot_02_drop_power_vs_ncladding()
    f02_10 = plot_02_sensor_through_sweep_nc(n_curves=200)
    f02_11 = plot_02_final_through_sweep_nc(n_curves=200)
    f02_12 = plot_02_drop_spectrum_heatmap_nc(drop_k=1)
    f02_13 = plot_02_drop_spectrum_heatmap_nc(drop_k=13)
    f02_14 = plot_02_all_drop_heatmaps_nc()
    f02_15 = plot_02_power_heatmap_drops_vs_ncladding()
    f02_16 = plot_02_resonance_tracking_nc()
    plt.show()
    print(f"\n  [kappa_02] Figuras → {FIGURES_DIR_02}")
    print(f"  [kappa_02] HDF5   → {HDF5_PATH_02}")
else:
    print("  [kappa_02] Sin datos disponibles — ejecuta run_interconnect_sweep_02().")

# ═════════════════════════════════════════════════════════════════════════════
#  END CELL B  (kappa_02)
# ═════════════════════════════════════════════════════════════════════════════

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL C — Comparación P_drop vs n_cladding: FWHM × anillo                 ║
# ║                                                                            ║
# ║  RESULTADO: UNA SOLA FIGURA con las 9 curvas (3 anillos × 3 FWHM)        ║
# ║                                                                            ║
# ║  CONVENCIÓN DE VISUALIZACIÓN                                               ║
# ║  COLOR  → identifica el anillo espectrómetro                               ║
# ║    RING_2  (drop_k=1)  → azul   "#2166ac"                                 ║
# ║    RING_8  (drop_k=7)  → verde  "#1a9641"                                 ║
# ║    RING_14 (drop_k=13) → rojo   "#d6604d"                                 ║
# ║                                                                            ║
# ║  ESTILO DE LÍNEA → identifica la variante κ / FWHM                        ║
# ║    κ original (FWHM mayor)     → línea sólida gruesa   "-"   lw=2.2       ║
# ║    κ₀₃       (FWHM intermedio) → línea a trazos        "--"  lw=1.8       ║
# ║    κ₀₂       (FWHM menor)      → línea punto-raya      "-."  lw=1.6       ║
# ║                                                                            ║
# ║  DEPENDENCIAS HEREDADAS (no se redefinen):                                  ║
# ║    sweep_results, sweep_results_03, sweep_results_02                       ║
# ║    n_cladding  (Cell 6)                                                    ║
# ║    N_DROPS, DROP_LABELS, DATA_DIR, FIGURES_DIR                             ║
# ║    plt, np, ticker, log                                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import matplotlib.lines as mlines

# ─────────────────────────────────────────────────────────────────────────────
# Parámetros de la comparación
# ─────────────────────────────────────────────────────────────────────────────

# Anillos a mostrar (drop_k 1-based: 1=RING_2, 7=RING_8, 13=RING_14)
COMPARE_DROP_INDICES = [1, 7]

# Color fijo por anillo — consistente en toda la figura
RING_COLORS = {
    1:  "#2166ac",   # RING_2  — azul
    7:  "#1a9641",   # RING_8  — verde
    13: "#d6604d",   # RING_14 — rojo
}

# Estilo de línea fijo por variante κ (FWHM) — consistente en todos los anillos
KAPPA_STYLES = [
    {
        "label"   : r"$\kappa_{500}$ (FWHM 500 pm)",
        "results" : sweep_results,
        "ls"      : "-",
        "lw"      : 2.2,
        "marker"  : "o",
        "ms"      : 7.0,
        "markevery_frac": 25,
        "alpha"   : 0.95,
        "zorder"  : 4,
    },
    {
        "label"   : r"$\kappa_{300}$  (FWHM 300 pm)",
        "results" : sweep_results_03,
        "ls"      : "--",
        "lw"      : 1.8,
        "marker"  : "s",
        "ms"      : 7,
        "markevery_frac": 25,
        "alpha"   : 0.90,
        "zorder"  : 3,
    },
    {
        "label"   : r"$\kappa_{100}$  (FWHM 100 pm)",
        "results" : sweep_results_02,
        "ls"      : "-.",
        "lw"      : 1.6,
        "marker"  : "^",
        "ms"      : 7.0,
        "markevery_frac": 25,
        "alpha"   : 0.85,
        "zorder"  : 2,
    },
]

# Directorio de salida (hereda FIGURES_DIR de Cell 1)
FIGURES_COMPARE_DIR = DATA_DIR / "figures_comparison"
FIGURES_COMPARE_DIR.mkdir(parents=True, exist_ok=True)


# ─────────────────────────────────────────────────────────────────────────────
# Utilidad de extracción
# ─────────────────────────────────────────────────────────────────────────────
def _get_nc_power(results: dict, drop_k: int):
    """Retorna (nc_valid, p_valid) filtrando solo puntos computados."""
    mask = results["computed"].astype(bool)
    nc_v = n_cladding[mask]
    p_v  = results["drop_power_dBm"][mask, drop_k - 1]
    return nc_v, p_v


# ─────────────────────────────────────────────────────────────────────────────
# Verificación de disponibilidad de datos
# ─────────────────────────────────────────────────────────────────────────────
def _check_results_available() -> bool:
    ok = True
    for name, res in [
        ("sweep_results (κ original)", sweep_results),
        ("sweep_results_03 (κ₀₃)",    sweep_results_03),
        ("sweep_results_02 (κ₀₂)",    sweep_results_02),
    ]:
        wl     = res.get("wavelengths_m")
        n_comp = int(res["computed"].sum())
        if wl is None or n_comp == 0:
            log.warning(f"  [CELL C] Sin datos: {name}")
            ok = False
        else:
            log.info(f"  [CELL C] OK — {name}  ({n_comp} pts computados)")
    return ok


# ─────────────────────────────────────────────────────────────────────────────
# FIGURA ÚNICA — 9 curvas en un solo panel
#
# Organización visual:
#   • 3 colores  → identifican el anillo (RING_2 / RING_8 / RING_14)
#   • 3 estilos  → identifican el FWHM   (κ original / κ₀₃ / κ₀₂)
#
# Leyenda en dos bloques separados:
#   Bloque superior → color por anillo   (3 proxies)
#   Bloque inferior → estilo por FWHM   (3 proxies)
# ─────────────────────────────────────────────────────────────────────────────
def plot_all_rings_all_fwhm_single_figure(
    figsize=(11, 6.5),
    save: bool = True,
) -> plt.Figure:
    """
    Una sola figura con las 9 curvas P_det vs n_cladding.

    Ejes
    ----
    X : índice de refracción del cladding  n_clad  ∈ [1.33, 1.37]
    Y : potencia integrada en el detector  P_det  [dBm]

    Código visual doble
    -------------------
    Color      → identifica el anillo espectrómetro
    Estilo     → identifica la variante de acoplamiento / FWHM

    Leyenda externa en dos grupos
    ------------------------------
    • "Anillo espectrómetro"  — proxies de color, línea sólida neutra
    • "FWHM / variante κ"     — proxies negros con el estilo de línea
    """
    fig, ax = plt.subplots(figsize=figsize)

    # ── Trazar las 9 curvas ───────────────────────────────────────────────────
    # Orden: para cada variante κ, los 3 anillos juntos
    # (mantiene los grupos de estilo coherentes en la leyenda)
    for ks in KAPPA_STYLES:
        for dk in COMPARE_DROP_INDICES:
            nc_v, p_v = _get_nc_power(ks["results"], dk)
            me = max(1, len(nc_v) // ks["markevery_frac"])
            ax.plot(
                nc_v, p_v,
                color     = RING_COLORS[dk],
                ls        = ks["ls"],
                lw        = ks["lw"],
                marker    = ks["marker"],
                ms        = ks["ms"],
                markevery = me,
                alpha     = ks["alpha"],
                zorder    = ks["zorder"],
            )

    # ── Etiquetas de ejes ─────────────────────────────────────────────────────
    ax.set_xlabel(
        "$n_{clad}$",
        fontsize=18,
        labelpad=8,
    )
    ax.set_ylabel(
        "$P_{det}$  (dB)",
        fontsize=18,
        labelpad=8,
    )
    ax.set_title(
        "Respuesta de potencia drop vs $n_{clad}$  —  9 curvas  "
        "(3 anillos  ×  3 variantes FWHM)\n"
        r"$P_{det} = P_{src} \cdot \langle|T_{drop}(\lambda)|^2\rangle_\lambda$"
        "   [V3 — ONA multiport]",
        fontsize=12,
        pad=10,
    )

    # ── Formato menor del eje ─────────────────────────────────────────────────
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.grid(True, alpha=0.30, linewidth=0.5)

    # ── Leyenda bloque 1: ANILLO (color) ──────────────────────────────────────
    handles_rings = [
        mlines.Line2D(
            [], [],
            color  = RING_COLORS[dk],
            lw     = 2.2,
            ls     = "-",
            marker = "o",
            ms     = 10,
            label  = DROP_LABELS[dk - 1],
        )
        for dk in COMPARE_DROP_INDICES
    ]

    # ── Leyenda bloque 2: FWHM / κ (estilo de línea) ─────────────────────────
    handles_kappa = [
        mlines.Line2D(
            [], [],
            color  = "black",
            lw     = ks["lw"],
            ls     = ks["ls"],
            marker = ks["marker"],
            ms     = 10,
            label  = ks["label"],
        )
        for ks in KAPPA_STYLES
    ]

    # Leyenda 1 (anillos) — esquina superior derecha
    leg1 = ax.legend(
        handles        = handles_rings,
        title_fontsize = 9,
        fontsize       = 15,
        loc            = "upper right",
        framealpha     = 0.94,
        edgecolor      = "#AAAAAA",
    )
    ax.add_artist(leg1)   # permite añadir una segunda leyenda

    # Leyenda 2 (FWHM / κ) — parte inferior derecha
    ax.legend(
        handles        = handles_kappa,
        title_fontsize = 9,
        fontsize       = 15,
        loc            = "lower right",
        framealpha     = 0.94,
        edgecolor      = "#AAAAAA",
    )

    fig.tight_layout()

    # ── Guardado ──────────────────────────────────────────────────────────────
    if save:
        fname_png = FIGURES_COMPARE_DIR / "power_vs_ncladding_all9curves.png"
        fname_pdf = FIGURES_COMPARE_DIR / "power_vs_ncladding_all9curves.pdf"
        fig.savefig(fname_png, dpi=300, bbox_inches="tight")
        fig.savefig(fname_pdf,           bbox_inches="tight")
        # Copia adicional en FIGURES_DIR principal para fácil acceso
        fig.savefig(FIGURES_DIR / "power_vs_ncladding_all9curves.png",
                    dpi=300, bbox_inches="tight")
        log.info(f"Saved → power_vs_ncladding_all9curves.png/pdf")
        log.info(f"        {fname_png}")

    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Ejecución
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 72)
print("  CELL C — Comparación FWHM  │  color=anillo  │  estilo=κ")
print("  RESULTADO: UNA SOLA FIGURA con las 9 curvas")
print("=" * 72)
print(f"  Anillos    : {[DROP_LABELS[dk-1] for dk in COMPARE_DROP_INDICES]}")
print(f"  Colores    : {[RING_COLORS[dk] for dk in COMPARE_DROP_INDICES]}")
print(f"  κ variantes: {[ks['label'] for ks in KAPPA_STYLES]}")
print(f"  Salida     : {FIGURES_COMPARE_DIR}")
print()

if _check_results_available():
    fig_all9 = plot_all_rings_all_fwhm_single_figure(figsize=(11, 6.5), save=True)
    plt.show()
    print()
    print(f"  Figura guardada → {FIGURES_COMPARE_DIR / 'power_vs_ncladding_all9curves.png'}")
    print(f"                    {FIGURES_COMPARE_DIR / 'power_vs_ncladding_all9curves.pdf'}")
    print(f"  Copia rápida   → {FIGURES_DIR / 'power_vs_ncladding_all9curves.png'}")
else:
    print("\n  ⚠  Faltan datos.  Ejecuta las celdas A y/o B primero.")

# ═════════════════════════════════════════════════════════════════════════════
#  END CELL C  (figura única 9 curvas: 3 anillos × 3 FWHM)
# ═════════════════════════════════════════════════════════════════════════════

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL D — Tracking de resonancia superpuesto:                              ║
# ║           Sweep INTERCONNECT original (200 pts)  vs                        ║
# ║           Datos alternativos / experimentales (20 pts)                     ║
# ║                                                                             ║
# ║  OBJETIVO                                                                   ║
# ║  ─────────────────────────────────────────────────────────────────────────  ║
# ║  Generar una figura única que superpone:                                   ║
# ║    • Serie 1 (azul, continua)  : curva ya existente de                    ║
# ║      plot_resonance_tracking_nc() — 200 puntos del sweep principal        ║
# ║      λ_dip extraída de T_sensor_through_dB vs n_cladding                  ║
# ║    • Serie 2 (naranja, círculos) : 20 puntos nuevos con sus propios       ║
# ║      valores de n_cladding y λ_peak (Through = Drop en todos los pts)    ║
# ║      junto a su propio ajuste lineal y sensibilidad                        ║
# ║                                                                             ║
# ║  DEPENDENCIAS HEREDADAS — NO se redefinen aquí:                            ║
# ║    get_results(), _valid_mask(), n_cladding, wavelengths_m                 ║
# ║    T_sensor_through_dB, computed                                           ║
# ║    FIGURES_DIR, DATA_DIR                                                   ║
# ║    plt, np, ticker, log                                                    ║
# ║    (todas definidas en Cells 1–7 del notebook principal)                  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────────────
# Dataset alternativo — 20 puntos
# Formato del log original:
#   [01/20] n_cladding = 1.3300  →  Peak Through: 1549.963 nm
#   ...
#   [20/20] n_cladding = 1.3700  →  Peak Through: 1557.915 nm
#
# NOTA: Peak Through == Peak Drop en todos los puntos, por lo que
#       se usa Peak Through como longitud de onda de resonancia.
# ─────────────────────────────────────────────────────────────────────────────

ALT_N_CLADDING = np.array([
    1.3300, 1.3321, 1.3342, 1.3363, 1.3384,
    1.3405, 1.3426, 1.3447, 1.3468, 1.3489,
    1.3511, 1.3532, 1.3553, 1.3574, 1.3595,
    1.3616, 1.3637, 1.3658, 1.3679, 1.3700,
])

ALT_LAMBDA_THROUGH_NM = np.array([
    1549.963, 1550.414, 1550.815, 1551.216, 1551.617,
    1552.019, 1552.420, 1552.822, 1553.225, 1553.677,
    1554.080, 1554.483, 1554.936, 1555.339, 1555.743,
    1556.197, 1556.601, 1557.056, 1557.511, 1557.915,
])

# Usamos Peak Through (= Peak Drop) como longitud de onda de resonancia
ALT_LAMBDA_RES_NM = ALT_LAMBDA_THROUGH_NM.copy()

# ── Ajuste lineal del dataset alternativo ──────────────────────────────────
coeffs_alt = np.polyfit(ALT_N_CLADDING, ALT_LAMBDA_RES_NM, 1)
sens_alt    = coeffs_alt[0]                               # nm/RIU
fit_alt     = np.poly1d(coeffs_alt)(ALT_N_CLADDING)
r2_alt      = 1.0 - (
    np.sum((ALT_LAMBDA_RES_NM - fit_alt) ** 2) /
    np.sum((ALT_LAMBDA_RES_NM - ALT_LAMBDA_RES_NM.mean()) ** 2)
)

print("=" * 65)
print("  CELL D — Resonance Tracking Overlay  │  2 series")
print("=" * 65)
print(f"  Dataset alternativo  :  {len(ALT_N_CLADDING)} puntos")
print(f"  n_cladding range     :  {ALT_N_CLADDING[0]:.4f}  →  {ALT_N_CLADDING[-1]:.4f}")
print(f"  λ_res range          :  {ALT_LAMBDA_RES_NM[0]:.3f}  →  {ALT_LAMBDA_RES_NM[-1]:.3f} nm")
print(f"  Sensibilidad (alt)   :  S = {sens_alt:.2f} nm/RIU    R² = {r2_alt:.6f}")
print("=" * 65)


# ─────────────────────────────────────────────────────────────────────────────
# Función principal del plot superpuesto
# ─────────────────────────────────────────────────────────────────────────────

def plot_resonance_tracking_overlay(
    results=None,
    figsize: tuple = (8, 5.5),
    save: bool = True,
    label_serie1: str = "INTCN  (200 pts)",
    label_serie2: str = "varFDTD (20 pts)",
    color_serie1: str = "#2166ac",
    color_serie2: str = "#e6550d",
    save_stem: str = "nc_resonance_tracking_overlay",
) -> "plt.Figure":
    """
    Superpone dos series de tracking de resonancia en una sola figura.

    Las etiquetas de cada serie se muestran mediante anotaciones con flecha
    directamente sobre cada recta de ajuste, incluyendo la sensibilidad S y R².
    No se usa leyenda convencional.

    Serie 1 — sweep principal (n_cladding × λ_dip extraída de
              T_sensor_through_dB, idéntico a plot_resonance_tracking_nc)

    Serie 2 — dataset alternativo de 20 puntos definido en
              ALT_N_CLADDING / ALT_LAMBDA_RES_NM.

    Parámetros
    ----------
    results      : dict devuelto por get_results(); si None se obtiene automáticamente.
    figsize      : tamaño de la figura en pulgadas.
    save         : si True, guarda PNG y PDF en FIGURES_DIR.
    label_serie1 : etiqueta de la primera recta (texto de la anotación).
    label_serie2 : etiqueta de la segunda recta (texto de la anotación).
    color_serie1 : color hexadecimal Serie 1.
    color_serie2 : color hexadecimal Serie 2.
    save_stem    : nombre base del fichero de salida (sin extensión).

    Retorna
    -------
    fig : matplotlib.figure.Figure
    """
    # ── Carga de resultados del sweep principal ───────────────────────────────
    if results is None:
        results = get_results()

    mask  = _valid_mask(results)
    nc_v  = n_cladding[mask]                           # (n_valid,)
    T_v   = results["T_sensor_through_dB"][mask, :]    # (n_valid, n_wl)
    wl_nm = results["wavelengths_m"] * 1e9             # (n_wl,)

    # ── Extracción del dip (misma lógica que plot_resonance_tracking_nc) ──────
    dip_idx = np.argmin(T_v, axis=1)
    lam_dip = wl_nm[dip_idx]

    # ── Ajuste lineal Serie 1 ─────────────────────────────────────────────────
    coeffs1 = np.polyfit(nc_v, lam_dip, 1)
    sens1   = coeffs1[0]
    fit1    = np.poly1d(coeffs1)(nc_v)
    r2_1    = 1.0 - (
        np.sum((lam_dip - fit1) ** 2) /
        np.sum((lam_dip - lam_dip.mean()) ** 2)
    )

    # ── Ajuste lineal Serie 2 ─────────────────────────────────────────────────
    coeffs2 = coeffs_alt
    sens2   = sens_alt
    r2_2    = r2_alt

    # Rango X unificado para extender la recta de ajuste de Serie 2
    nc_ext = np.array([
        min(nc_v.min(), ALT_N_CLADDING.min()),
        max(nc_v.max(), ALT_N_CLADDING.max()),
    ])

    # ── Figura ────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=figsize)

    # — Serie 1: scatter + recta de ajuste ────────────────────────────────────
    ax.scatter(
        nc_v, lam_dip,
        s=20, zorder=6,
        color=color_serie1,
    )
    ax.plot(
        nc_v, fit1,
        ls="--", lw=1.8,
        color=color_serie1,
        zorder=5,
    )

    # — Serie 2: scatter + recta de ajuste ────────────────────────────────────
    ax.scatter(
        ALT_N_CLADDING, ALT_LAMBDA_RES_NM,
        s=55, marker="o",
        edgecolors=color_serie2,
        facecolors="none",
        linewidths=1.8,
        zorder=8,
    )
    ax.plot(
        nc_ext, np.poly1d(coeffs2)(nc_ext),
        ls="-.", lw=1.8,
        color=color_serie2,
        zorder=7,
    )

    # ── Anotaciones con flecha — Serie 1 (INTERCONNECT) ──────────────────────
    # Punto de anclaje: 30 % del recorrido de nc_v
    ann1_x = nc_v[int(len(nc_v) * 0.30)]
    ann1_y = float(np.poly1d(coeffs1)(ann1_x))
    # Texto desplazado hacia abajo-izquierda para no solapar la recta
    ann1_tx = ann1_x + 0.028
    ann1_ty = ann1_y - 0
    ax.annotate(
        f"{label_serie1}\n"
        f"$S = {sens1:.2f}$ nm/RIU\n"
        ,
        xy=(ann1_x, ann1_y),
        xytext=(ann1_tx, ann1_ty),
        fontsize=23,
        color=color_serie1,
        fontweight="bold",
        ha="right",
        va="top",
        arrowprops=dict(
            arrowstyle="->",
            color=color_serie1,
            lw=1.4,
            connectionstyle="arc3,rad=-0.25",
        ),
        bbox=dict(
            boxstyle="round,pad=0.35",
            facecolor="white",
            edgecolor=color_serie1,
            alpha=0.88,
            linewidth=1.0,
        ),
        zorder=10,
    )

    # ── Anotaciones con flecha — Serie 2 (varFDTD) ───────────────────────────
    # Punto de anclaje: 68 % del recorrido del array alternativo
    ann2_x = ALT_N_CLADDING[int(len(ALT_N_CLADDING) * 0.68)]
    ann2_y = float(np.poly1d(coeffs2)(ann2_x))
    # Texto desplazado hacia arriba-derecha
    ann2_tx = ann2_x - 0.03
    ann2_ty = ann2_y + 0.55
    ax.annotate(
        f"{label_serie2}\n"
        f"$S = {sens2:.2f}$ nm/RIU\n"
        ,
        xy=(ann2_x, ann2_y),
        xytext=(ann2_tx, ann2_ty),
        fontsize=23,
        color=color_serie2,
        fontweight="bold",
        ha="left",
        va="bottom",
        arrowprops=dict(
            arrowstyle="->",
            color=color_serie2,
            lw=1.4,
            connectionstyle="arc3,rad=0.25",
        ),
        bbox=dict(
            boxstyle="round,pad=0.35",
            facecolor="white",
            edgecolor=color_serie2,
            alpha=0.88,
            linewidth=1.0,
        ),
        zorder=10,
    )

    # ── Etiquetas y título ────────────────────────────────────────────────────
    ax.set_xlabel(
        "$n_{clad}$",
        fontsize=20,
    )
    ax.set_ylabel(
        r"$\lambda$ (nm)",
        fontsize=20,
    )
    ax.set_title(
        "Tracking de resonancia — Sensor through (ONA input 1)\n"
        "Comparación: sweep principal vs dataset alternativo",
        fontsize=12,
    )

    # ── Ticks menores (estilo heredado del notebook) ──────────────────────────
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())

    fig.tight_layout()

    # ── Guardado ──────────────────────────────────────────────────────────────
    if save:
        fig.savefig(FIGURES_DIR / f"{save_stem}.png", dpi=300, bbox_inches="tight")
        fig.savefig(FIGURES_DIR / f"{save_stem}.pdf",           bbox_inches="tight")
        log.info(f"Saved → {save_stem}.png/pdf  ({FIGURES_DIR})")

    log.info(f"Sensibilidad Serie 1 : {sens1:.4f} nm/RIU   R²={r2_1:.8f}")
    log.info(f"Sensibilidad Serie 2 : {sens2:.4f} nm/RIU   R²={r2_2:.8f}")
    log.info(f"Δ sensibilidad       : {sens2 - sens1:+.4f} nm/RIU")

    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Ejecución
# ─────────────────────────────────────────────────────────────────────────────
_res_d = get_results()
fig_overlay = plot_resonance_tracking_overlay(_res_d)
plt.show()

print()
print(f"  Figura guardada → {FIGURES_DIR / 'nc_resonance_tracking_overlay.png'}")
print(f"                    {FIGURES_DIR / 'nc_resonance_tracking_overlay.pdf'}")

# ═════════════════════════════════════════════════════════════════════════════
#  END CELL D  (resonance tracking overlay: 2 series superpuestas)
# ═════════════════════════════════════════════════════════════════════════════